In [12]:
from typing import List, Any
import os
import weaviate
import json
import pandas as pd
from langchain_weaviate import WeaviateVectorStore
from sentence_transformers import SentenceTransformer
from IPython.display import display, Markdown
from dotenv import load_dotenv


In [2]:
load_dotenv('../.env.example/.env')

True

In [13]:
# weaviate Keys
WEAVIATE_URL = os.environ["WEAVIATE_URL"]
WEAVIATE_API_KEY = os.environ["WEAVIATE_API_KEY"]

In [14]:
weaviate_client = weaviate.connect_to_weaviate_cloud(
    cluster_url=WEAVIATE_URL,
    auth_credentials=WEAVIATE_API_KEY,
)

UnexpectedStatusCodeError: Meta endpoint! Unexpected status code: 503, with response body: None.

In [10]:
weaviate_client.is_ready()

NameError: name 'weaviate_client' is not defined

In [20]:
class SentenceTransformersEmbeddings:
    def __init__(self, model_name: str):
        self.model = SentenceTransformer(model_name)

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        # returns a list of embeddings for documents
        return self.model.encode(texts).tolist()

    def embed_query(self, text: str) -> List[float]:
        # returns a single embedding for a query
        return self.model.encode([text])[0].tolist()

In [21]:
embedding_model = SentenceTransformersEmbeddings('sentence-transformers/all-mpnet-base-v2')

Loading weights: 100%|██████████| 199/199 [00:05<00:00, 33.84it/s, Materializing param=pooler.dense.weight]                         
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
v = embedding_model.embed_query("rape sentencing guidelines, punishment for rape, statutory penalties for rape, judicial discretion in rape cases")

In [15]:
Euro_Laws = weaviate_client.collections.use("Euro_Laws")

In [16]:
response = Euro_Laws.query.near_vector(
    near_vector= v,
    limit=5
)

In [11]:
for obj in response.objects:
    print (obj.properties)

{'act_name': 'Directive 2012/29/EU of the European Parliament and of the Council of 25Ã\x82 October 2012 establishing minimum standards on the rights, support and protection of victims of crime, and replacing Council Framework Decision 2001/220/JHA', 'total_chunks': 31, 'status': 'In Force', 'legal_basis': '12010E082; 12010E294', 'eurovoc': 'crime against individuals; restorative justice; aid for victims; access to justice; AFSJ', 'act_type': 'Directive', 'celex': '32012L0029', 'chunk_number': 9, 'document_length': 78554, 'authors': 'European Parliament; European Council', 'subject_matter': 'criminal law;  justice;  European construction', 'cites': 'dec_framw/2002/475; 52012XX0209%2802%29; 32001R45; dec_framw/2008/977; 42000A0712%2801%29; 52010XG0504%2801%29; dec_framw/2009/948; 52009IP0098%2801%29; 32011L99; 32011L93; 52011IP0127; 32011G0628%2801%29; 32011L36', 'text': "competent authorities are aware of the victim and throughout criminal proceedings and for an appropriate time after 

In [17]:
celex_ids = []

for obj in response.objects:
    celex_ids.append(obj.properties['celex'])

celex_ids

['32011L0093', '32012L0029', '32005F0214', '32019D0417', '32012L0029']

In [13]:
celex_ids = []

for obj in response.objects:
    celex_ids.append(obj.properties['celex'])

celex_ids

['32012L0029', '32005F0214', '32011L0093', '32019D0417', '32008F0947']

In [17]:
from weaviate.classes.query import Filter

In [12]:
eur_docs = weaviate_client.collections.get("Euro_Law_Documents")

In [27]:
class weaviate_client :
    def collections ():

        def exists ():
           return 'exists'
        return exists ()
    
    

In [28]:
vectorstore = WeaviateVectorStore(
    client = weaviate_client,
    index_name = "Euro_Laws",
    text_key="text",
    embedding = embedding_model
)

AttributeError: 'function' object has no attribute 'exists'

In [1]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
docs = retriever.invoke("trespassing sentencing penalties fines imprisonment statutory sanctions jurisdiction")

NameError: name 'vectorstore' is not defined

In [21]:
docs[0].metadata.keys()

dict_keys(['total_chunks', 'document_length', 'eurovoc', 'chunk_number', 'subject_matter', 'celex', 'additional_info', 'authors', 'act_type', 'legal_basis', 'status', 'treaty', 'cites', 'act_name'])

In [15]:
def get_full_doc_weaviate(celex_id):

    response = eur_docs.query.fetch_objects(
    filters=Filter.by_property("celex").equal(celex_id),
    limit=1
)
    r = response.objects[0].properties

    # r is a dict with key, values for each doc
    return r    

In [10]:
def search_docs(query):
    celex_ids = []
    full_doc_info = """ """

    weaviate_client.connect()

    retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
    docs = retriever.invoke(query)

    for doc in docs:
        celex_ids.append(doc.metadata['celex'])

    celex_ids = list(set(celex_ids))    

    for i , celex_id in enumerate(celex_ids):
        f_doc_meta = get_full_doc_weaviate(celex_id)

        full_doc_info += f"""

        doc {i} :

        'celex': {f_doc_meta['celex']}
        'status': {f_doc_meta['status']}
        'act_type': {f_doc_meta['act_type']}
        'treaty': {f_doc_meta['treaty']}

        full_doc :

        {f_doc_meta['full_doc']}
{"=="*15} "END OF DOC" {"=="*15}
        """
    weaviate_client.close()
    
    return full_doc_info    

# hybrid search

In [8]:
retriever = vectorstore.as_retriever(
    search_type="hybrid",
    search_kwargs={
        "k": 5,
        "alpha": 0.5
    }
)
docs = retriever.invoke("trespassing sentencing penalties fines imprisonment statutory sanctions jurisdiction")

ValidationError: 1 validation error for VectorStoreRetriever
  Value error, search_type of hybrid not allowed. Valid values are: ('similarity', 'similarity_score_threshold', 'mmr') [type=value_error, input_value={'vectorstore': <langchai... {'k': 5, 'alpha': 0.5}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

In [26]:
weaviate_client.close()

In [18]:
display(Markdown(search_docs("trespassing sentencing penalties fines imprisonment statutory sanctions jurisdiction")))

 

        doc 0 :

        'celex': 32005F0212
        'status': In Force
        'act_type': Decision_FRAMW
        'treaty': TEU (1992)

        full_doc :

        15.3.2005 EN Official Journal of the European Union L 68/49 COUNCIL FRAMEWORK DECISION 2005/212/JHA of 24 February 2005 on Confiscation of Crime-Related Proceeds, Instrumentalities and Property THE COUNCIL OF THE EUROPEAN UNION, Having regard to the Treaty on European Union, and in particular Articles 29, 31(1)(c) and 34(2)(b) thereof, Having regard to the initiative of the Kingdom of Denmark (1), Having regard to the opinion of the European Parliament, Whereas: (1) The main motive for cross-border organised crime is financial gain. In order to be effective, therefore, any attempt to prevent and combat such crime must focus on tracing, freezing, seizing and confiscating the proceeds from crime. However, this is made difficult, inter alia, as a result of differences between Member States legislation in this area. (2) In the conclusions of the Vienna European Council of December 1998, the European Council called for a strengthening of EU efforts to combat international organised crime in accordance with an action plan on how best to implement the provisions of the Treaty of Amsterdam in an area of freedom, security and justice (2). (3) Pursuant to paragraph 50(b) of the Vienna Action Plan, within five years of the entry into force of the Treaty of Amsterdam, national provisions governing seizures and confiscation of the proceeds from crime must be improved and approximated where necessary, taking account of the rights of third parties in bona fide. (4) Paragraph 51 of the conclusions of the Tampere European Council of 15 and 16 October 1999 stresses that money laundering is at the very heart of organised crime, and should be rooted out wherever it occurs and that the European Council is determined to ensure that concrete steps are taken to trace, freeze, seize and confiscate the proceeds from crime. The European Council also calls, in paragraph 55, for the approximation of criminal law and procedures on money laundering (e.g. tracing, freezing and confiscating funds). (5) Pursuant to Recommendation 19 in the 2000 action plan entitled The prevention and control of organised crime: a European Union strategy for the beginning of the new millennium, which was approved by the Council on 27 March 2000 (3), an examination should be made of the possible need for an instrument which, taking into account best practice in the Member States and with due respect for fundamental legal principles, introduces the possibility of mitigating, under criminal, civil or fiscal law, as appropriate, the onus of proof regarding the source of assets held by a person convicted of an offence related to organised crime. (6) Pursuant to Article 12, on confiscation and seizure, of the UN Convention of 12 December 2000 against Transnational Organised Crime, States Parties may consider the possibility of requiring that an offender demonstrate the lawful origin of alleged proceeds of crime or other property liable to confiscation, to the extent that such a requirement is consistent with the principles of their domestic law and with the nature of judicial proceedings. (7) All Member States have ratified the Council of Europe Convention of 8 November 1990 on Laundering, Search, Seizure and Confiscation of the Proceeds from Crime. Some Member States have submitted declarations with regard to Article 2 of the Convention concerning confiscation so as to be obliged to confiscate proceeds only from a number of specified offences. (8) The Council Framework Decision 2001/500/JHA (4) lays down provisions on money laundering, the identification, tracing, freezing, seizing and confiscation of instrumentalities and the proceeds from crime. Under that Framework Decision, Member States are also obliged not to make or uphold reservations in respect of the provisions of the Council of Europe Convention concerning confiscation, insofar as the offence is punishable by deprivation of liberty or a detention order for a maximum of more than one year. (9) The existing instruments in this area have not to a sufficient extent achieved effective cross-border cooperation with regard to confiscation as there are still a number of Member States which are unable to confiscate the proceeds from all offences punishable by deprivation of liberty for more than one year. (10) The aim of this Framework Decision is to ensure that all Member States have effective rules governing the confiscation of proceeds from crime, inter alia, in relation to the onus of proof regarding the source of assets held by a person convicted of an offence related to organised crime. This Decision is linked to a Danish draft Framework Decision on the mutual recognition within the European Union of decisions concerning the confiscation of proceeds from crime and asset-sharing, which is being submitted at the same time. (11) This Framework Decision does not prevent a Member State from applying its fundamental principles relating to due process, in particular the presumption of innocence, property rights, freedom of association, freedom of the press and freedom of expression in other media, HAS ADOPTED THIS FRAMEWORK DECISION: Article 1 Definitions For the purposes of this Framework Decision:  proceeds means any economic advantage from criminal offences. It may consist of any form of property as defined in the following indent,  property includes property of any description, whether corporeal or incorporeal, movable or immovable, and legal documents or instruments evidencing title to or interest in such property,  instrumentalities means any property used or intended to be used, in any manner, wholly or in part, to commit a criminal offence or criminal offences,  confiscation means a penalty or measure, ordered by a court following proceedings in relation to a criminal offence or criminal offences, resulting in the final deprivation of property,  legal person means any entity having such status under the applicable national law, except for States or other public bodies in the exercise of State authority and for public international organisations. Article 2 Confiscation 1. Each Member State shall take the necessary measures to enable it to confiscate, either wholly or in part, instrumentalities and proceeds from criminal offences punishable by deprivation of liberty for more than one year, or property the value of which corresponds to such proceeds. 2. In relation to tax offences, Member States may use procedures other than criminal procedures to deprive the perpetrator of the proceeds of the offence. Article 3 Extended powers of confiscation 1. Each Member State shall as a minimum adopt the necessary measures to enable it, under the circumstances referred to in paragraph 2, to confiscate, either wholly or in part, property belonging to a person convicted of an offence (a) committed within the framework of a criminal organisation as defined in Joint Action 98/733/JHA of 21 December 1998 on making it a criminal offence to participate in a criminal organisation in the Member States of the European Union (5), when the offence is covered by:  Council Framework Decision 2000/383/JHA of 29 May 2000 on increasing protection by criminal penalties and other sanctions against counterfeiting in connection with the introduction of the euro (6),  Council Framework Decision 2001/500/JHA of 26 June 2001 on money laundering, the identification, tracing, freezing, seizing and confiscation of instrumentalities and the proceeds of crime (7),  Council Framework Decision 2002/629/JHA of 19 July 2002 on combating trafficking in human beings (8),  Council Framework Decision 2002/946/JHA of 28 November 2002 on the strengthening of the penal framework to prevent the facilitation of unauthorised entry, transit and residence (9),  Council Framework Decision 2004/68/JHA of 22 December 2003 on combating the sexual exploitation of children and child pornography (10),  Council Framework Decision 2004/757/JHA of 25 October 2004 laying down minimum provisions on the constituent elements of criminal acts and penalties in the field of illicit drug trafficking (11), (b) which is covered by the Council Framework Decision 2002/475/JHA of 13 June 2002 on combating terrorism (12), provided that the offence according to the Framework Decisions referred to above  regarding offences other than money laundering are punishable with criminal penalties of a maximum of at least between 5 and 10 years of imprisonment,  regarding money laundering, are punishable with criminal penalties of a maximum of at least 4 years of imprisonment, and the offence is of such a nature that it can generate financial gain. 2. Each Member State shall take the necessary measures to enable confiscation under this Article at least: (a) where a national court based on specific facts is fully convinced that the property in question has been derived from criminal activities of the convicted person during a period prior to conviction for the offence referred to in paragraph 1 which is deemed reasonable by the court in the circumstances of the particular case, or, alternatively, (b) where a national court based on specific facts is fully convinced that the property in question has been derived from similar criminal activities of the convicted person during a period prior to conviction for the offence referred to in paragraph 1 which is deemed reasonable by the court in the circumstances of the particular case, or, alternatively, (c) where it is established that the value of the property is disproportionate to the lawful income of the convicted person and a national court based on specific facts is fully convinced that the property in question has been derived from the criminal activity of that convicted person. 3. Each Member State may also consider adopting the necessary measures to enable it to confiscate, in accordance with the conditions set out in paragraphs 1 and 2, either wholly or in part, property acquired by the closest relations of the person concerned and property transferred to a legal person in respect of which the person concerned  acting either alone or in conjunction with his closest relations  has a controlling influence. The same shall apply if the person concerned receives a significant part of the legal persons income. 4. Member States may use procedures other than criminal procedures to deprive the perpetrator of the property in question. Article 4 Legal remedies Each Member State shall take the necessary measures to ensure that interested parties affected by measures under Articles 2 and 3 have effective legal remedies in order to preserve their rights. Article 5 Safeguards This Framework Decision shall not have the effect of altering the obligation to respect fundamental rights and fundamental principles, including in particular the presumption of innocence, as enshrined in Article 6 of the Treaty on European Union. Article 6 Implementation 1. Member States shall adopt the necessary measures to comply with this Framework Decision by 15 March 2007. 2. Member States shall transmit to the General Secretariat of the Council and to the Commission, by 15 March 2007, the text of the provisions transposing into their national law the obligations imposed on them under this Framework Decision. In accordance with a report established on the basis of this information and a written report from the Commission, the Council shall assess, by 15 June 2007, the extent to which Member States have taken the necessary measures in order to comply with this Framework Decision. Article 7 Entry into force This Framework Decision shall enter into force on the day of its publication in the Official Journal of the European Union. Done at Brussels, 24 February 2005. For the Council The President N. SCHMIT (1) OJ C 184, 2.8.2002, p. 3. (2) OJ C 19, 23.1.1999, p. 1. (3) OJ C 124, 3.5.2000, p. 1. (4) OJ L 182, 5.7.2001, p. 1. (5) OJ L 351, 29.12.1998, p. 1. (6) OJ L 140, 14.6.2000, p. 1. (7) OJ L 182, 5.7.2001, p. 1. (8) OJ L 203, 1.8.2002, p. 1. (9) OJ L 328, 5.12.2002, p. 1. (10) OJ L 13, 20.1.2004, p. 44. (11) OJ L 335, 11.11.2004, p. 8. (12) OJ L 164, 22.6.2002, p. 3.
============================== "END OF DOC" ==============================
        

        doc 1 :

        'celex': 32009D0316
        'status': In Force
        'act_type': Decision
        'treaty': TEU (1992)

        full_doc :

        7.4.2009 EN Official Journal of the European Union L 93/33 COUNCIL DECISION 2009/316/JHA of 6 April 2009 on the establishment of the European Criminal Records Information System (ECRIS) in application of Article 11 of Framework Decision 2009/315/JHA THE COUNCIL OF THE EUROPEAN UNION, Having regard to the Treaty on European Union, and in particular Articles 31 and 34(2)(c) thereof, Having regard to the Council Framework Decision 2009/315/JHA of 26 February 2009 on the organisation and content of the exchange of information extracted from the criminal record between Member States (1), and in particular Article 11(4) thereof, Having regard to the proposal from the Commission, Having regard to the opinion of the European Parliament (2), Whereas: (1) Article 29 of the Treaty on European Union states that the Union's objective is to provide citizens with a high level of safety in the area of freedom, security and justice. This objective presupposes the systematic exchange between the competent authorities of the Member States of information extracted from criminal records in a way that would guarantee its common understanding and the efficiency of such exchange. (2) Information on convictions handed down against Member States' nationals by other Member States does not circulate efficiently on the current basis of the European Convention on Mutual Assistance in Criminal Matters of 20 April 1959. Therefore, there is a need for more efficient and accessible procedures of exchange of such information at European Union level. (3) The need to improve the exchange of information on convictions was prioritised in the European Council Declaration on Combating Terrorism of 25 and 26 March 2004 and was subsequently reiterated in the Hague Programme (3) and in the Action Plan (4) on its implementation. Furthermore, the computerised interconnection of criminal records at European Union level was recognised as a political priority by the European Council in its Conclusions of 21 and 22 June 2007. (4) The computerised interconnection of criminal records is part of the E-Justice project, which was acknowledged as a priority by the European Council several times in 2007. (5) A pilot project is currently being developed with a view to interconnecting criminal records. Its achievements constitute a valuable basis for further work on computerised exchange of information at the European Union level. (6) This Decision aims to implement Framework Decision 2009/315/JHA in order to build and develop a computerised system of exchange of information on convictions between Member States. Such a system should be capable of communicating information on convictions in a form which is easily understandable. Therefore, a standardised format allowing information to be exchanged in a uniform, electronic and easily computer-translatable way as well as any other means of organising and facilitating electronic exchanges of information on convictions between central authorities of Member States should be set up. (7) This Decision is based on the principles established by Framework Decision 2009/315/JHA and applies and supplements those principles from a technical standpoint. (8) The categories of data to be entered into the system, the purposes for which the data is to be entered, the criteria for its entry, the authorities permitted to access the data, and some specific rules on protection of personal data are defined in the Framework Decision 2009/315/JHA. (9) Neither this Decision nor Framework Decision 2009/315/JHA establishes any obligation to exchange information about non-criminal rulings. (10) Since the objective of this Decision is not to harmonise national systems of criminal records there is no obligation for a convicting Member State to change its internal system of criminal records as regards the use of information for domestic purposes. (11) The European Criminal Records System (ECRIS) is a decentralised information technology system. The criminal records data should be stored solely in databases operated by Member States, and there should be no direct online access to criminal records databases of other Member States. Member States should bear the responsibility for the operation of national criminal records databases and for the efficient exchanges of information between themselves. The common communication infrastructure of ECRIS should be initially the Trans European Services for Telematics between Administrations (S-TESTA) network. All expenditure concerning the common communication infrastructure should be covered by the general budget of the European Union. (12) The reference tables of categories of offences and categories of penalties and measures provided for in this Decision should facilitate the automatic translation and should enable the mutual understanding of the information transmitted by using a system of codes. The content of the tables is the result of the analysis of the needs of all 27 Member States. That analysis took into account the pilot project categorisation and the results of the clustering exercise of various national offences and penalties and measures. Moreover, in case of the table of offences, it also took into consideration the existing harmonised common definitions on the European and international level as well as the Eurojust and Europol data models. (13) In order to ensure the mutual understanding and transparency of the common categorisation, each Member State should submit the list of national offences and penalties and measures falling in each category referred to in the respective table. Member States may provide a description of offences and penalties and measures and, given the usefulness of such description, they should be encouraged to do so. Such information should be made accessible to Member States. (14) The reference tables of categories of offences and categories of penalties and measures provided for in this Decision are not designed to set up legal equivalences between offences and penalties and measures existing at national level. They are a tool aimed at helping the recipient to gain better understanding of the fact(s) and type of penalty(ies) or measure(s) contained in the information transmitted. The accuracy of the codes mentioned cannot be fully guaranteed by the Member State supplying the information and it should not preclude the competent authorities in the receiving Member State from interpreting the information. (15) The reference tables of categories of offences and categories of penalties and measures should be revised and updated in accordance with the procedure for the adoption of implementing measures for decisions provided for in the Treaty on European Union. (16) Members States and the Commission should inform and consult one another within the Council in accordance with the modalities set out in the Treaty on European Union, with a view to drawing up a non-binding manual for practitioners which should address the procedures governing the exchange of information, in particular modalities of identification of offenders, common understanding of the categories of offences and penalties and measures, and explanation of problematic national offences and penalties and measures, and ensuring the coordination necessary for the development and operation of ECRIS. (17) In order to accelerate the development of ECRIS, the Commission should adopt a number of technical measures to assist Member States in preparing the technical infrastructure for interconnecting their criminal records databases. The Commission may provide reference implementation software, namely appropriate software enabling Member States to make this interconnection, which they may choose to apply instead of their own interconnection software implementing a common set of protocols enabling the exchange of information between criminal records databases. (18) Council Framework Decision 2008/977/JHA of 27 November 2008 on the protection of personal data processed in the framework of police and judicial cooperation in criminal matters (5) should apply in the context of computerised exchange of information extracted from criminal records of Member States, providing for an adequate level of data protection when information is exchanged between Member States, whilst allowing for Member States to require higher standards of protection to national data processing. (19) Since the objective of this Decision, namely the development of a computerised system for the exchange of information on convictions between Member States, cannot be adequately achieved by the Member States unilaterally, and can therefore, by reason of the necessity for coordinated action in the European Union, be better achieved at the level of the European Union, the Council may adopt measures, in accordance with the principle of subsidiarity referred to in Article 2 of the Treaty on European Union and set out in Article 5 of the Treaty establishing the European Community. In accordance with the principle of proportionality, as set out in the Article 5 of the Treaty establishing the European Community, this Decision does not go beyond what is necessary in order to achieve that objective. (20) This Decision respects fundamental rights and observes the principles recognised in particular by Article 6 of the Treaty on European Union and reflected by the Charter of Fundamental Rights of the European Union, HAS DECIDED AS FOLLOWS: Article 1 Subject matter This Decision establishes the European Criminal Records Information System (ECRIS). This Decision also establishes the elements of a standardised format for the electronic exchange of information extracted from criminal records between the Member States, in particular as regards information on the offence giving rise to the conviction and information on the content of the conviction, as well as other general and technical implementation means related to organising and facilitating the exchange of information. Article 2 Definitions For the purposes of this Decision, the definitions laid down in Framework Decision 2009/315/JHA shall apply. Article 3 European Criminal Records Information System (ECRIS) 1. ECRIS is a decentralised information technology system based on the criminal records databases in each Member State. It is composed of the following elements: (a) an interconnection software built in compliance with a common set of protocols enabling the exchange of information between Member States' criminal records databases; (b) a common communication infrastructure that provides an encrypted network. 2. This Decision is not aimed at establishing any centralised criminal records database. All criminal records data shall be stored solely in databases operated by the Member States. 3. Central authorities of the Member States referred to in Article 3 of Framework Decision 2009/315/JHA shall not have direct online access to criminal records databases of other Member States. The best available techniques identified together by Member States with the support of the Commission shall be employed to ensure the confidentiality and integrity of criminal records information transmitted to other Member States. 4. The interconnection software and databases storing, sending and receiving information extracted from criminal records shall operate under the responsibility of the Member State concerned. 5. The common communication infrastructure shall be the S-TESTA communications network. Any further developments thereof or any alternative secure network shall ensure that the common communication infrastructure in place continues to meet the conditions set out in paragraph 6. 6. The common communication infrastructure shall be operated under the responsibility of the Commission, and shall fulfil the security requirements and thoroughly respond to the needs of ECRIS. 7. In order to ensure the efficient operation of ECRIS, the Commission shall provide general support and technical assistance, including the collection and drawing up of statistics referred to in Article 6(2)(b)(i) and the reference implementation software. 8. Notwithstanding the possibility of using the European Union financial programmes in accordance with the applicable rules, each Member State shall bear its own costs arising from the implementation, administration, use and maintenance of its criminal records database and the interconnection software referred to in paragraph 1. The Commission shall bear the costs arising from the implementation, administration, use, maintenance and future developments of the common communication infrastructure of ECRIS, as well as the implementation and future developments of the reference implementation software. Article 4 Format of transmission of information 1. When transmitting information in accordance with Article 4(2) and (3) and Article 7 of Framework Decision 2009/315/JHA relating to the name or legal classification of the offence and to the applicable legal provisions, Member States shall refer to the corresponding code for each of the offences referred to in the transmission, as provided for in the table of offences in Annex A. By way of exception, where the offence does not correspond to any specific sub-category, the open category code of the relevant or closest category of offences or, in the absence of the latter, an other offences code, shall be used for that particular offence. Member States may also provide available information relating to the level of completion and the level of participation in the offence and, where applicable, to the existence of total or partial exemption from criminal responsibility or to recidivism. 2. When transmitting information in accordance with Article 4(2) and (3) and Article 7 of Framework Decision 2009/315/JHA relating to the contents of the conviction, notably the sentence as well as any supplementary penalties, security measures and subsequent decisions modifying the enforcement of the sentence, Member States shall refer to the corresponding code for each of the penalties and measures referred to in the transmission, as provided for in the table of penalties and measures in Annex B. By way of exception, where the penalty or measure does not correspond to any specific sub-category, the open category code of the relevant or closest category of penalties and measures or, in the absence of the latter, an other penalties and measures code, shall be used for that particular penalty or measure. Member States shall also provide, where applicable, available information relating to the nature and/or conditions of execution of the penalty or measure imposed as provided for in the parameters of Annex B. The parameter non-criminal ruling shall be indicated only in cases where information on such a ruling is provided on a voluntary basis by the Member State of nationality of the person concerned, when replying to a request for information on convictions. Article 5 Information on national offences and penalties and measures 1. The following information shall be provided by the Member States to the General Secretariat of the Council, with a view in particular to drawing up the non-binding manual for practitioners referred to in Article 6(2)(a): (a) the list of national offences in each of the categories referred to in the table of offences in Annex A. The list shall include the name or legal classification of the offence and reference to the applicable legal provisions. It may also include a short description of the constitutive elements of the offence; (b) the list of types of sentences, possible supplementary penalties and security measures and possible subsequent decisions modifying the enforcement of the sentence as defined in national law, in each of the categories referred to in the table of penalties and measures in Annex B. It may also include a short description of the specific penalty or measure. 2. The lists and descriptions referred to in paragraph 1 shall be regularly updated by Member States. Updated information shall be sent to the General Secretariat of the Council. 3. The General Secretariat of the Council shall communicate to the Member States and to the Commission the information received pursuant to this Article. Article 6 Implementing measures 1. The Council, acting by a qualified majority and after consulting the European Parliament, shall adopt any modifications of Annexes A and B as may be necessary. 2. The representatives of the relevant departments of the administrations of the Member States and the Commission shall inform and consult one another within the Council with a view to: (a) drawing up a non-binding manual for practitioners setting out the procedure for the exchange of information through ECRIS, addressing in particular the modalities of identification of offenders, as well as recording the common understanding of the categories of offences and penalties and measures listed respectively in Annexes A and B; (b) coordinating their action for the development and operation of ECRIS, concerning in particular: (i) the establishment of logging systems and procedures making it possible to monitor the functioning of ECRIS and the establishment of non-personal statistics relating to the exchange through ECRIS of information extracted from criminal records; (ii) the adoption of technical specifications of the exchange, including security requirements, in particular the common set of protocols; (iii) the establishment of procedures verifying the conformity of the national software applications with the technical specifications. Article 7 Report The Commission services shall regularly publish a report concerning the exchange, through ECRIS, of information extracted from the criminal record based in particular on the statistics referred to in Article 6(2)(b)(i). This report shall be published for the first time one year after submitting the report referred to in Article 13(3) of Framework Decision 2009/315/JHA. Article 8 Implementation and time limits 1. Member States shall take the necessary measures to comply with the provisions of this Decision by 7 April 2012. 2. Member States shall use the format specified in Article 4 and comply with the means of organising and facilitating exchanges of information laid down in this Decision from the date notified in accordance with Article 11(6) of Framework Decision 2009/315/JHA. Article 9 Taking of effect This Decision shall take effect on the day of its publication in the Official Journal of the European Union. Done at Luxembourg, 6 April 2009. For the Council The President J. POSPÃ Ã IL (1) See page 23 of this Official Journal. (2) Opinion delivered on 9 October 2008 (not yet published in the Official Journal). (3) OJ C 53, 3.3.2005, p. 1. (4) OJ C 198, 12.8.2005, p. 1. (5) OJ L 350, 30.12.2008, p. 60. ANNEX A Common table of offences categories referred to in Article 4 Parameters Level of completion: Completed act C Attempt or preparation A Non-transmitted element Ã Level of participation: Perpetrator M Aider and abettor or instigator/organiser, conspirator H Non-transmitted element Ã Exemption from criminal responsibility: Insanity or diminished responsibility S Recidivism R Code Categories and sub-categories of offences 0100 00 open category Crimes within the jurisdiction of the International Criminal Court 0101 00 Genocide 0102 00 Crimes against humanity 0103 00 War crimes 0200 00 open category Participation in a criminal organisation 0201 00 Directing a criminal organisation 0202 00 Knowingly taking part in the criminal activities of a criminal organisation 0203 00 Knowingly taking part in the non-criminal activities of a criminal organisation 0300 00 open category Terrorism 0301 00 Directing a terrorist group 0302 00 Knowingly participating in the activities of a terrorist group 0303 00 Financing of terrorism 0304 00 Public provocation to commit a terrorist offence 0305 00 Recruitment or training for terrorism 0400 00 open category Trafficking in human beings 0401 00 Trafficking in human beings for the purposes of labour or services exploitation 0402 00 Trafficking in human beings for the purposes of the exploitation of the prostitution of others or other forms of sexual exploitation 0403 00 Trafficking in human beings for the purposes of organ or human tissue removal 0404 00 Trafficking in human beings for the purpose of slavery, practices similar to slavery or servitude 0405 00 Trafficking in human beings for the purposes of labour or services exploitation of a minor 0406 00 Trafficking in human beings for the purposes of the exploitation of the prostitution of minors or other forms of their sexual exploitation 0407 00 Trafficking in human beings for the purposes of organ or human tissue removal of a minor 0408 00 Trafficking in human beings for the purpose of slavery, practices similar to slavery or servitude of a minor 0500 00 open category Illicit trafficking (1) and other offences related to weapons, firearms, their parts and components, ammunition and explosives 0501 00 Illicit manufacturing of weapons, firearms, their parts and components, ammunition and explosives 0502 00 Illicit trafficking of weapons, firearms, their parts and components ammunition and explosives at national level (2) 0503 00 Illicit exportation or importation of weapons, firearms, their parts and components, ammunition and explosives 0504 00 Unauthorised possession or use of weapons, firearms, their parts and components, ammunition and explosives 0600 00 open category Environmental crime 0601 00 Destroying or damaging protected fauna and flora species 0602 00 Unlawful discharges of polluting substances or ionising radiation into air, soil or water 0603 00 Offences related to waste, including hazardous waste 0604 00 Offences related to illicit trafficking (1) in protected fauna and flora species or parts thereof 0605 00 Unintentional environmental offences 0700 00 open category Offences related to drugs or precursors, and other offences against public health 0701 00 Offences related to illicit trafficking (3) in narcotic drugs, psychotropic substances and precursors not exclusively for own personal consumption 0702 00 Illicit consumption of drugs and their acquisition, possession, manufacture or production exclusively for own personal consumption 0703 00 Aiding or inciting others to use narcotic drugs or psychotropic substances illicitly 0704 00 Manufacture or production of narcotic drugs not exclusively for personal consumption 0800 00 open category Crimes against the person 0801 00 Intentional killing 0802 00 Aggravated cases of intentional killing (4) 0803 00 Unintentional killing 0804 00 Intentional killing of a new-born by his/her mother 0805 00 Illegal abortion 0806 00 Illegal euthanasia 0807 00 Offences related to committing suicide 0808 00 Violence causing death 0809 00 Causing grievous bodily injury, disfigurement or permanent disability 0810 00 Unintentionally causing grievous bodily injury, disfigurement or permanent disability 0811 00 Causing minor bodily injury 0812 00 Unintentionally causing minor bodily injury 0813 00 Exposing to danger of loss of life or grievous bodily injury 0814 00 Torture 0815 00 Failure to offer aid or assistance 0816 00 Offences related to organ or tissue removal without authorisation or consent 0817 00 Offences related to illicit trafficking (3) in human organs and tissue 0818 00 Domestic violence or threat 0900 00 open category Offences against personal liberty, dignity and other protected interests, including racism and xenophobia 0901 00 Kidnapping, kidnapping for ransom, illegal restraint 0902 00 Unlawful arrest or deprivation of liberty by public authority 0903 00 Hostage-taking 0904 00 Unlawful seizure of an aircraft or ship 0905 00 Insults, slander, defamation, contempt 0906 00 Threats 0907 00 Duress, pressure, stalking, harassment or aggression of a psychological or emotional nature 0908 00 Extortion 0909 00 Aggravated extortion 0910 00 Illegal entry into private property 0911 00 Invasion of privacy other than illegal entry into private property 0912 00 Offences against protection of personal data 0913 00 Illegal interception of data or communication 0914 00 Discrimination on grounds of gender, race, sexual orientation, religion or ethnic origin 0915 00 Public incitement to racial discrimination 0916 00 Public incitement to racial hatred 0917 00 Blackmail 1000 00 open category Sexual offences 1001 00 Rape 1002 00 Aggravated rape (5) other than rape of a minor 1003 00 Sexual assault 1004 00 Procuring for prostitution or sexual act 1005 00 Indecent exposure 1006 00 Sexual harassment 1007 00 Soliciting by a prostitute 1008 00 Sexual exploitation of children 1009 00 Offences related to child pornography or indecent images of minors 1010 00 Rape of a minor 1011 00 Sexual assault of a minor 1100 00 open category Offences against family law 1101 00 Illicit sexual relations between close family members 1102 00 Polygamy 1103 00 Evading the alimony or maintenance obligation 1104 00 Neglect or desertion of a minor or a disabled person 1105 00 Failure to comply with an order to produce a minor or removal of a minor 1200 00 open category Offences against the State, public order, course of justice or public officials 1201 00 Espionage 1202 00 High treason 1203 00 Offences related to elections and referendum 1204 00 Attempt against life or health of the Head of State 1205 00 Insult of the State, Nation or State symbols 1206 00 Insult or resistance to a representative of public authority 1207 00 Extortion, duress, pressure towards a representative of public authority 1208 00 Assault or threat on a representative of public authority 1209 00 Public order offences, breach of the public peace 1210 00 Violence during sports events 1211 00 Theft of public or administrative documents 1212 00 Obstructing or perverting the course of justice, making false allegations in the course of criminal or judicial proceedings, perjury 1213 00 Unlawful impersonation of a person or an authority 1214 00 Escape from lawful custody 1300 00 open category Offences against public property or public interests 1301 00 Public, social security or family benefit fraud 1302 00 Fraud affecting European benefits or allowances 1303 00 Offences related to illegal gambling 1304 00 Obstructing of public tender procedures 1305 00 Active or passive corruption of a civil servant, a person holding public office or public authority 1306 00 Embezzlement, misappropriation or other diversion of property by a public official 1307 00 Abuse of a function by a public official 1400 00 open category Tax and customs offences 1401 00 Tax offences 1402 00 Customs offences 1500 00 open category Economic and trade related offences 1501 00 Bankruptcy or fraudulent insolvency 1502 00 Breach of accounting regulation, embezzlement, concealment of assets or unlawful increase in a companys liabilities 1503 00 Violation of competition rules 1504 00 Laundering of proceeds from crime 1505 00 Active or passive corruption in the private sector 1506 00 Revealing a secret or breaching an obligation of secrecy 1507 00 Insider trading 1600 00 open category Offences against property or causing damage to goods 1601 00 Unlawful appropriation 1602 00 Unlawful appropriation or diversion of energy 1603 00 Fraud, including swindling 1604 00 Dealing in stolen goods 1605 00 Illicit trafficking (6) in cultural goods, including antiques and works of art 1606 00 Intentional damage or destruction of property 1607 00 Unintentional damage or destruction of property 1608 00 Sabotage 1609 00 Offences against industrial or intellectual property 1610 00 Arson 1611 00 Arson causing death or injury to persons 1612 00 Forest arson 1700 00 open category Theft offences 1701 00 Theft 1702 00 Theft after unlawful entry into property 1703 00 Theft, using violence or weapons, or using threat of violence or weapons against person 1704 00 Forms of aggravated theft which do not involve use of violence or weapons, or use of threat of violence or weapons, against persons. 1800 00 open category Offences against information systems and other computer-related crime 1801 00 Illegal access to information systems 1802 00 Illegal system interference 1803 00 Illegal data interference 1804 00 Production, possession, dissemination of or trafficking in computer devices or data enabling commitment of computer-related offences 1900 00 open category Forgery of means of payment 1901 00 Counterfeiting or forging currency, including the euro 1902 00 Counterfeiting of non-cash means of payment 1903 00 Counterfeiting or forging public fiduciary documents 1904 00 Putting into circulation/using counterfeited or forged currency, non-cash means of payment or public fiduciary documents 1905 00 Possession of a device for the counterfeiting or forgery of currency or public fiduciary documents 2000 00 open category Falsification of documents 2001 00 Falsification of a public or administrative document by a private individual 2002 00 Falsification of a document by a civil servant or a public authority 2003 00 Supply or acquisition of a forged public or administrative document; supply or acquisition of a forged document by a civil servant or a public authority 2004 00 Using forged public or administrative documents 2005 00 Possession of a device for the falsification of public or administrative documents 2006 00 Forgery of private documents by a private individual 2100 00 open category Offences against traffic regulations 2101 00 Dangerous driving 2102 00 Driving under the influence of alcohol or narcotic drugs 2103 00 Driving without a licence or while disqualified 2104 00 Failure to stop after a road accident 2105 00 Avoiding a road check 2106 00 Offences related to road transport 2200 00 open category Offences against labour law 2201 00 Unlawful employment 2202 00 Offences relating to remuneration, including social security contributions 2203 00 Offences relating to working conditions, health and safety at work 2204 00 Offences relating to access to or exercise of a professional activity 2205 00 Offences relating to working hours and rest time 2300 00 open category Offences against migration law 2301 00 Unauthorised entry or residence 2302 00 Facilitation of unauthorised entry and residence 2400 00 open category Offences against military obligations 2500 00 open category Offences related to hormonal substances and other growth promoters 2501 00 Illicit importation, exportation or supply of hormonal substances and other grown promoters 2600 00 open category Offences related to nuclear materials or other hazardous radioactive substances 2601 00 Illicit importation, exportation, supply or acquisition of nuclear or radioactive materials 2700 00 open category Other offences 2701 00 Other intentional offences 2702 00 Other unintentional offences (1) Unless otherwise specified in this category, trafficking means import, export, acquisition, sale, delivery, movement or transfer. (2) For the purpose of this sub-category trafficking includes acquisition, sale, delivery, movement or transfer. (3) For the purpose of this sub-category trafficking includes import, export, acquisition, sale, delivery, movement or transfer. (4) For example: particularly grave circumstances. (5) For example rape with particular cruelty. (6) Trafficking includes import, export, acquisition, sale, delivery, movement or transfer. ANNEX B Common table of penalties and measures categories referred to in Article 4 Code Categories and sub-categories of offences 1000 open category Deprivation of freedom 1001 Imprisonment 1002 Life imprisonment 2000 open category Restriction of personal freedom 2001 Prohibition from frequenting some places 2002 Restriction to travel abroad 2003 Prohibition to stay in some places 2004 Prohibition from entry to a mass event 2005 Prohibition to enter in contact with certain persons through whatever means 2006 Placement under electronic surveillance (1) 2007 Obligation to report at specified times to a specific authority 2008 Obligation to stay/reside in a certain place 2009 Obligation to be at the place of residence on the set time 2010 Obligation to comply with the probation measures ordered by the court, including the obligation to remain under supervision 3000 open category Prohibition of a specific right or capacity 3001 Disqualification from function 3002 Loss/suspension of capacity to hold or to be appointed to public office 3003 Loss/suspension of the right to vote or to be elected 3004 Incapacity to contract with public administration 3005 Ineligibility to obtain public subsidies 3006 Cancellation of the driving licence (2) 3007 Suspension of driving licence 3008 Prohibition to drive certain vehicles 3009 Loss/suspension of the parental authority 3010 Loss/suspension of right to be an expert in court proceedings/witness under oath/juror 3011 Loss/suspension of right to be a legal guardian (3) 3012 Loss/suspension of right of decoration or title 3013 Prohibition to exercise professional, commercial or social activity 3014 Prohibition from working or activity with minors 3015 Obligation to close an establishment 3016 Prohibition to hold or to carry weapons 3017 Withdrawal of a hunting/fishing license 3018 Prohibition to issue cheques or to use payment/credit cards 3019 Prohibition to keep animals 3020 Prohibition to possess or use certain items other than weapons 3021 Prohibition to play certain games/sports 4000 open category Prohibition or expulsion from territory 4001 Prohibition from national territory 4002 Expulsion from national territory 5000 open category Personal obligation 5001 Submission to medical treatment or other forms of therapy 5002 Submission to a social-educational programme 5003 Obligation to be under the care/control of the family 5004 Educational measures 5005 Socio-judicial probation 5006 Obligation of training/working 5007 Obligation to provide judicial authorities with specific information 5008 Obligation to publish the judgment 5009 Obligation to compensate for the prejudice caused by the offence 6000 open category Penalty on personal property 6001 Confiscation 6002 Demolition 6003 Restoration 7000 open category Placing in an institution 7001 Placing in a psychiatric institution 7002 Placing in a detoxification institution 7003 Placing in an educational institution 8000 open category Financial penalty 8001 Fine 8002 Day-fine (4) 8003 Fine for the benefit of a special recipient (5) 9000 open category Working penalty 9001 Community service or work 9002 Community service or work accompanied with other restrictive measures 10000 open category Military penalty 10001 Loss of military rank (6) 10002 Expulsion from professional military service 10003 Military imprisonment 11000 open category Exemption/deferment of sentence/penalty, warning 12000 open category Other penalties and measures Parameters (to be specified where applicable) Ã ¸ Penalty m Measure a Suspended penalty/measure b Partially suspended penalty/measure c Suspended penalty/measure with probation/supervision d Partially suspended penalty/measure with probation/supervision e Conversion of penalty/measure f Alternative penalty/measure imposed as principal penalty g Alternative penalty/measure imposed initially in case of non-respect of the principal penalty h Revocation of suspended penalty/measure i Subsequent formation of an overall penalty j Interruption of enforcement/postponement of the penalty/measure (7) k Remission of the penalty l Remission of the suspended penalty n End of penalty o Pardon p Amnesty q Release on parole (liberation of a person before end of the sentence under certain conditions) r Rehabilitation (with or without the deletion of penalty from criminal records) s Penalty or measure specific to minors t Non-criminal ruling (8) (1) Fixed or mobile placement. (2) Reapplication in order to obtain a new driving licence is necessary. (3) Legal guardian for a person who is legally incompetent or for a minor. (4) Fine expressed in daily units. (5) E.g.: for an institution, association, foundation or a victim. (6) Military demotion. (7) Does not lead to avoidance of enforcement of penalty. (8) This parameter will be indicated only when such information is provided in reply to the request received by the Member State of nationality of the person concerned.
============================== "END OF DOC" ==============================
        

        doc 2 :

        'celex': 32005F0214
        'status': In Force
        'act_type': Decision_FRAMW
        'treaty': TEU (1992)

        full_doc :

        22.3.2005 EN Official Journal of the European Union L 76/16 COUNCIL FRAMEWORK DECISION 2005/214/JHA of 24 February 2005 on the application of the principle of mutual recognition to financial penalties THE COUNCIL OF THE EUROPEAN UNION, Having regard to the Treaty on European Union, and in particular Articles 31(a) and 34(2)(b) thereof, Having regard to the initiative of the United Kingdom of Great Britain and Northern Ireland, the French Republic and the Kingdom of Sweden (1), Having regard to the opinion of the European Parliament (2), Whereas: (1) The European Council meeting in Tampere on 15 and 16 October 1999 endorsed the principle of mutual recognition, which should become the cornerstone of judicial cooperation in both civil and criminal matters within the Union. (2) The principle of mutual recognition should apply to financial penalties imposed by judicial or administrative authorities for the purpose of facilitating the enforcement of such penalties in a Member State other than the State in which the penalties are imposed. (3) On 29 November 2000 the Council, in accordance with the Tampere conclusions, adopted a programme of measures to implement the principle of mutual recognition of decisions in criminal matters (3), giving priority to the adoption of an instrument applying the principle of mutual recognition to financial penalties (measure 18). (4) This Framework Decision should also cover financial penalties imposed in respect of road traffic offences. (5) This Framework Decision respects fundamental rights and observes the principles recognised by Article 6 of the Treaty and reflected by the Charter of Fundamental Rights of the European Union (4), in particular Chapter VI thereof. Nothing in this Framework Decision may be interpreted as prohibiting refusal to execute a decision when there are reasons to believe, on the basis of objective elements, that the financial penalty has the purpose of punishing a person on the grounds of his or her sex, race, religion, ethnic origin, nationality, language, political opinions or sexual orientation, or that that person's position may be prejudiced for any of these reasons. (6) This Framework Decision does not prevent a Member State from applying its constitutional rules relating to due process, freedom of association, freedom of the press and freedom of expression in other media, HAS ADOPTED THIS FRAMEWORK DECISION: Article 1 Definitions For the purposes of this Framework Decision: (a) decision shall mean a final decision requiring a financial penalty to be paid by a natural or legal person where the decision was made by: (i) a court of the issuing State in respect of a criminal offence under the law of the issuing State; (ii) an authority of the issuing State other than a court in respect of a criminal offence under the law of the issuing State, provided that the person concerned has had an opportunity to have the case tried by a court having jurisdiction in particular in criminal matters; (iii) an authority of the issuing State other than a court in respect of acts which are punishable under the national law of the issuing State by virtue of being infringements of the rules of law, provided that the person concerned has had an opportunity to have the case tried by a court having jurisdiction in particular in criminal matters; (iv) a court having jurisdiction in particular in criminal matters, where the decision was made regarding a decision as referred to in point (iii); (b) financial penalty shall mean the obligation to pay: (i) a sum of money on conviction of an offence imposed in a decision; (ii) compensation imposed in the same decision for the benefit of victims, where the victim may not be a civil party to the proceedings and the court is acting in the exercise of its criminal jurisdiction; (iii) a sum of money in respect of the costs of court or administrative proceedings leading to the decision; (iv) a sum of money to a public fund or a victim support organisation, imposed in the same decision. A financial penalty shall not include:  orders for the confiscation of instrumentalities or proceeds of crime,  orders that have a civil nature and arise out of a claim for damages and restitution and which are enforceable in accordance with Council Regulation (EC) No 44/2001 of 22 December 2000 on jurisdiction and the recognition and enforcement of judgments in civil and commercial matters (5); (c) issuing State shall mean the Member State in which a decision within the meaning of this Framework Decision was delivered; (d) executing State shall mean the Member State to which a decision has been transmitted for the purpose of enforcement. Article 2 Determination of the competent authorities 1. Each Member State shall inform the General Secretariat of the Council which authority or authorities, under its national law, are competent according to this Framework Decision, when that Member State is the issuing State or the executing State. 2. Notwithstanding Article 4, each Member State may designate, if it is necessary as a result of the organisation of its internal system, one or more central authorities responsible for the administrative transmission and reception of the decisions and to assist the competent authorities. 3. The General Secretariat of the Council shall make the information received available to all Member States and the Commission. Article 3 Fundamental rights This Framework Decision shall not have the effect of amending the obligation to respect fundamental rights and fundamental legal principles as enshrined in Article 6 of the Treaty. Article 4 Transmission of decisions and recourse to the central authority 1. A decision, together with a certificate as provided for in this Article, may be transmitted to the competent authorities of a Member State in which the natural or legal person against whom a decision has been passed has property or income, is normally resident or, in the case of a legal person, has its registered seat. 2. The certificate, the standard form for which is given in the Annex, must be signed, and its contents certified as accurate, by the competent authority in the issuing State. 3. The decision or a certified copy of it, together with the certificate, shall be transmitted by the competent authority in the issuing State directly to the competent authority in the executing State by any means which leaves a written record under conditions allowing the executing State to establish its authenticity. The original of the decision, or a certified copy of it, and the original of the certificate, shall be sent to the executing State if it so requires. All official communications shall also be made directly between the said competent authorities. 4. The issuing State shall only transmit a decision to one executing State at any one time. 5. If the competent authority in the executing State is not known to the competent authority in the issuing State, the latter shall make all necessary inquiries, including via the contact points of the European Judicial Network (6) in order to obtain the information from the executing State. 6. When an authority in the executing State which receives a decision has no jurisdiction to recognise it and take the necessary measures for its execution, it shall, ex officio, transmit the decision to the competent authority and shall inform the competent authority in the issuing State accordingly. 7. The United Kingdom and Ireland, respectively, may state in a declaration that the decision together with the certificate must be sent via its central authority or authorities specified by it in the declaration. These Member States may at any time by a further declaration limit the scope of such a declaration for the purpose of giving greater effect to paragraph 3. They shall do so when the provisions on mutual assistance of the Schengen Implementation Convention are put into effect for them. Any declaration shall be deposited with the General Secretariat of the Council and notified to the Commission. Article 5 Scope 1. The following offences, if they are punishable in the issuing State and as they are defined by the law of the issuing State, shall, under the terms of this Framework Decision and without verification of the double criminality of the act, give rise to recognition and enforcement of decisions:  participation in a criminal organisation,  terrorism,  trafficking in human beings,  sexual exploitation of children and child pornography,  illicit trafficking in narcotic drugs and psychotropic substances,  illicit trafficking in weapons, munitions and explosives,  corruption,  fraud, including that affecting the financial interests of the European Communities within the meaning of the Convention of 26 July 1995 on the protection of the European Communities' financial interests,  laundering of the proceeds of crime,  counterfeiting currency, including of the euro,  computer-related crime,  environmental crime, including illicit trafficking in endangered animal species and in endangered plant species and varieties,  facilitation of unauthorised entry and residence,  murder, grievous bodily injury,  illicit trade in human organs and tissue,  kidnapping, illegal restraint and hostage-taking,  racism and xenophobia,  organised or armed robbery,  illicit trafficking in cultural goods, including antiques and works of art,  swindling,  racketeering and extortion,  counterfeiting and piracy of products,  forgery of administrative documents and trafficking therein,  forgery of means of payment,  illicit trafficking in hormonal substances and other growth promoters,  illicit trafficking in nuclear or radioactive materials,  trafficking in stolen vehicles,  rape,  arson,  crimes within the jurisdiction of the International Criminal Court,  unlawful seizure of aircraft/ships,  sabotage,  conduct which infringes road traffic regulations, including breaches of regulations pertaining to driving hours and rest periods and regulations on hazardous goods,  smuggling of goods,  infringements of intellectual property rights,  threats and acts of violence against persons, including violence during sport events,  criminal damage,  theft,  offences established by the issuing State and serving the purpose of implementing obligations arising from instruments adopted under the EC Treaty or under Title VI of the EU Treaty. 2. The Council may decide to add other categories of offences to the lists in paragraph 1 at any time, acting unanimously after consultation of the European Parliament under the conditions laid down in Article 39(1) of the EU Treaty. The Council shall consider, in the light of the report submitted to it pursuant to Article 20(5), whether the list should be extended or amended. The Council shall consider the issue further at a later stage on the basis of a report on the practical application of the Framework Decision established by the Commission within 5 years after the date mentioned in Article 20(1). 3. For offences other than those covered by paragraph 1, the executing State may make the recognition and execution of a decision subject to the condition that the decision is related to conduct which would constitute an offence under the law of the executing State, whatever the constituent elements or however it is described. Article 6 Recognition and execution of decisions The competent authorities in the executing State shall recognise a decision which has been transmitted in accordance with Article 4 without any further formality being required and shall forthwith take all the necessary measures for its execution, unless the competent authority decides to invoke one of the grounds for non-recognition or non-execution provided for in Article 7. Article 7 Grounds for non-recognition and non-execution 1. The competent authorities in the executing State may refuse to recognise and execute the decision if the certificate provided for in Article 4 is not produced, is incomplete or manifestly does not correspond to the decision. 2. The competent authority in the executing State may also refuse to recognise and execute the decision if it is established that: (a) decision against the sentenced person in respect of the same acts has been delivered in the executing State or in any State other than the issuing or the executing State, and, in the latter case, that decision has been executed; (b) in one of the cases referred to in Article 5(3), the decision relates to acts which would not constitute an offence under the law of the executing State; (c) the execution of the decision is statute-barred according to the law of the executing State and the decision relates to acts which fall within the jurisdiction of that State under its own law. (d) the decision relates to acts which: (i) are regarded by the law of the executing State as having been committed in whole or in part in the territory of the executing State or in a place treated as such, or (ii) have been committed outside the territory of the issuing State and the law of the executing State does not allow prosecution for the same offences when committed outside its territory; (e) there is immunity under the law of the executing State, which makes it impossible to execute the decision; (f) the decision has been imposed on a natural person who under the law of the executing State due to his or her age could not yet have been held criminally liable for the acts in respect of which the decision was passed; (g) according to the certificate provided for in Article 4, the person concerned (i) in case of a written procedure was not, in accordance with the law of the issuing State, informed personally or via a representative, competent according to national law, of his right to contest the case and of time limits of such a legal remedy, or (ii) did not appear personally, unless the certificate states:  that the person was informed personally, or via a representative, competent according to national law, of the proceedings in accordance with the law of the issuing State, or  that the person has indicated that he or she does not contest the case; (h) the financial penalty is below EUR 70 or the equivalent to that amount. 3. In cases referred to in paragraphs 1 and 2(c) and (g), before deciding not to recognise and to execute a decision, either totally or in part, the competent authority in the executing State shall consult the competent authority in the issuing State, by any appropriate means, and shall, where appropriate, ask it to supply any necessary information without delay. Article 8 Determination of the amount to be paid 1. Where it is established that the decision is related to acts which were not carried out within the territory of the issuing State, the executing State may decide to reduce the amount of the penalty enforced to the maximum amount provided for acts of the same kind under the national law of the executing State, when the acts fall within the jurisdiction of that State. 2. The competent authority of the executing State shall, if necessary, convert the penalty into the currency of the executing State at the rate of exchange obtaining at the time when the penalty was imposed. Article 9 Law governing enforcement 1. Without prejudice to paragraph 3 of this Article, and to Article 10, the enforcement of the decision shall be governed by the law of the executing State in the same way as a financial penalty of the executing State. The authorities of the executing State alone shall be competent to decide on the procedures for enforcement and to determine all the measures relating thereto, including the grounds for termination of enforcement. 2. In the case where the sentenced person is able to furnish proof of a payment, totally or in part, in any State, the competent authority of the executing State shall consult the competent authority of the Issuing State in the way provided for in Article 7(3). Any part of the penalty recovered in whatever manner in any State shall be deducted in full from the amount, which is to be enforced in the executing State. 3. A financial penalty imposed on a legal person shall be enforced even if the executing State does not recognise the principle of criminal liability of legal persons. Article 10 Imprisonment or other alternative sanction by way of substitution for non-recovery of the financial penalty Where it is not possible to enforce a decision, either totally or in part, alternative sanctions, including custodial sanctions, may be applied by the executing State if its laws so provide in such cases and the issuing State has allowed for the application of such alternative sanctions in the certificate referred to in Article 4. The severity of the alternative sanction shall be determined in accordance with the law of the executing State, but shall not exceed any maximum level stated in the certificate transmitted by the issuing State. Article 11 Amnesty, pardon, review of sentence 1. Amnesty and pardon may be granted by the issuing State and also by the executing State. 2. Without prejudice to the Article 10, only the issuing State may determine any application for review of the decision. Article 12 Termination of enforcement 1. The competent authority of the issuing State shall forthwith inform the competent authority of the executing State of any decision or measure as a result of which the decision ceases to be enforceable or is withdrawn from the executing State for any other reason. 2. The executing State shall terminate enforcement of the decision as soon as it is informed by the competent authority of the issuing State of that decision or measure. Article 13 Accrual of monies obtained from enforcement of decisions Monies obtained from the enforcement of decisions shall accrue to the executing State unless otherwise agreed between the issuing and the executing State, in particular in the cases referred to in Article 1(b)(ii). Article 14 Information from the executing State The competent authority of the executing State shall without delay inform the competent authority of the issuing State by any means which leaves a written record: (a) of the transmission of the decision to the competent authority, according to Article 4(6); (b) of any decision not to recognise and execute a decision, according to Articles 7 or 20(3), together with the reasons for the decision; (c) of the total or partial non-execution of the decision for the reasons referred to in Article 8, Article 9(1) and (2), and Article 11(1); (d) of the execution of the decision as soon as the execution has been completed; (e) of the application of alternative sanction, according to Article 10. Article 15 Consequences of transmission of a decision 1. Subject to paragraph 2, the issuing State may not proceed with the execution of a decision transmitted pursuant to Article 4. 2. The right of execution of the decision shall revert to the issuing State: (a) upon it being informed by the executing State of the total or partial non-execution or the non-recognition or the non-enforcement of the decision in the case of Article 7, with the exception of Article 7(2)(a), in the case of Article 11(1), and in the case of Article 20(3); or (b) when the executing State has been informed by the issuing State that the decision has been withdrawn from the executing State pursuant to Article 12. 3. If, after transmission of a decision in accordance with Article 4, an authority of the issuing State receives any sum of money which the sentenced person has paid voluntarily in respect of the decision, that authority shall inform the competent authority in the executing State without delay. Article 9(2) shall apply. Article 16 Languages 1. The certificate, the standard form for which is given in the Annex, must be translated into the official language or one of the official languages of the executing State. Any Member State may, either when this Framework Decision is adopted or at a later date, state in a declaration deposited with the General Secretariat of the Council that it will accept a translation in one or more other official languages of the Institutions of the Union. 2. The execution of the decision may be suspended for the time necessary to obtain its translation at the expense of the executing State. Article 17 Costs Member States shall not claim from each other the refund of costs resulting from application of this Framework Decision. Article 18 Relationship with other agreements and arrangements This Framework Decision shall not preclude the application of bilateral or multilateral agreements or arrangements between Member States in so far as such agreements or arrangements allow the prescriptions of this Framework Decision to be exceeded and help to simplify or facilitate further the procedures for the enforcement of financial penalties. Article 19 Territorial application This Framework Decision shall apply to Gibraltar. Article 20 Implementation 1. Member States shall take the necessary measures to comply with the provisions of this Framework Decision by 22 March 2007. 2. Each Member State may for a period of up to five years from the date of entry into force of this Framework Decision limit its application to: (a) decisions mentioned in Article 1 (a)(i) and (iv); and/or (b) with regard to legal persons, decisions related to conduct for which a European instrument provides for the application of the principle of liability of legal persons. Any Member State that wants to make use of this paragraph, shall notify a declaration to that effect to the Secretary General of the Council upon the adoption of this Framework Decision. The declaration shall be published in the Official Journal of the European Union. 3. Each Member State may, where the certificate referred to in Article 4 gives rise to an issue that fundamental rights or fundamental legal principles as enshrined in Article 6 of the Treaty may have been infringed, oppose the recognition and the execution of decisions. The procedure referred to in Article 7(3) shall apply. 4. Any Member State may apply the principle of reciprocity in relation to any Member State making use of paragraph 2. 5. Member States shall transmit to the General Secretariat of the Council and to the Commission the text of the provisions transposing into their national law the obligations imposed on them under this Framework Decision. On the basis of a report established on the basis of this information by the Commission, the Council shall, no later than 22 March 2008, assess the extent to which Member States have complied with this Framework Decision. 6. The General Secretariat of the Council shall notify the Member States and the Commission of the declarations made pursuant to Articles 4(7) and 16. 7. Without prejudice to Article 35(7) of the Treaty, a Member State which has experienced repeated difficulties or lack of activity by another Member State in the mutual recognition and execution of decisions, which have not been solved through bilateral consultations, may inform the Council with a view to evaluating the implementation of this Framework Decision at Member State level. 8. Any Member State which during a calendar year has applied paragraph 3, shall in the beginning of the following calendar year inform the Council and the Commission of cases in which the grounds referred to in that provision for non-recognition or non-execution of a decision have been applied. 9. Within seven years after the entry into force of this Framework Decision, the Commission shall establish a report on the basis of the information received, accompanied by any initiatives it may deem appropriate. The Council shall on the basis of the report review this Article with a view to considering whether paragraph 3 shall be retained or replaced by a more specific provision. Article 21 Entry into force This Framework Decision shall enter into force on the day of its publication in the Official Journal of the European Union. Done at Brussels, 24 February 2005. For the Council The President N. SCHMIT (1) OJ C 278, 2.10.2001, p. 4. (2) OJ C 271 E, 7.11.2002, p. 423. (3) OJ C 12, 15.1.2001, p. 10. (4) OJ C 364, 18.12.2000, p. 1. (5) OJ L 12, 16.1.2001, p. 1. Regulation as last amended by Commission Regulation (EC) No 2245/2004 (OJ L 381, 28.12.2004, p. 10). (6) Council Joint Action 98/428/JHA of 29 June 1998 on the creation of a European Judicial Network (OJ L 191, 7.7.1998, p. 4). ANNEX CERTIFICATE referred to in Article 4 of Council Framework Decision 2005/214/JHA on the application of the principle of mutual recognition to financial penalties (a) * Issuing State: * Executing State: (b) The authority which issued the decision imposing the financial penalty: Official name: Address: File reference ( ¦) Tel. No: (country code) (area/city code) Fax No (country code) (area/city code) E-mail (when available) Languages in which it is possible to communicate with the issuing authority Contact details for person(s) to contact to obtain additional information for the purpose of the enforcement of the decision or, where applicable, for the purpose of the transfer to the issuing State of monies obtained from the enforcement (name, title/grade, tel. No., fax No., and, when available, E-mail) (c) The authority competent for the enforcement of the decision imposing the financial penalty in the issuing State (if the authority is different from the authority under point (b)): Official name: Address: Tel. No: (country code) (area/city code) Fax No (country code) (area/city code) E-mail (when available) Languages in which it is possible to communicate with the authority competent for the enforcement Contact details for person(s) to contact to obtain additional information for the purpose of the enforcement of the decision or, where applicable, for the purpose of the transfer to the issuing State of monies obtained from the enforcement (name, title/grade, tel. No., fax No., and, when available, E-mail): (d) Where a central authority has been made responsible for the administrative transmission of decisions imposing financial penalties in the issuing State: Name of the central authority: Contact person, if applicable (title/grade and name): Address: File reference Tel. No: (country code) (area/city code) Fax No: (country code) (area/city code) E-mail (when available): (e) The authority or authorities which may be contacted (in the case where point (c) and/or (d) has been filled): Ã¯   Authority mentioned under point (b) Can be contacted for questions concerning: Ã¯   Authority mentioned under point (c) Can be contacted for questions concerning: Ã¯   Authority mentioned under point (d) Can be contacted for questions concerning: (f) Information regarding the natural or legal person on which the financial penalty has been imposed: 1. In case of a natural person Name: Forename(s): Maiden name, where applicable: Aliases, where applicable: Sex: Nationality: Identity number or social security number (when available): Date of birth: Place of birth: Last known address: Language(s) which the person understands (if known): (a) If the decision is transmitted to the executing State because the person against whom the decision has been passed is normally resident, add the following information: Normal residence in the executing State: (b) If the decision is transmitted to the executing State because the person against whom the decision has been passed has property in the executing State, add the following information: Description of the property of the person: Location of the property of the person: (c) If the decision is transmitted to the executing State because the person against whom the decision has been passed has income in the executing State, add the following information: Description of the source(s) of income of the person: Location of the source(s) of income of the person: 2. In case of a legal person: Name: Form of legal person: Registration number (if available) (1): Registered seat (if available) (1): Address of the legal person: (a) If the decision is transmitted to the executing State because the legal person against whom the decision has been passed has property in the executing State, add the following information: Description of the property of the legal person: Location of the property of the legal person: (b) If the decision is transmitted to the executing State because the legal person against whom the decision has been passed has income in the executing State, add the following information: Description of the source(s) of income of the legal person: Location of the source(s) of income of the legal person: (g) The decision imposing a financial penalty: 1. The nature of the decision imposing the financial penalty (tick the relevant box): Ã¯   (i) Decision of a court of the issuing State in respect of a criminal offence under the law of the issuing State Ã¯   (ii) Decision of an authority of the issuing State other than a court in respect of a criminal offence under the law of the issuing State. It is confirmed that the person concerned has had an opportunity to have the case tried by a court having jurisdiction in particular in criminal matters. Ã¯   (iii) Decision of an authority of the issuing State other than a court in respect of acts which are punishable under the national law of the issuing State by virtue of being infringements of the rules of law. It is confirmed that the person concerned has had an opportunity to have the case tried by a court having jurisdiction in particular in criminal matters. Ã¯   (iv) Decision of a court having jurisdiction in particular in criminal matters regarding a decision as referred to in point iii. The decision was made on (date) The decision became final on (date) Reference number of the decision (if available): The financial penalty constitutes an obligation to pay (tick the relevant box(es) and indicate the amount(s) with indication of currency): Ã¯   (i) A sum of money on conviction of an offence imposed in a decision. Amount: Ã¯   (ii) Compensation imposed in the same decision for the benefit of victims, where the victim may not be a civil party to the proceedings and the court is acting in its exercise of its criminal jurisdiction. Amount: Ã¯   (iii) A sum of money in respect of the costs of court or administrative proceedings leading to the decision. Amount: Ã¯   (iv) A sum of money to a public fund or a victim support organisation, imposed in the same decision. Amount: The total amount of the financial penalty with indication of currency: 2. A summary of facts and a description of the circumstances in which the offence(s) has(have) been committed, including time and place: Nature and legal classification of the offence(s) and the applicable statutory provision/code on basis of which the decision was made: 3. To the extent that the offence(s) identified under point 2 above constitute(s) one or more of the following offences, confirm that by ticking the relevant box(es): Ã¯   participation in a criminal organisation; Ã¯   terrorism; Ã¯   trafficking in human beings; Ã¯   sexual exploitation of children and child pornography; Ã¯   illicit trafficking in narcotic drugs and psychotropic substances; Ã¯   illicit trafficking in weapons, munitions and explosives; Ã¯   corruption; Ã¯   fraud, including that affecting the financial interests of the European Communities within the meaning of the Convention of 26 July 1995 on the protection of the European Communities' financial interests; Ã¯   laundering of the proceeds of crime; Ã¯   counterfeiting currency, including of the euro; Ã¯   computer-related crime; Ã¯   environmental crime, including illicit trafficking in endangered animal species and in endangered plant species and varieties; Ã¯   facilitation of unauthorised entry and residence; Ã¯   murder, grievous bodily injury; Ã¯   illicit trade in human organs and tissue; Ã¯   kidnapping, illegal restraint and hostage-taking; Ã¯   racism and xenophobia; Ã¯   organised or armed robbery; Ã¯   illicit trafficking in cultural goods, including antiques and works of art; Ã¯   swindling; Ã¯   racketeering and extortion; Ã¯   counterfeiting and piracy of products; Ã¯   forgery of administrative documents and trafficking therein; Ã¯   forgery of means of payment; Ã¯   illicit trafficking in hormonal substances and other growth promoters; Ã¯   illicit trafficking in nuclear or radioactive materials; Ã¯   trafficking in stolen vehicles; Ã¯   rape; Ã¯   arson; Ã¯   crimes within the jurisdiction of the International Criminal Court; Ã¯   unlawful seizure of aircraft/ships; Ã¯   sabotage; Ã¯   conduct which infringes road traffic regulations, including breaches of regulations pertaining to driving hours and rest periods and regulations on hazardous goods; Ã¯   smuggling of goods; Ã¯   infringements of intellectual property rights; Ã¯   threats and acts of violence against persons, including violence during sport events; Ã¯   criminal damage; Ã¯   theft; Ã¯   offences established by the issuing State and serving the purpose of implementing obligations arising from instruments adopted under the EC Treaty or under Title VI of the EU Treaty. If this box is ticked, indicate the exact provisions of the instrument adopted on the basis of the EC Treaty or the EU Treaty that the offence relates to: 4. To the extent that the offence(s) identified under point 2 above are not covered by point 3, give a full description of the offence(s) concerned: (h) Status of the decision imposing the financial penalty 1. Confirm that (tick the boxes): Ã¯   (a) the decision is a final decision Ã¯   (b) to the knowledge of the authority issuing the Certificate, a decision against the same person in respect of the same acts has not been delivered in the executing State and that no such decision delivered in any State other than the issuing State or the executing State has been executed. 2. Indicate if the case been subject to a written procedure: Ã¯   (a) No, it has not. Ã¯   (b) Yes, it has. It is confirmed that the person concerned was, in accordance with the law of the issuing State, informed personally or via a representative competent according to national law of his right to contest the case and of time limits of such a legal remedy 3. Indicate if the person concerned appeared personally in the proceedings: Ã¯   (a) Yes, he or she did. Ã¯   (b) No, he or she did not. It is confirmed: Ã¯   that the person was informed personally, or via a representative competent according to national law, of the proceedings in accordance with the law of the issuing State, or Ã¯   that the person has indicated that he or she does not contest the case 4. Partial payment of the penalty If any part of the penalty has already been paid to the issuing State, or, to the knowledge of the authority issuing the Certificate, to any other State, indicate the amount which has been paid: (i) Alternative sanctions, including custodial sanctions 1. State whether the issuing State allows for the application by the executing State of alternative sanctions in case it is not possible to enforce the decision imposing a penalty, either totally or in part: Ã¯   yes Ã¯   no 2. If yes, state which sanctions may be applied (nature of the sanctions, maximum level of the sanctions): Ã¯   Custody. Maximum period: Ã¯   Community service (or equivalent). Maximum period Ã¯   Other sanctions. Description: (j) Other circumstances relevant to the case (optional information): (k) The text of the decision imposing the financial penalty is attached to the certificate. Signature of the authority issuing the certificate and/or its representative certifying the content of the certificate as accurate: Name: Post held (title/grade): Date: Official stamp (if available) (1) Where a decision is transmitted to the executing State because the legal person against whom the decision has been passed has its registered seat in that State, Registration number and Registered seat must be completed.
============================== "END OF DOC" ==============================
        

        doc 3 :

        'celex': 32017L1371
        'status': In Force
        'act_type': Directive
        'treaty': TFEU

        full_doc :

        28.7.2017 EN Official Journal of the European Union L 198/29 DIRECTIVE (EU) 2017/1371 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL of 5 July 2017 on the fight against fraud to the Union's financial interests by means of criminal law THE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE EUROPEAN UNION, Having regard to the Treaty on the Functioning of the European Union, and in particular Article 83(2) thereof, Having regard to the proposal from the European Commission, After transmission of the draft legislative act to the national parliaments, Having regard to the opinion of the Committee of the Regions (1) Acting in accordance with the ordinary legislative procedure (2), Whereas: (1) The protection of the Union's financial interests concerns not only the management of budget appropriations, but extends to all measures which negatively affect or which threaten to negatively affect its assets and those of the Member States, to the extent that those measures are of relevance to Union policies. (2) The Convention drawn up on the basis of Article K.3 of the Treaty on European Union, on the protection of the European Communities' financial interests of 26 July 1995 (3), including the Protocols thereto of 27 September 1996 (4), of 29 November 1996 (5) and of 19 June 1997 (6) (the Convention) establishes minimum rules relating to the definition of criminal offences and sanctions in the area of fraud affecting the Union's financial interests. The Member States drew up the Convention, in which it was noted that fraud affecting Union revenue and expenditure in many cases was not confined to a single country and was often committed by organised criminal networks. On that basis, it was already recognised in the Convention that the protection of the Union's financial interests called for the criminal prosecution of fraudulent conduct injuring those interests. In parallel, Council Regulation (EC, Euratom) No 2988/95 (7) was adopted. That Regulation lays down general rules relating to homogenous checks and to administrative measures and penalties concerning irregularities with regard to Union law while, at the same time, referring to sectoral rules in that area, fraudulent actions as defined in the Convention and the application of the Member States' criminal law and proceedings. (3) Union policy in the area of the protection of the Union's financial interests has already been the subject of harmonisation measures such as Regulation (EC, Euratom) No 2988/95. In order to ensure the implementation of Union policy in this area, it is essential to continue to approximate the criminal law of the Member States by complementing the protection of the Union's financial interests under administrative and civil law for the most serious types of fraud-related conduct in that field, whilst avoiding inconsistencies, both within and among those areas of law. (4) The protection of the Union's financial interests calls for a common definition of fraud falling within the scope of this Directive, which should cover fraudulent conduct with respect to revenues, expenditure and assets at the expense of the general budget of the European Union (the Union budget), including financial operations such as borrowing and lending activities. The notion of serious offences against the common system of value added tax (VAT) as established by Council Directive 2006/112/EC (8) (the common VAT system) refers to the most serious forms of VAT fraud, in particular carrousel fraud, VAT fraud through missing traders, and VAT fraud committed within a criminal organisation, which create serious threats to the common VAT system and thus to the Union budget. Offences against the common VAT system should be considered to be serious where they are connected with the territory of two or more Member States, result from a fraudulent scheme whereby those offences are committed in a structured way with the aim of taking undue advantage of the common VAT system and the total damage caused by the offences is at least EUR 10 000 000. The notion of total damage refers to the estimated damage that results from the entire fraud scheme, both to the financial interests of the Member States concerned and to the Union, excluding interest and penalties. This Directive aims to contribute to the efforts to fight those criminal phenomena. (5) When the Commission implements the Union budget under shared or indirect management, it may delegate budget implementation tasks to the Member States or entrust them to bodies, offices or agencies established pursuant to the Treaties or to other entities or persons. In the event of such shared or indirect management, the Union's financial interests should benefit from the same level of protection as they do when under the direct management of the Commission. (6) For the purposes of this Directive, procurement-related expenditure is any expenditure in connection with the public contracts determined by Article 101(1) of Regulation (EU, Euratom) No 966/2012 of the European Parliament and of the Council (9). (7) Union money laundering law is fully applicable to money laundering involving property derived from the criminal offences covered by this Directive. A reference made to that law should ensure that the sanctioning regime introduced by this Directive applies to all serious cases of criminal offences against the Union's financial interests. (8) Corruption constitutes a particularly serious threat to the Union's financial interests, which can in many cases also be linked to fraudulent conduct. Since all public officials have a duty to exercise judgment or discretion impartially, the giving of bribes in order to influence a public official's judgment or discretion and the taking of such bribes should be included in the definition of corruption, irrespective of the law or regulations applicable in the particular official's country or to the international organisation concerned. (9) The Union's financial interests can be negatively affected by certain types of conduct of a public official who is entrusted with the management of funds or assets, whether he or she is in charge or acts in a supervisory capacity, which types of conduct aim at misappropriating funds or assets, contrary to the intended purpose and whereby the Union's financial interests are damaged. There is therefore a need to introduce a precise definition of criminal offences covering such conduct. (10) As regards the criminal offences of passive corruption and misappropriation, there is a need to include a definition of public officials covering all relevant officials, whether holding a formal office in the Union, in the Member States or in third countries. Private persons are increasingly involved in the management of Union funds. In order to protect Union funds adequately from corruption and misappropriation, the definition of public official therefore needs to cover persons who do not hold formal office but who are nonetheless assigned and exercise, in a similar manner, a public service function in relation to Union funds, such as contractors involved in the management of such funds. (11) With regard to the criminal offences provided for in this Directive, the notion of intention must apply to all the elements constituting those criminal offences. The intentional nature of an act or omission may be inferred from objective, factual circumstances. Criminal offences which do not require intention are not covered by this Directive. (12) This Directive does not oblige Member States to provide for sanctions of imprisonment for the commission of criminal offences that are not of a serious nature, in cases where intent is presumed under national law. (13) Some criminal offences against the Union's financial interests are in practice often closely related to the criminal offences covered by Article 83(1) of the Treaty on the Functioning of the European Union (TFEU) and Union legislative acts that are based on that provision. Coherence between such legislative acts and this Directive should therefore be ensured in the wording of this Directive. (14) Insofar as the Union's financial interests can be damaged or threatened by conduct attributable to legal persons, legal persons should be liable for the criminal offences, as defined in this Directive, which are committed on their behalf. (15) In order to ensure equivalent protection of the Union's financial interests throughout the Union by means of measures which should act as a deterrent, Member States should provide for certain types and levels of sanctions when the criminal offences defined in this Directive are committed. The levels of sanctions should not go beyond what is proportionate for the offences. (16) As this Directive provides for minimum rules, Member States are free to adopt or maintain more stringent rules for criminal offences affecting the Union's financial interests. (17) This Directive does not affect the proper and effective application of disciplinary measures or penalties other than of a criminal nature. Sanctions that cannot be equated to criminal sanctions, which are imposed on the same person for the same conduct, can be taken into account when sentencing that person for a criminal offence defined in this Directive. For other sanctions, the principle of prohibition of being tried or punished twice in criminal proceedings for the same criminal offence (ne bis in idem) should be fully respected. This Directive does not criminalise behaviour which is not also subject to disciplinary penalties or other measures concerning a breach of official duties, in cases where such disciplinary penalties or other measures can be applied to the persons concerned. (18) Sanctions with regard to natural persons should, in certain cases, provide for a maximum penalty of at least four years of imprisonment. Such cases should include at least those involving considerable damage done or advantage gained whereby the damage or advantage should be presumed to be considerable when it involves more than EUR 100 000. Where a Member State's law does not provide for an explicit threshold for considerable damage or advantage as a basis for a maximum penalty, the Member State should ensure that the amount of damage or advantage is taken into account by its courts in the determination of sanctions for fraud and other criminal offences affecting the Union's financial interests. This Directive does not prevent Member States from providing for other elements which would indicate the serious nature of a criminal offence, for instance when the damage or advantage is potential, but of very considerable nature. However, for offences against the common VAT system, the threshold as of which the damage or advantage should be presumed to be considerable is, in conformity with this Directive, EUR 10 000 000. The introduction of minimum levels of maximum imprisonment sanctions is necessary in order to ensure equivalent protection of the Union's financial interests throughout the Union. The sanctions are intended to serve as a strong deterrent for potential offenders, with effect throughout the Union. (19) Member States should ensure that the fact that a criminal offence is committed within a criminal organisation as defined in Council Framework Decision 2008/841/JHA (10) is considered to be an aggravating circumstance in accordance with the applicable rules established by their legal systems. They should ensure that the aggravating circumstance is made available to judges for their consideration when sentencing offenders, although there is no obligation on judges to take the aggravating circumstance into account in their sentence. Member States are not obliged to provide for the aggravating circumstance where national law provides for the criminal offences as defined in Framework Decision 2008/841/JHA to be punishable as a separate criminal offence and this may lead to more severe sanctions. (20) Given, in particular, the mobility of perpetrators and of the proceeds stemming from illegal activities at the expense of the Union's financial interests, as well as the complex cross-border investigations which this entails, each Member State should establish its jurisdiction in order to enable it to counter such activities. Each Member State should thereby ensure that its jurisdiction covers criminal offences which are committed using information and communication technology accessed from its territory. (21) Given the possibility of multiple jurisdictions for cross-border criminal offences falling under the scope of this Directive, the Member States should ensure that the principle of ne bis in idem is respected in full in the application of national law transposing this Directive. (22) Member States should lay down rules concerning limitation periods necessary in order to enable them to counter illegal activities at the expense of the Union's financial interests. In cases of criminal offences punishable by a maximum sanction of at least four years of imprisonment, the limitation period should be at least five years from the time when the criminal offence was committed. This should be without prejudice to those Member States which do not set limitation periods for investigation, prosecution and enforcement. (23) Without prejudice to the rules on cross-border cooperation and mutual legal assistance in criminal matters and to other rules under Union law, in particular under Regulation (EU, Euratom) No 883/2013 of the European Parliament and of the Council (11), there is a need for appropriate provision to be made for cooperation to ensure effective action against the criminal offences defined in this Directive affecting the Union's financial interests, including exchange of information between the Member States and the Commission as well as technical and operational assistance provided by the Commission to the competent national authorities as they may need to facilitate coordination of their investigations. Such assistance should not entail the participation of the Commission in the investigation or prosecution procedures of individual criminal cases conducted by the national authorities. The Court of Auditors and the auditors responsible for auditing the budgets of the Union institutions, bodies, offices and agencies should disclose to the European Anti-Fraud Office (OLAF) and to other competent authorities any fact which could be qualified as a criminal offence under this Directive, and Member States should ensure that national audit bodies within the meaning of Article 59 of Regulation (EU, Euratom) No 966/2012 do the same, in accordance with Article 8 of Regulation (EU, Euratom) No 883/2013. (24) The Commission should report to the European Parliament and to the Council on the measures taken by Member States to comply with this Directive. The report may be accompanied, if necessary, by proposals taking into consideration possible evolutions, in particular regarding the financing of the Union budget. (25) The Convention should be replaced by this Directive for the Member States bound by it. (26) For the application of point (d) of Article 3(4) of Directive (EU) 2015/849 of the European Parliament and of the Council (12), the reference to serious fraud affecting the Union's financial interests as defined in Article 1(1) and Article2(1) of the Convention should be construed as fraud affecting the Union's financial interests as defined in Article 3 and in Article 7(3) of this Directive or, as regards offences against the common VAT system, as defined in Article 2(2) of this Directive. (27) Proper implementation of this Directive by the Member States includes the processing of personal data by the competent national authorities, and the exchange of such data between Member States on the one hand, and between competent Union bodies on the other. The processing of personal data at national level between national competent authorities should be regulated by the acquis of the Union. The exchange of personal data between Member States should be carried out in accordance with Directive (EU) 2016/680 of the European Parliament and of the Council (13). To the extent that the Union institutions, bodies, offices and agencies process personal data, Regulation (EC) No 45/2001 of the European Parliament and of the Council (14) or, where applicable, other Union legal acts regulating the processing of personal data by those bodies, offices and agencies as well as the applicable rules concerning the confidentiality of judicial investigations, should apply. (28) The intended dissuasive effect of the application of criminal law sanctions requires particular caution with regard to fundamental rights. This Directive respects fundamental rights and observes the principles recognised in particular by the Charter of Fundamental Rights of the European Union (the Charter) and in particular the right to liberty and security, the protection of personal data, the freedom to choose an occupation and right to engage in work, the freedom to conduct a business, the right to property, the right to an effective remedy and to a fair trial, the presumption of innocence and the right of defence, the principles of the legality and proportionality of criminal offences and sanctions, as well as the principle of ne bis in idem. This Directive seeks to ensure full respect for those rights and principles and must be implemented accordingly. (29) Member States should take the necessary measures to ensure the prompt recovery of sums and their transfer to the Union budget, without prejudice to the relevant Union sector-specific rules on financial corrections and recovery of amounts unduly spent. (30) Administrative measures and penalties play an important role in the protection of the Union's financial interests. This Directive does not exempt Member States from the obligation to apply and implement administrative Union measures and penalties within the meaning of Articles 4 and 5 of Regulation (EC, Euratom) No 2988/95. (31) This Directive should oblige Member States to provide in their national law for criminal penalties in respect of the acts of fraud and fraud-related criminal offences affecting the Union's financial interests to which this Directive applies. This Directive should not create obligations regarding the application of such penalties or any other available system of law enforcement to individual cases. Member States may in principle continue to apply administrative measures and penalties in parallel in the area covered by this Directive. In the application of national law transposing this Directive, Member States should, however, ensure that the imposition of criminal sanctions for criminal offences in accordance with this Directive and of administrative measures and penalties does not lead to a breach of the Charter. (32) This Directive should not affect the competences of Member States to structure and organise their tax administration as they see fit to ensure the correct determination, assessment and collection of value added tax, as well as the effective application of VAT law. (33) This Directive applies without prejudice to the provisions on the lifting of the immunities contained in the TFEU, Protocol No 3 on the Statute of the Court of Justice of the European Union and Protocol No 7 on the Privileges and Immunities of the European Union, annexed to the TFEU and to the Treaty on European Union(TEU), and the texts implementing them, or similar provisions incorporated in national law. In the transposition of this Directive into national law as well as in the application of national law transposing this Directive, those privileges and immunities, including the respect for the freedom of the Member's mandate, are fully taken into account. (34) This Directive is without prejudice to the general rules and principles of national criminal law on the application and execution of sentences in accordance with the concrete circumstances in each individual case. (35) Since the objective of this Directive cannot be sufficiently achieved by the Member States but can rather, by reason of its scale and effects, be better achieved at Union level, the Union may adopt measures, in accordance with the principle of subsidiarity as set out in Article 5 TEU. In accordance with the principle of proportionality as set out in that Article, this Directive does not go beyond what is necessary to achieve that objective. (36) In accordance with Article 3 and Article 4a(1) of Protocol No 21 on the position of United Kingdom and Ireland in respect of the area of freedom, security and justice, annexed to the TEU and to the TFEU, Ireland has notified its wish to take part in the adoption and application of this Directive. (37) In accordance with Articles 1 and 2 of Protocol No 21 on the position of the United Kingdom and Ireland in respect of the area of freedom, security and justice, annexed to the TEU and to the TFEU, and without prejudice to Article 4 of that Protocol, the United Kingdom is not taking part in the adoption of this Directive and is not bound by it or subject to its application. (38) In accordance with Articles 1 and 2 of Protocol No 22 on the position of Denmark, annexed to the TEU and to the TFEU, Denmark is not taking part in the adoption of this Directive and is not bound by it or subject to its application. (39) The European Court of Auditors has been consulted and has adopted an opinion (15), HAVE ADOPTED THIS DIRECTIVE: TITLE I SUBJECT MATTER, DEFINITIONS AND SCOPE Article 1 Subject matter This Directive establishes minimum rules concerning the definition of criminal offences and sanctions with regard to combatting fraud and other illegal activities affecting the Union's financial interests, with a view to strengthening protection against criminal offences which affect those financial interests, in line with the acquis of the Union in this field. Article 2 Definitions and scope 1. For the purposes of this Directive, the following definitions apply: (a) Union's financial interests means all revenues, expenditure and assets covered by, acquired through, or due to: (i) the Union budget; (ii) the budgets of the Union institutions, bodies, offices and agencies established pursuant to the Treaties or budgets directly or indirectly managed and monitored by them; (b) legal person means an entity having legal personality under the applicable law, except for States or public bodies in the exercise of State authority and for public international organisations. 2. In respect of revenue arising from VAT own resources, this Directive shall apply only in cases of serious offences against the common VAT system. For the purposes of this Directive, offences against the common VAT system shall be considered to be serious where the intentional acts or omissions defined in point (d) of Article 3(2) are connected with the territory of two or more Member States of the Union and involve a total damage of at least EUR 10 000 000. 3. The structure and functioning of the tax administration of the Member States are not affected by this Directive. TITLE II CRIMINAL OFFENCES WITH REGARD TO FRAUD AFFECTING THE UNION'S FINANCIAL INTERESTS Article 3 Fraud affecting the Union's financial interests 1. Member States shall take the necessary measures to ensure that fraud affecting the Union's financial interests constitutes a criminal offence when committed intentionally. 2. For the purposes of this Directive, the following shall be regarded as fraud affecting the Union's financial interests: (a) in respect of non-procurement-related expenditure, any act or omission relating to: (i) the use or presentation of false, incorrect or incomplete statements or documents, which has as its effect the misappropriation or wrongful retention of funds or assets from the Union budget or budgets managed by the Union, or on its behalf; (ii) non-disclosure of information in violation of a specific obligation, with the same effect; or (iii) the misapplication of such funds or assets for purposes other than those for which they were originally granted; (b) in respect of procurement-related expenditure, at least when committed in order to make an unlawful gain for the perpetrator or another by causing a loss to the Union's financial interests, any act or omission relating to: (i) the use or presentation of false, incorrect or incomplete statements or documents, which has as its effect the misappropriation or wrongful retention of funds or assets from the Union budget or budgets managed by the Union, or on its behalf; (ii) non-disclosure of information in violation of a specific obligation, with the same effect; or (iii) the misapplication of such funds or assets for purposes other than those for which they were originally granted, which damages the Union's financial interests; (c) in respect of revenue other than revenue arising from VAT own resources referred to in point (d), any act or omission relating to: (i) the use or presentation of false, incorrect or incomplete statements or documents, which has as its effect the illegal diminution of the resources of the Union budget or budgets managed by the Union, or on its behalf; (ii) non-disclosure of information in violation of a specific obligation, with the same effect; or (iii) misapplication of a legally obtained benefit, with the same effect; (d) in respect of revenue arising from VAT own resources, any act or omission committed in cross-border fraudulent schemes in relation to: (i) the use or presentation of false, incorrect or incomplete VAT-related statements or documents, which has as an effect the diminution of the resources of the Union budget; (ii) non-disclosure of VAT-related information in violation of a specific obligation, with the same effect; or (iii) the presentation of correct VAT-related statements for the purposes of fraudulently disguising the non-payment or wrongful creation of rights to VAT refunds. Article 4 Other criminal offences affecting the Union's financial interests 1. Member States shall take the necessary measures to ensure that money laundering as described in Article 1(3) of Directive (EU) 2015/849 involving property derived from the criminal offences covered by this Directive constitutes a criminal offence. 2. Member States shall take the necessary measures to ensure that passive and active corruption, when committed intentionally, constitute criminal offences. (a) For the purposes of this Directive, passive corruption means the action of a public official who, directly or through an intermediary, requests or receives advantages of any kind, for himself or for a third party, or accepts a promise of such an advantage, to act or to refrain from acting in accordance with his duty or in the exercise of his functions in a way which damages or is likely to damage the Union's financial interests. (b) For the purposes of this Directive, active corruption means the action of a person who promises, offers or gives, directly or through an intermediary, an advantage of any kind to a public official for himself or for a third party for him to act or to refrain from acting in accordance with his duty or in the exercise of his functions in a way which damages or is likely to damage the Union's financial interests. 3. Member States shall take the necessary measures to ensure that misappropriation, when committed intentionally, constitutes a criminal offence. For the purposes of this Directive, misappropriation means the action of a public official who is directly or indirectly entrusted with the management of funds or assets to commit or disburse funds or appropriate or use assets contrary to the purpose for which they were intended in any way which damages the Union's financial interests. 4. For the purposes of this Directive, public official means: (a) a Union official or a national official, including any national official of another Member State and any national official of a third country: (i) Union official means a person who is:  an official or other servant engaged under contract by the Union within the meaning of the Staff Regulations of Officials and the Conditions of Employment of Other Servants of the European Union laid down in Council Regulation (EEC, Euratom, ECSC) No 259/68 (16) (the Staff Regulations), or  seconded to the Union by a Member State or by any public or private body, who carries out functions equivalent to those performed by Union officials or other servants. Without prejudice to the provisions on privileges and immunities contained in Protocols No 3 and No 7, Members of the Union institutions, bodies, offices and agencies, set up in accordance with the Treaties and the staff of such bodies shall be assimilated to Union officials, inasmuch as the Staff Regulations do not apply to them; (ii) national official shall be understood by reference to the definition of official or public official in the national law of the Member State or third country in which the person in question carries out his or her functions. Nevertheless, in the case of proceedings involving a national official of a Member State, or a national official of a third country, initiated by another Member State, the latter shall not be bound to apply the definition of national official except insofar as that definition is compatible with its national law. The term national official shall include any person holding an executive, administrative or judicial office at national, regional or local level. Any person holding a legislative office at national, regional or local level shall be assimilated to a national official; (b) any other person assigned and exercising a public service function involving the management of or decisions concerning the Union's financial interests in Member States or third countries. TITLE III GENERAL PROVISIONS RELATING TO FRAUD AND OTHER CRIMINAL OFFENCES AFFECTING THE UNION'S FINANCIAL INTERESTS Article 5 Incitement, aiding and abetting, and attempt 1. Member States shall take the necessary measures to ensure that inciting, and aiding and abetting the commission of any of the criminal offences referred to in Articles 3 and 4 are punishable as criminal offences. 2. Member States shall take the necessary measures to ensure that an attempt to commit any of the criminal offences referred to in Article 3 and Article 4(3) is punishable as a criminal offence. Article 6 Liability of legal persons 1. Member States shall take the necessary measures to ensure that legal persons can be held liable for any of the criminal offences referred to in Articles 3, 4 and 5 committed for their benefit by any person, acting either individually or as part of an organ of the legal person, and having a leading position within the legal person, based on: (a) a power of representation of the legal person; (b) an authority to take decisions on behalf of the legal person; or (c) an authority to exercise control within the legal person. 2. Member States shall also take the necessary measures to ensure that legal persons can be held liable where the lack of supervision or control by a person referred to in paragraph 1 of this Article has made possible the commission, by a person under its authority, of any of the criminal offences referred to in Article 3, 4 or 5 for the benefit of that legal person. 3. Liability of legal persons under paragraphs 1 and 2 of this Article shall not exclude the possibility of criminal proceedings against natural persons who are perpetrators of the criminal offences referred to in Articles 3 and 4 or who are criminally liable under Article 5. Article 7 Sanctions with regard to natural persons 1. As regards natural persons, Member States shall ensure that the criminal offences referred to in Articles 3, 4 and 5 are punishable by effective, proportionate and dissuasive criminal sanctions. 2. Member States shall take the necessary measures to ensure that the criminal offences referred to in Articles 3 and 4 are punishable by a maximum penalty which provides for imprisonment. 3. Member States shall take the necessary measures to ensure that the criminal offences referred to in Articles 3 and 4 are punishable by a maximum penalty of at least four years of imprisonment when they involve considerable damage or advantage. The damage or advantage resulting from the criminal offences referred to in points (a), (b) and (c) of Article 3(2) and in Article 4 shall be presumed to be considerable where the damage or advantage involves more than EUR 100 000. The damage or advantage resulting from the criminal offences referred to in point (d) of Article 3(2) and subject to Article 2(2) shall always be presumed to be considerable. Member States may also provide for a maximum sanction of at least four years of imprisonment in other serious circumstances defined in their national law. 4. Where a criminal offence referred to in point (a), (b) or (c) of Article 3(2) or in Article 4 involves damage of less than EUR 10 000 or an advantage of less than EUR 10 000, Member States may provide for sanctions other than criminal sanctions. 5. Paragraph 1 shall be without prejudice to the exercise of disciplinary powers by the competent authorities against public officials. Article 8 Aggravating circumstance Member States shall take the necessary measures to ensure that where a criminal offence referred to in Article 3, 4 or 5 is committed within a criminal organisation in the sense of Framework Decision 2008/841/JHA, this shall be considered to be an aggravating circumstance. Article 9 Sanctions with regard to legal persons Member States shall take the necessary measures to ensure that a legal person held liable pursuant to Article 6 is subject to effective, proportionate and dissuasive sanctions, which shall include criminal or non-criminal fines and may include other sanctions, such as: (a) exclusion from entitlement to public benefits or aid; (b) temporary or permanent exclusion from public tender procedures; (c) temporary or permanent disqualification from the practice of commercial activities; (d) placing under judicial supervision; (e) judicial winding-up; (f) temporary or permanent closure of establishments which have been used for committing the criminal offence. Article 10 Freezing and confiscation Member States shall take the necessary measures to enable the freezing and confiscation of instrumentalities and proceeds from the criminal offences referred to in Articles 3, 4 and 5. Member States bound by Directive 2014/42/EU of the European Parliament and of the Council (17) shall do so in accordance with that Directive. Article 11 Jurisdiction 1. Each Member State shall take the necessary measures to establish its jurisdiction over the criminal offences referred to in Articles 3, 4 and 5 where: (a) the criminal offence is committed in whole or in part within its territory; or (b) the offender is one of its nationals. 2. Each Member State shall take the necessary measures to establish its jurisdiction over the criminal offences referred to in Articles 3, 4 and 5 where the offender is subject to the Staff Regulations at the time of the criminal offence. Each Member State may refrain from applying the rules on jurisdiction established in this paragraph or may apply them only in specific cases or only where specific conditions are fulfilled and shall inform the Commission thereof. 3. A Member State shall inform the Commission where it decides to extend its jurisdiction to criminal offences referred to in Article 3, 4 or 5 which have been committed outside its territory in any of the following situations: (a) the offender is a habitual resident in its territory; (b) the criminal offence is committed for the benefit of a legal person established in its territory; or (c) the offender is one of its officials who acts in his or her official duty. 4. In cases referred to in point (b) of paragraph 1, Member States shall take the necessary measures to ensure that the exercise of their jurisdiction is not subject to the condition that a prosecution can be initiated only following a report made by the victim in the place where the criminal offence was committed, or a denunciation from the State of the place where the criminal offence was committed. Article 12 Limitation periods for criminal offences affecting the Union's financial interests 1. Member States shall take the necessary measures to provide for a limitation period that enables the investigation, prosecution, trial and judicial decision of criminal offences referred to in Articles 3, 4 and 5 for a sufficient period of time after the commission of those criminal offences, in order for those criminal offences to be tackled effectively. 2. Member States shall take the necessary measures to enable the investigation, prosecution, trial and judicial decision of criminal offences referred to in Articles 3, 4 and 5 which are punishable by a maximum sanction of at least four years of imprisonment, for a period of at least five years from the time when the offence was committed. 3. By way of derogation from paragraph 2, Member States may establish a limitation period that is shorter than five years, but not shorter than three years, provided that the period may be interrupted or suspended in the event of specified acts. 4. Member States shall take the necessary measures to enable the enforcement of: (a) a penalty of more than one year of imprisonment; or alternatively (b) a penalty of imprisonment in the case of a criminal offence which is punishable by a maximum sanction of at least four years of imprisonment, imposed following a final conviction for a criminal offence referred to in Article 3, 4 or 5, for at least five years from the date of the final conviction. That period may include extensions of the limitation period arising from interruption or suspension. Article 13 Recovery This Directive shall be without prejudice to the recovery of the following: (1) at Union level of sums unduly paid in the context of the commission of the criminal offences referred to in point (a), (b) or (c) of Article 3(2), or in Article 4 or 5; (2) at national level, of any VAT not paid in the context of the commission of the criminal offences referred in point (d) of Article 3(2), or in Article 4 or 5. Article 14 Interaction with other applicable legal acts of the Union The application of administrative measures, penalties and fines as laid down in Union law, in particular those within the meaning of Articles 4 and 5 of Regulation (EC, Euratom) No 2988/95, or in national law adopted in compliance with a specific obligation under Union law, shall be without prejudice to this Directive. Member States shall ensure that any criminal proceedings initiated on the basis of national provisions implementing this Directive do not unduly affect the proper and effective application of administrative measures, penalties and fines that cannot be equated to criminal proceedings, laid down in Union law or national implementing provisions. TITLE IV FINAL PROVISIONS Article 15 Cooperation between the Member States and the Commission (OLAF) and other Union institutions, bodies, offices or agencies 1. Without prejudice to the rules on cross-border cooperation and mutual legal assistance in criminal matters, the Member States, Eurojust, the European Public Prosecutor's Office and the Commission shall, within their respective competences, cooperate with each other in the fight against the criminal offences referred to in Articles 3, 4 and 5. To that end the Commission, and where appropriate, Eurojust, shall provide such technical and operational assistance as the competent national authorities need to facilitate coordination of their investigations. 2. The competent authorities in the Member States may, within their competences, exchange information with the Commission so as to make it easier to establish the facts and to ensure effective action against the criminal offences referred to in Articles 3, 4 and 5. The Commission and the competent national authorities shall take into account in each specific case the requirements of confidentiality and the rules on data protection. Without prejudice to national law on access to information, a Member State may, to that end, when supplying information to the Commission, set specific conditions covering the use of information, whether by the Commission or by another Member State to which the information is passed. 3. The Court of Auditors and auditors responsible for auditing the budgets of the Union institutions, bodies, offices and agencies established pursuant to the Treaties, and the budgets managed and audited by the institutions, shall disclose to OLAF and to other competent authorities any fact of which they become aware when carrying out their duties, which could be qualified as a criminal offence referred to in Article 3, 4 or 5. Member States shall ensure that national audit bodies do the same. Article 16 Replacement of the Convention on the protection of the European Communities' financial interests The Convention on the protection of the European Communities' financial interests of 26 July 1995, including the Protocols thereto of 27 September 1996, of 29 November 1996 and of 19 June 1997, is hereby replaced by this Directive for the Member States bound by it, with effect from 6 July 2019. For the Member States bound by this Directive, references to the Convention shall be construed as references to this Directive. Article 17 Transposition 1. Member States shall adopt and publish, by 6 July 2019, the laws, regulations and administrative provisions necessary to comply with this Directive. They shall immediately communicate the text of those measures to the Commission. They shall apply those measures from 6 July 2019. When Member States adopt those measures, they shall contain a reference to this Directive or be accompanied by such a reference on the occasion of their official publication. They shall also include a statement that, for the Member States bound by this Directive, references in existing laws, regulations and administrative provisions to the Convention replaced by this Directive shall be construed as references to this Directive. Member States shall determine how such reference is to be made and how that statement is to be formulated. 2. Member States shall communicate to the Commission the text of the main provisions of national law which they adopt in the field covered by this Directive. Article 18 Reporting and assessment 1. The Commission shall by 6 July 2021 submit a report to the European Parliament and the Council, assessing the extent to which the Member States have taken the necessary measures in order to comply with this Directive. 2. Without prejudice to reporting obligations laid down in other Union legal acts, Member States shall, on an annual basis, submit the following statistics on the criminal offences referred to in Articles 3, 4 and 5 to the Commission, if they are available at a central level in the Member State concerned: (a) the number of criminal proceedings initiated, dismissed, resulting in an acquittal, resulting in a conviction and ongoing; (b) the amounts recovered following criminal proceedings and the estimated damage. 3. The Commission shall, by 6 July 2024 and taking into account its report submitted pursuant to paragraph 1 and the Member States' statistics submitted pursuant to paragraph 2, submit a report to the European Parliament and to the Council, assessing the impact of national law transposing this Directive on the prevention of fraud to the Union's financial interests. 4. The Commission shall, by 6 July 2022 and on the basis of the statistics submitted by Member States, pursuant to paragraph 2, submit a report to the European Parliament and to the Council, assessing, with regard to the general objective to strengthen the protection of the Union's financial interests, whether: (a) the threshold indicated in Article 2(2) is appropriate; (b) the provisions relating to limitation periods as referred to in Article 12 are sufficiently effective; (c) this Directive effectively addresses cases of procurement fraud. 5. The reports referred to in paragraphs 3 and 4 shall be accompanied, if necessary, by a legislative proposal, which may include a specific provision on procurement fraud. Article 19 Entry into force This Directive shall enter into force on the twentieth day following that of its publication in the Official Journal of the European Union. Article 20 Addressees This Directive is addressed to the Member States in accordance with the Treaties. Done at Strasbourg, 5 July 2017. For the European Parliament The President A. TAJANI For the Council The President M. MAASIKAS (1) OJ C 391, 18.12.2012, p. 134. (2) Position of the European Parliament of 16 April 2014 (not yet published in the Official Journal) and position of the Council at first reading of 25 April 2017 (OJ C 184, 9.6.2017, p. 1). Position of the European Parliament of 5 July 2017 (not yet published in the Official Journal). (3) OJ C 316, 27.11.1995, p. 48. (4) OJ C 313, 23.10.1996, p. 1. (5) OJ C 151, 20.5.1997, p. 1. (6) OJ C 221, 19.7.1997, p. 11. (7) Council Regulation (EC, Euratom) No 2988/95 of 18 December 1995 on the protection of the European Communities' financial interests (OJ L 312, 23.12.1995, p. 1). (8) Council Directive 2006/112/EC of 28 November 2006 on the common system of value added tax (OJ L 347, 11.12.2006, p. 1). (9) Regulation (EU, Euratom) No 966/2012 of the European Parliament and of the Council of 25 October 2012 on the financial rules applicable to the general budget of the Union and repealing Council Regulation (EC, Euratom) No 1605/2002 (OJ L 298, 26.10.2012, p. 1). (10) Council Framework Decision 2008/841/JHA of 24 October 2008 on the fight against organised crime (OJ L 300, 11.11.2008, p. 42). (11) Regulation (EU, Euratom) No 883/2013 of the European Parliament and of the Council of 11 September 2013 concerning investigations conducted by the European Anti-Fraud Office (OLAF) and repealing Regulation (EC) No 1073/1999 of the European Parliament and of the Council and Council Regulation (Euratom) No 1074/1999 (OJ L 248, 18.9.2013, p. 1). (12) Directive (EU) 2015/849 of the European Parliament and of the Council of 20 May 2015 on the prevention of the use of the financial system for the purposes of money laundering or terrorist financing, amending Regulation (EU) No 648/2012 of the European Parliament and of the Council, and repealing Directive 2005/60/EC of the European Parliament and of the Council and Commission Directive 2006/70/EC (OJ L 141, 5.6.2015, p. 73). (13) Directive (EU) 2016/680 of the European Parliament and of the Council of 27 April 2016 on the protection of natural persons with regard to the processing of personal data by competent authorities for the purposes of the prevention, investigation, detection or prosecution of criminal offences or the execution of criminal penalties, and on the free movement of such data, and repealing Council Framework Decision 2008/977/JHA (OJ L 119, 4.5.2016, p. 89). (14) Regulation (EC) No 45/2001 of the European Parliament and of the Council of 18 December 2000 on the protection of individuals with regard to the processing of personal data by the Community institutions and bodies and on the free movement of such data (OJ L 8, 12.1.2001, p. 1). (15) OJ C 383, 12.12.2012, p. 1. (16) OJ L 56, 4.3.1968, p. 1. (17) Directive 2014/42/EU of the European Parliament and of the Council of 3 April 2014 on the freezing and confiscation of instrumentalities and proceeds of crime in the European Union (OJ L 127, 29.4.2014, p. 39).
============================== "END OF DOC" ==============================
        

In [12]:
display(Markdown(search_docs("Drug dealing Sentences")))

 

        doc 0 :

        'celex': 31997R2046
        'status': Not in Force
        'act_type': Regulation
        'treaty': TEC (1992)

        full_doc :

        Avis juridique important|31997R2046Council Regulation (EC) No 2046/97 of 13 October 1997 on north-south cooperation in the campaign against drugs and drug addiction Official Journal L 287 , 21/10/1997 P. 0001 - 0005COUNCIL REGULATION (EC) No 2046/97 of 13 October 1997 on north-south cooperation in the campaign against drugs and drug addictionTHE COUNCIL OF THE EUROPEAN UNION,Having regard to the Treaty establishing the European Community, and in particular Article 130w thereof,Having regard to the proposal from the Commission (1),Acting in accordance with the procedure laid down in Article 189c of the Treaty (2),Whereas the impact on the structures of a developing society of an economy based on the production of drugs, or which derives a substantial revenue from them, undermines a country's smooth integration into the world economy;Whereas the breakdown of social structures in developing countries due to drug consumption and the related industry is detrimental to sustainable social development and the attainment of the goals of Community policy in the sphere of development cooperation as defined in Article 130u of the Treaty;Whereas as part of the effort to combat the supply of drugs it is essential, in particular, to effect a radical reduction of poverty in the south and to offer the population a lawful alternative to the growing of illegal crops;Whereas institutional support should be given to those developing countries which so request so that they can combat drugs more effectively;Whereas in a communication to the Parliament and to the Council dated 23 June 1994, the Commission presented its guidelines for a European Union plan of action on drugs for 1995-1999, including measures at international level;Whereas Parliament stated its views on the guidelines in its Opinion on the communication, adopted on 15 June 1995;Whereas the Fourth ACP-EC Convention and the cooperation, association and partnership agreements concluded by the European Community with developing countries contain clauses on cooperation to curb drug abuse and drug trafficking, the monitoring of trade in precursors, chemical products and psychotropic substances and the exchange of relevant information, including measures in the field of money laundering; whereas there is a relationship between the campaign against drugs and drug addiction and the aims of the cooperation policy pursued by the Community and its developing-country partners;Whereas the international community's strategy to curb drug abuse and drug trafficking is based on universal accession to the Single Convention on narcotic drugs of 1961, as amended by the Protocol of 1972, the Convention on psychotropic substances of 1971 and the International Convention against illicit traffic in narcotic drugs and psychotropic substances of 1988, and on the systematic implementation of those conventions at national and international level;Whereas the European Community is a party to the Convention of 1988, in particular by virtue of Article 12 of that Convention, and has adopted Community legislation based on the recommendations of the Chemicals Action Task Force set up by the G7 and the President of the Commission in 1989, the effectiveness of which would be generally enhanced by the adoption of the relevant legislation and procedures in other parts of the world;Whereas effective action against drugs must also encompass measures against the laundering of money from drug trafficking, such as the adoption of a suitable legal framework and appropriate mechanisms in the countries concerned;Whereas human rights must be duly respected in implementing measures under this Regulation;Whereas the Member States of the European Community have endorsed the policy statement and general plan of action adopted by the UN General Assembly at its 17th special session;Whereas a financial reference amount within the meaning of point 2 of the Declaration by the European Parliament, the Council and the Commission of 6 March 1995 is included in this Regulation for the period 1998-2000, without thereby affecting the powers of the budgetary authority as they are defined by the Treaty,HAS ADOPTED THIS REGULATION:Article 1 In the framework of its development cooperation policy and taking account of the harmful effects on development efforts of the production, trading and consumption of drugs, the Community shall carry out cooperation activities in the field of drugs and drug addiction in developing countries, giving priority to those which have demonstrated political will at the highest level to solve their drug problem. The existence of such will may be demonstrated inter alia by ratification of the Single Convention of 1961 as amended by the Protocol of 1972, the Convention of 1971 and the Convention of 1988. Commitment on the part of developing countries shall take the form, inter alia, of the implementation of domestic legislation against laundering of money generated through illicit drugs.Article 2 The assistance provided under this Regulation shall complement and reinforce assistance provided under other instruments of development cooperation.Article 3 The Community shall give priority at the request of a partner country to supporting the preparation of a national drug control master plan, in close consultation with the United Nations International Drug Control Programme (UNDCP). These plans will identify objectives, strategies and priorities in the campaign against drugs and the related resource requirements (including financial requirements), thus establishing an integrated, multidisciplinary and multisectoral approach designed to maximize the efficiency of national drug control programmes and international assistance.The prevention of drug addiction, together with demand reduction, shall be addressed in a consistent policy comprising education and objective information about the consequences of addiction, targeted especially at young people.Community cooperation shall take place in a spirit of dialogue reflecting the genuine cultural differences which affect the perception of drug-related problems, this being crucial to ensure the social and political viability of drug control strategies.Article 4 Preferably operating within the strategic framework established by the national plans, the Community shall also support specific operations capable of a measurable impact (i.e. effective and tangible results within a time-limit set in advance) in the following areas:- development of institutional capacity, in particular for the implementation of:- National Drug Control Plans by developing countries,- agreements between the Community and certain developing countries, in particular to combat the diversion of chemical precursors and to curb money laundering,- demand reduction, in particular through analysis of local patterns, the introduction of measures to control trade in and consumption of narcotics and psychotropic substances, treatment and reintegration of drug addicts, as well as risk limitation. These measures must be integrated into policies on health and education, development and combating poverty and social and economic exclusion,- promotion of pilot alternative development projects, conceived as a process by which the production of illicit drug crops is eventually both combatted and eliminated through appropriate rural development measures in the context of sustained national economic growth. These projects shall comprise social and economic measures which take into account factors contributing to illicit production as well as measures which may facilitate improved use of commercial preferences. In this connection, it shall be systematically examined whether it is possible to make more use of other Community financial instruments (e.g. ALA) and the European Development Fund for alternative development projects,- financing of studies, seminars and fora for the exchange of experience in the above fields.Particular attention shall be paid to the participation of local people and target groups in identifying, planning and carrying out operations.The Community shall only finance projects in which respect for human rights is guaranteed.Article 5 The cooperation partners eligible for financial support under this Regulation shall be regional and international organizations, in particular UNDCP, local- and Member State-based non-governmental organizations, national, provincial and local government departments and agencies, community-based organizations, institutes and public and private operators.Article 6 1. The instruments to be employed in the course of the operations referred to in Articles 3 and 4 shall include studies, technical assistance, training or other services, supplies and works, along with audits and evaluation and monitoring missions.2. According to the needs of the operations concerned, Community financing may cover both capital investment, other than the purchase of real estate, and operating costs in foreign or local currency. However, with the exception of training programmes, operating costs may normally be covered only during the start-up phase and on a degressive basis.3. A financial contribution from the partners defined in Article 5 shall be sought for each cooperation operation. Their contribution will be requested within the limits of the possibilities available to the parties concerned and depending on the nature of the operation concerned.4. A financial contribution from the local partners, particularly in respect of operating costs, shall be sought as a matter of priority in the case of projects intended to launch long term activities, so as to ensure the viability of such projects once Community funding comes to an end.5. Opportunities may be sought for cofinancing with other fund providers, and especially with Member States.6. The Commission will ensure that the Community character of the aid provided under this Regulation is highlighted.7. In order to achieve the objectives of consistency and complementarity referred to in the Treaty and with the aim of guaranteeing optimum effectiveness of all these operations, the Commission may take all necessary coordination measures, including in particular:(a) a system for the systematic exchange and analysis of information on operations financed and those which the Community and the Member States propose to finance;(b) on-the-spot coordination of the implementation of operations through regular meetings and exchange of information between representatives of the Commission and of the Member States in the beneficiary country.8. In order to obtain the greatest possible impact globally and nationally, the Commission, in liaison with the Member States, shall take any initiative necessary for ensuring proper coordination and close collaboration with the beneficiary countries and the providers of funds and other international agencies involved, in particular those forming part of the United Nations system and more specifically the UNDCP.Article 7 Financial support under this Regulation shall take the form of grants.Article 8 The financial reference amount for the implementation of this programme during the period 1998-2000 shall be ECU 30 million.Annual appropriations shall be authorized by the budgetary authority within the limits of the financial perspectives.Article 9 1. The Commission shall be responsible for appraising, approving and managing operations covered by this Regulation in accordance with the budgetary and other procedures in force, and in particular those laid down in the Financial Regulation applicable to the general budget of the European Communities.2. Projects and programme appraisal shall take into account the following factors:- effectiveness and viability of operations,- cultural, social, gender and environmental aspects,- institutional development necessary to achieve project goals,- experience gained from operations of the same kind.3. Decisions relating to grants of more than ECU 2 million for individual operations financed under this Regulation and any changes resulting in an increase of more than 20 % in the sum initially approved for such an operation shall be adopted under the procedure laid down in Article 10.The Commission shall inform the Committee referred to in Article 10 succinctly of the financing decisions which it intends to take with regard to projects and programmes of less than ECU 2 million in value. The information shall be made available not later than one week before the decision is taken.4. The Commission shall be authorized to approve, without recourse to the opinion of the Committee provided for in Article 10, any supplementary commitments needed for covering expected or real cost overruns in connection with the operations, where the overrun or additional requirement is less than or equal to 20 % of the initial commitment fixed by the financing decision.5. All financing agreements or contracts concluded under this Regulation shall provide for the Commission and the Court of Auditors to conduct on-the-spot checks according to the usual procedures laid down by the Commission under the rules in force, and in particular those of the Financial Regulation applicable to the general budget of the European Communities.6. Where operations are the subject of financing agreements between the Community and the recipient country, such agreement shall stipulate that the payment of taxes, duties or any other charges is not to be covered by the Community.7. Participation in invitations to tender and the award of contracts shall be open on equal terms to natural and legal persons of the Member States and of the recipient country. It may be extended to other developing countries.8. Supplies shall originate in the Member States, the recipient country or other developing countries. In exceptional cases, where circumstances warrant, supplies may originate elsewhere.9. Particular attention will be given to:- the pursuit of cost-effectiveness and sustainable impact in project design,- the clear definition and monitoring of objectives and indications of achievement for all projects.Article 10 1. The Commission shall be assisted by the geographically-determined committee competent for development.2. The representative of the Commission shall submit to the committee a draft of the measures to be taken. The committee shall deliver its opinion on the draft, within a time limit which the chairman may lay down according to the urgency of the matter. The opinion shall be delivered by the majority laid down in Article 148 (2) of the Treaty in the case of decisions which the Council is required to adopt on a proposal from the Commission. The votes of the representatives of the Member States within the committee shall be weighted in the manner set out in that Article. The Chairman shall not vote.The Commission shall adopt the measures envisaged if they are in accordance with the opinion of the committee.If the measures envisaged are not in accordance with the opinion of the committee, or if no opinion is delivered, the Commission shall without delay submit to the Council a proposal relating to the measures to be taken. The Council shall act by a qualified majority.If, on the expiry of a period of three months from the date of referral to the Council, the Council has not acted, the proposed measures shall be adopted by the Commission.3. An exchange of views shall take place once a year on the basis of a presentation by the representative of the Commission of the general guidelines for the operations to be carried out in the year ahead, in the framework of a joint meeting of the committees referred to in paragraph 1.Article 11 1. At the end of each budget year, the Commission shall present a report to Parliament and the Council summarizing the operations financed in the course of that year and evaluating the implementation of this Regulation over that period.The summary shall in particular contain information about those with whom contracts have been concluded.2. The Commission shall regularly assess operations financed by the Community with a view to establishing whether the objectives aimed at by such operations have been achieved and to providing guidelines for improving the effectiveness of future operations. The Commission shall submit to the Committee referred to in Article 10 a summary of the assessments made which, if appropriate, may be examined by the Committee. The assessment reports shall be made available to any Member States requesting them.3. The Commission shall inform the Member States, at the latest one month after its decision, of the operations and projects approved, stating their cost and nature, the recipient country and partners.Article 12 1. This Regulation shall enter into force on the third day following that of its publication in the Official Journal of the European Communities.2. Three years after this Regulation enters into force, the Commission shall submit to the European Parliament and the Council an overall assessment of operations financed by the Community under this Regulation together with suggestions regarding the future of this Regulation and, where necessary, proposals for amending or terminating it.This Regulation shall be binding in its entirety and directly applicable in all Member States.Done at Luxembourg, 13 October 1997.For the CouncilThe PresidentJ.-C. JUNCKER(1) OJ C 242, 19. 9. 1995, p. 8.(2) Opinion of the European Parliament of 19 April 1996 (OJ C 141, 13. 5. 1996, p. 252), Council Common Position of 22 November 1996, (OJ C 6, 9. 1. 1997, p. 1) and Decision of the European Parliament of 13 March 1997 (OJ C 115/97, 14. 4. 1997, p. 127).
============================== "END OF DOC" ==============================
        

        doc 1 :

        'celex': 32014D0688
        'status': Not in Force
        'act_type': Decision_IMPL
        'treaty': TFEU (2008)

        full_doc :

        1.10.2014 EN Official Journal of the European Union L 287/22 COUNCIL IMPLEMENTING DECISION of 25 September 2014 on subjecting 4-iodo-2,5-dimethoxy-N-(2-methoxybenzyl)phenethylamine (25I-NBOMe), 3,4-dichloro-N-[[1-(dimethylamino)cyclohexyl]methyl]benzamide (AH-7921), 3,4-methylenedioxypyrovalerone (MDPV) and 2-(3-methoxyphenyl)-2-(ethylamino)cyclohexanone (methoxetamine) to control measures (2014/688/EU) THE COUNCIL OF THE EUROPEAN UNION, Having regard to the Treaty on the Functioning of the European Union, Having regard to Council Decision 2005/387/JHA of 10 May 2005 on the information exchange, risk-assessment and control of new psychoactive substances (1), and in particular Article 8(3) thereof, Having regard to the proposal from the European Commission, Whereas: (1) Risk assessment reports on the new psychoactive substances 4-iodo-2,5-dimethoxy-N-(2-methoxybenzyl)phenethylamine (25I-NBOMe), 3,4-dichloro-N-[[1-(dimethylamino)cyclohexyl]methyl]benzamide (AH-7921), 3,4-methylenedioxypyrovalerone (MDPV) and 2-(3-methoxyphenyl)-2-(ethylamino)cyclohexanone (methoxetamine) were drawn up in compliance with Decision 2005/387/JHA by a special session of the extended Scientific Committee of the European Monitoring Centre for Drugs and Drug Addiction (EMCDDA), and were subsequently submitted to the Commission and to the Council on 23 April 2014. (2) 25I-NBOMe, AH-7921, MDPV and methoxetamine had not been under assessment at the United Nations' level by the time the risk assessment was requested at Union level, but they were evaluated in June 2014 by the Expert Committee on Drug Dependence of the World Health Organization. (3) 25I-NBOMe, AH-7921, MDPV and methoxetamine have no established or acknowledged medical use (human or veterinary). Apart from their use in analytical reference materials, and in scientific research investigating their chemistry, pharmacology and toxicology as a result of their emergence on the drug market  and, in the case of 25I-NBOMe, also in the field of neurochemistry  there is no indication that they are being used for other purposes. (4) 25I-NBOMe is a potent synthetic derivative of 2,5-dimethoxy-4-iodophenethylamine (2C-I), a classical serotonergic hallucinogen, which was subject to risk assessment and to control measures and criminal sanctions at Union level from 2003 by Council Decision 2003/847/JHA (2). (5) The specific physical effects of 25I-NBOMe are difficult to determine because there are no published studies assessing its acute and chronic toxicity, its psychological and behavioural effects, and dependence potential, and because of the limited information and data available. Clinical observations of individuals who have used this substance suggest that it has hallucinogenic effects and has the potential for inducing severe agitation, confusion, intense auditory and visual hallucinations, aggression, violent accidents and self-induced trauma. (6) There have been four deaths associated with 25I-NBOMe registered in three Member States. Severe toxicity associated with its use has been reported in four Member States, which notified 32 non-fatal intoxications. If this new psychoactive substance were to become more widely available and used, the implications for individual and public health could be significant. There is no information available on the social risks associated with 25I-NBOMe. (7) 22 Member States and Norway have reported to the EMCDDA and European Police Office (Europol) that they detected 25I-NBOMe. No prevalence data is available on the use of 25I-NBOMe, but the limited information that exists suggests that it may be consumed in a wide range of settings, such as at home, in bars, nightclubs and at music festivals. (8) 25I-NBOMe is openly marketed and sold on the internet as a research chemical and information from seizures, collected samples, user websites and internet retailers suggests that it is being sold as a drug in its own right and also marketed as a legal replacement for LSD. EMCDDA identified more than 15 internet retailers selling this substance, who may be based within the Union and China. (9) The risk assessment report reveals that there is limited scientific evidence available on 25I-NBOMe and points out that further research would be needed to determine the health and social risks that it poses. However, the available evidence and information provides sufficient ground for subjecting 25I-NBOMe to control measures across the Union. As a result of the health risks that it poses, as documented by its detection in several reported fatalities, of the fact that users may unknowingly consume it and of the lack of medical value or use of the substance, 25I-NBOMe should be subjected to control measures across the Union. (10) Since six Member States control 25I-NBOMe under national legislation complying with the obligations of the 1971 United Nations Convention on Psychotropic Substances, and seven Member States use other legislative measures to control it, subjecting this substance to control measures across the Union would help avoid the emergence of obstacles to cross-border law enforcement and judicial cooperation, and would help protect against the risks that its availability and use can pose. (11) AH-7921 is a structurally atypical synthetic opioid analgesic commonly known by internet suppliers, user websites and media as doxylam. It can be easily confused with doxylamine, an antihistaminic medicine with sedative-hypnotic properties, which could lead to unintentional overdoses. (12) The specific physical effects of AH-7921 are difficult to determine because there are no published studies assessing its acute and chronic toxicity, its psychological, behavioural effects, and dependence potential, as well as the limited information and data available. Based on user reports, the effects of AH-7921 appear to resemble those of classical opioids with the feeling of mild euphoria, itchiness and relaxation; nausea appears to be a typical adverse effect. In addition to self-experimentation with AH-7921, as well as recreational use, some of the users report self-medicating with this new drug to relieve pain, others to alleviate withdrawal symptoms due to cessation of the use of other opioids. This may indicate a potential of AH-7921 to spread among the injecting opioid population. (13) There is no prevalence data on the use of AH-7921, but the information available suggests that it is not widely used, and that when it is used, that use is in the home environment. (14) 15 fatalities were recorded in three Member States between December 2012 and September 2013 where AH-7921, alone or in combination with other substances, was detected in post-mortem samples. While it is not possible to determine with certainty the role of AH-7921 in all of those fatalities, in some cases it has been specifically noted in the cause of death. One Member State reported six non-fatal intoxications associated with AH-7921. If this new psychoactive substance were to become more widely available and used, the implications for individual and public health could be significant. There is no information available on the social risks associated with AH-7921. (15) The risk assessment report reveals that there is limited scientific evidence available on AH-7921 and points out that further research would be needed to determine the health and social risks that it poses. However, the available evidence and information provides sufficient ground for subjecting AH-7921 to control measures across the Union. As a result of the health risks that it poses, as documented by its detection in several reported fatalities, of the fact that users may unknowingly consume it, and of the lack of medical value or use of the substance, AH-7921 should be subjected to control measures across the Union. (16) Since one Member State controls AH-7921 under national legislation complying with the obligations of the 1971 United Nations Convention on Psychotropic Substances and five Member States use other legislative measures to control it, subjecting this substance to control measures across the Union would help avoid the emergence of obstacles in cross-border law enforcement and judicial cooperation, and would help protect against the risks that its availability and use can pose. (17) MDPV is a ring-substituted synthetic derivative of cathinone chemically related to pyrovalerone, which are both subject to control under the 1971 United Nations Convention on Psychotropic Substances. (18) Information on the chronic and acute toxicity associated with MDPV, as well as on psychological and behavioural effects, and on dependence potential, is not collected uniformly across the Union. Information from published studies, confirmed by clinical cases, suggests that the psychopharmacological profile observed for MDPV is similar to that for cocaine and methamphetamine, albeit more potent and longer lasting. Furthermore, MDPV was found to be 10 times more potent in its ability to induce locomotor activation, tachycardia and hypertension. (19) Users' websites indicate that its acute toxicity can provoke adverse effects on humans, similar to those associated with other stimulants. These include paranoid psychosis, tachycardia, hypertension, diaphoresis, breathing problems, severe agitation, auditory and visual hallucinations, profound anxiety, hyperthermia, violent outbursts and multiple organ dysfunctions. (20) 108 fatalities were registered in eight Member States and Norway between September 2009 and August 2013, where MDPV has been detected in post-mortem biological samples or implicated in the cause of death. A total of 525 non-fatal intoxications associated with MDPV have been reported by eight Member States. If this new psychoactive substance were to become more widely available and used, the implications for individual and public health could be significant. (21) The detection of MDPV has also been reported in biological samples related to fatal and non-fatal road traffic accidents, or driving under the influence of drugs, in four Member States since 2009. (22) MDPV has been present in the Union drug market since November 2008 and 27 Member States, Norway and Turkey reported multi-kilogram seizures of the substance. MDPV is being sold as a substance in its own right, but it has also been detected in combination with other substances. It is widely available from internet suppliers and retailers, head shops and street-level dealers. There are some indications that suggest a degree of organisation in the tableting and distribution of this substance in the Union. (23) The risk assessment report reveals that further research would be needed to determine the health and social risks posed by MDPV. However, the available evidence and information provides sufficient ground for subjecting MDPV to control measures across the Union. As a result of the health risks that it poses, as documented by its detection in several reported fatalities, of the fact that users may unknowingly consume it, and of the lack of medical value or use of the substance, MDPV should be subjected to control measures across the Union. (24) Since 21 Member States control MDPV under national legislation complying with the obligations of the 1971 United Nations Convention on Psychotropic Substances and four Member States use other legislative measures to control it, subjecting this substance to control measures across the Union would help avoid the emergence of obstacles in cross-border law enforcement and judicial cooperation, and would protect against the risks that its availability and use can pose. (25) Methoxetamine is an arylcyclohexylamine substance which is chemically similar to ketamine and the internationally controlled substance phencyclidine (PCP). Like ketamine and PCP, it has dissociative properties. (26) There are no studies assessing the chronic and acute toxicity associated with methoxetamine, as well as its psychological and behavioural effects, and dependence potential. Self-reported experiences from user websites suggest adverse effects similar to ketamine intoxication. These include nausea and severe vomiting, difficulty in breathing, seizures, disorientation, anxiety, catatonia, aggression, hallucination, paranoia and psychosis. In addition, acute methoxetamine intoxications may include stimulant effects (agitation, tachycardia and hypertension) and cerebral features, which are not expectable with acute ketamine intoxication. (27) Twenty deaths associated with methoxetamine were reported by six Member States that detected the substance in post-mortem samples. Used alone or in combination with other substances, methoxetamine was detected in 20 non-fatal intoxications reported by five Member States. If this new psychoactive substance were to become more widely available and used, the implications for individual and public health could be significant. (28) 23 Member States, Turkey and Norway have reported that they detected methoxetamine, since November 2010. Information suggests that it is sold and used as a substance in its own right, but it is also sold as a legal replacement for ketamine by internet retailers, head shops and street-level drug dealers. (29) Multi-kilogram quantities in powder form were seized within the Union, but there is no information on the possible involvement of organised crime. The manufacture of methoxetamine does not require sophisticated equipment. (30) Prevalence data are limited to non-representative studies in two Member States. Those studies suggest that the prevalence of the use of methoxetamine is lower than that of ketamine. The available information suggests that it may be consumed in a wide range of settings, including at home, in bars, nightclubs and at music festivals. (31) The risk assessment report reveals that further research would be needed to determine the health and social risks posed by methoxetamine. However, the available evidence and information provides sufficient grounds for subjecting methoxetamine to control measures across the Union. As a result of the health risks that it poses, as documented by its detection in several reported fatalities, of the fact that users may unknowingly consume it, and of the lack of medical value or use, methoxetamine should be subjected to control measures across the Union. (32) Since nine Member States control methoxetamine under national legislation complying with the obligations of the 1971 United Nations Convention on Psychotropic Substances and nine Member States use other legislative measures to control it, subjecting this substance to control measures across the Union would help avoid the emergence of obstacles in cross-border law enforcement and judicial cooperation, and would protect against the risks that its availability and use can pose. (33) Decision 2005/387/JHA reserves to the Council implementing powers with a view to giving a quick and expertise-based response at the Union level to the emergence of new psychoactive substances detected and reported by the Member States, by submitting those substances to control measures across the Union. As the conditions and procedure for triggering the exercise of such implementing powers have been met, an implementing decision should be adopted in order to put 25I-NBOMe, AH-7921, MDPV and methoxetamine under control across the Union, HAS ADOPTED THIS DECISION: Article 1 The following new psychoactive substances shall be subjected to control measures across the Union: (a) 4-iodo-2,5-dimethoxy-N-(2-methoxybenzyl) phenethylamine(25I-NBOMe); (b) 3,4-dichloro-N-[[1-dimethylamino) cyclohexyl]methyl] benzamide (AH-7921); (c) 3,4-methylenedioxypyrovalerone (MDPV); (d) 2-(3-methoxyphenyl)-2-(ethylamino)cyclohexanone (methoxetamine). Article 2 By 2 October 2015, Member States shall subject in accordance with their national legislation, the new psychoactive substances referred to in Article 1 to control measures and criminal penalties, as provided for under their legislation complying with their obligations under the 1971 United Nations Convention on Psychotropic Substances. Article 3 This Decision shall enter into force on the twentieth day following that of its publication in the Official Journal of the European Union. Done at Brussels, 25 September 2014. For the Council The President F. GUIDI (1) OJ L 127, 20.5.2005, p. 32. (2) Council Decision 2003/847/JHA of 27 November 2003 concerning control measures and criminal sanctions in respect of the new synthetic drugs 2C-I, 2C-T-2, 2C-T-7 and TMA-2 (OJ L 321, 6.12.2003, p. 64).
============================== "END OF DOC" ==============================
        

        doc 2 :

        'celex': 32004F0757
        'status': In Force
        'act_type': Decision_FRAMW
        'treaty': TEU (1992)

        full_doc :

        11.11.2004 EN Official Journal of the European Union L 335/8 COUNCIL FRAMEWORK DECISION 2004/757/JHA of 25 october 2004 laying down minimum provisions on the constituent elements of criminal acts and penalties in the field of illicit drug trafficking THE COUNCIL OF THE EUROPEAN UNION, Having regard to the Treaty on European Union, and in particular Article 31(e) and Article 34(2)(b) thereof, Having regard to the proposal from the Commission (1), Having regard to the opinion of the European Parliament (2), Whereas: (1) Illicit drug trafficking poses a threat to health, safety and the quality of life of citizens of the European Union, and to the legal economy, stability and security of the Member States. (2) The need for legislative action to tackle illicit drug trafficking has been recognised in particular in the Action Plan of the Council and the Commission on how best to implement the provisions of the Amsterdam Treaty on an area of freedom, security and justice (3), adopted by the Justice and Home Affairs Council in Vienna on 3 December 1998, the conclusions of the Tampere European Council of 15 and 16 October 1999, in particular point 48 thereof, the European Union's Drugs Strategy (2000-2004) endorsed by the Helsinki European Council from 10 to 12 December 1999 and the European Union's Action Plan on Drugs (2000-2004) endorsed by the European Council in Santa Maria da Feira on 19 and 20 June 2000. (3) It is necessary to adopt minimum rules relating to the constituent elements of the offences of illicit trafficking in drugs and precursors which will allow a common approach at European Union level to the fight against such trafficking. (4) By virtue of the principle of subsidiarity, European Union action should focus on the most serious types of drug offence. The exclusion of certain types of behaviour as regards personal consumption from the scope of this Framework Decision does not constitute a Council guideline on how Member States should deal with these other cases in their national legislation. (5) Penalties provided for by the Member States should be effective, proportionate and dissuasive, and include custodial sentences. To determine the level of penalties, factual elements such as the quantities and the type of drugs trafficked, and whether the offence was committed within the framework of a criminal organisation, should be taken into account. (6) Member States should be allowed to make provision for reducing the penalties when the offender has supplied the competent authorities with valuable information. (7) It is necessary to take measures to enable the confiscation of the proceeds of the offences referred to in this Framework Decision. (8) Measures should be taken to ensure that legal persons can be held liable for the criminal offences referred to by this Framework Decision which are committed for their benefit. (9) The effectiveness of the efforts made to tackle illicit drug trafficking depends essentially on the harmonisation of the national measures implementing this Framework Decision, HAS DECIDED AS FOLLOWS: Article 1 Definitions For the purposes of this Framework Decision: 1. drugs: shall mean any of the substances covered by the following United Nations Conventions: (a) the 1961 Single Convention on Narcotic Drugs (as amended by the 1972 Protocol); (b) the 1971 Vienna Convention on Psychotropic Substances. It shall also include the substances subject to controls under Joint Action 97/396/JHA of 16 June 1997 concerning the information exchange risk assessment and the control of new synthetic drugs (4); 2. precursors: shall mean any substance scheduled in the Community legislation giving effect to the obligations deriving from Article 12 of the United Nations Convention against Illicit Traffic in Narcotic Drugs and Psychotropic Substances of 20 December 1988; 3. legal person: shall mean any legal entity having such status under the applicable national law, except for States or other public bodies acting in the exercise of their sovereign rights and for public international organisations. Article 2 Crimes linked to trafficking in drugs and precursors 1. Each Member State shall take the necessary measures to ensure that the following intentional conduct when committed without right is punishable: (a) the production, manufacture, extraction, preparation, offering, offering for sale, distribution, sale, delivery on any terms whatsoever, brokerage, dispatch, dispatch in transit, transport, importation or exportation of drugs; (b) the cultivation of opium poppy, coca bush or cannabis plant; (c) the possession or purchase of drugs with a view to conducting one of the activities listed in (a); (d) the manufacture, transport or distribution of precursors, knowing that they are to be used in or for the illicit production or manufacture of drugs. 2. The conduct described in paragraph 1 shall not be included in the scope of this Framework Decision when it is committed by its perpetrators exclusively for their own personal consumption as defined by national law. Article 3 Incitement, aiding and abetting and attempt 1. Each Member State shall take the necessary measures to make incitement to commit, aiding and abetting or attempting one of the offences referred to in Article 2 a criminal offence. 2. A Member State may exempt from criminal liability the attempt to offer or prepare drugs referred to in Article 2(1)(a) and the attempt to possess drugs referred to in Article 2(1)(c). Article 4 Penalties 1. Each Member State shall take the measures necessary to ensure that the offences defined in Articles 2 and 3 are punishable by effective, proportionate and dissuasive criminal penalties. Each Member State shall take the necessary measures to ensure that the offences referred to in Article 2 are punishable by criminal penalties of a maximum of at least between one and three years of imprisonment. 2. Each Member State shall take the necessary measures to ensure that the offences referred to in Article 2(1)(a), (b) and (c) are punishable by criminal penalties of a maximum of at least between 5 and 10 years of imprisonment in each of the following circumstances: (a) the offence involves large quantities of drugs; (b) the offence either involves those drugs which cause the most harm to health, or has resulted in significant damage to the health of a number of persons. 3. Each Member State shall take the necessary measures to ensure that the offences referred to in paragraph 2 are punishable by criminal penalties of a maximum of at least 10 years of deprivation of liberty, where the offence was committed within the framework of a criminal organisation as defined in Joint Action 98/733/JHA of 21 December 1998 on making it a criminal offence to participate in a criminal organisation in the Member States of the European Union (5). 4. Each Member State shall take the necessary measures to ensure that the offences referred to in Article 2(1)(d) are punishable by criminal penalties of a maximum of at least between 5 and 10 years of deprivation of liberty, where the offence was committed within the framework of a criminal organisation as defined in Joint Action 98/733/JHA and the precursors are intended to be used in or for the production or manufacture of drugs under the circumstances referred to in paragraphs 2(a) or (b). 5. Without prejudice to the rights of victims and of other bona fide third parties, each Member State shall take the necessary measures to enable the confiscation of substances which are the object of offences referred to in Articles 2 and 3, instrumentalities used or intended to be used for these offences and proceeds from these offences or the confiscation of property the value of which corresponds to that of such proceeds, substances or instrumentalities. The terms confiscation, instrumentalities, proceeds and property shall have the same meaning as in Article 1 of the 1990 Council of Europe Convention on Laundering, Search, Seizure and Confiscation of the Proceeds from Crime. Article 5 Particular circumstances Notwithstanding Article 4, each Member State may take the necessary measures to ensure that the penalties referred to in Article 4 may be reduced if the offender: (a) renounces criminal activity relating to trafficking in drugs and precursors, and (b) provides the administrative or judicial authorities with information which they would not otherwise have been able to obtain, helping them to: (i) prevent or mitigate the effects of the offence, (ii) identify or bring to justice the other offenders, (iii) find evidence, or (iv) prevent further offences referred to in Articles 2 and 3. Article 6 Liability of legal persons 1. Each Member State shall take the necessary measures to ensure that legal persons can be held liable for any of the criminal offences referred to in Articles 2 and 3 committed for their benefit by any person, acting either individually or as a member of an organ of the legal person in question, who has a leading position within the legal person, based on one of the following: (a) a power of representation of the legal person; (b) an authority to take decisions on behalf of the legal person; (c) an authority to exercise control within the legal person. 2. Apart from the cases provided for in paragraph 1, each Member State shall take the necessary measures to ensure that legal persons can be held liable where the lack of supervision or control by a person referred to in paragraph 1 has made possible the commission of any of the offences referred to in Articles 2 and 3 for the benefit of that legal person by a person under its authority. 3. Liability of legal persons under paragraphs 1 and 2 shall not exclude criminal proceedings against natural persons who are perpetrators, instigators or accessories in any of the offences referred to in Articles 2 and 3. Article 7 Sanctions for legal persons 1. Member States shall take the necessary measures to ensure that a legal person held liable pursuant to Article 6(1) is punishable by effective, proportionate and dissuasive sanctions, which shall include criminal or non-criminal fines and may include other sanctions, such as: (a) exclusion from entitlement to tax relief or other benefits or public aid; (b) temporary or permanent disqualification from the pursuit of commercial activities; (c) placing under judicial supervision; (d) a judicial winding-up order; (e) temporary or permanent closure of establishments used for committing the offence; (f) in accordance with Article 4(5), the confiscation of substances which are the object of offences referred to in Articles 2 and 3, instrumentalities used or intended to be used for these offences and proceeds from these offences or the confiscation of property the value of which corresponds to that of such proceeds, substances or instrumentalities. 2. Each Member State shall take the necessary measures to ensure that a legal person held liable pursuant to Article 6(2) is punishable by effective, proportionate and dissuasive sanctions or measures. Article 8 Jurisdiction and prosecution 1. Each Member State shall take the necessary measures to establish its jurisdiction over the offences referred to in Articles 2 and 3 where: (a) the offence is committed in whole or in part within its territory; (b) the offender is one of its nationals; or (c) the offence is committed for the benefit of a legal person established in the territory of that Member State. 2. A Member State may decide that it will not apply, or that it will apply only in specific cases or circumstances, the jurisdiction rules set out in paragraphs 1(b) and 1(c) where the offence is committed outside its territory. 3. A Member State which, under its laws, does not extradite its own nationals shall take the necessary measures to establish its jurisdiction over and to prosecute, where appropriate, an offence referred to in Articles 2 and 3 when it is committed by one of its own nationals outside its territory. 4. Member States shall inform the General Secretariat of the Council and the Commission when they decide to apply paragraph 2, where appropriate with an indication of the specific cases or circumstances in which the decision applies. Article 9 Implementation and reports 1. Member States shall take the necessary measures to comply with the provisions of this Framework Decision by 12 May 2006. 2. By the deadline referred to in paragraph 1, Member States shall transmit to the General Secretariat of the Council and to the Commission the text of the provisions transposing into their national law the obligations imposed on them under this Framework Decision. The Commission shall, by 12 May 2009, submit a report to the European Parliament and to the Council on the functioning of the implementation of the Framework Decision, including its effects on judicial cooperation in the field of illicit drug trafficking. Following this report, the Council shall assess, at the latest within six months after submission of the report, whether Member States have taken the necessary measures to comply with this Framework Decision. Article 10 Territorial application This Framework Decision shall apply to Gibraltar. Article 11 Entry into force This Framework Decision shall enter into force on the day following its publication in the Official Journal of the European Union. Done at Luxembourg, 25 October 2004. For the Council The President R. VERDONK (1) OJ C 304 E, 30.10.2001, p. 172. (2) Opinion of 9 March 2004 (not yet published in the Official Journal). (3) OJ C 19, 23.1.1999, p. 1. (4) OJ L 167, 25.6.1997, p. 1. (5) OJ L 351, 29.12.1998, p. 1.
============================== "END OF DOC" ==============================
        

        doc 3 :

        'celex': 32013R1382
        'status': In Force
        'act_type': Regulation
        'treaty': TFEU (2008)

        full_doc :

        28.12.2013 EN Official Journal of the European Union L 354/73 REGULATION (EU) No 1382/2013 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL of 17 December 2013 establishing a Justice Programme for the period 2014 to 2020 (Text with EEA relevance) THE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE EUROPEAN UNION, Having regard to the Treaty on the Functioning of the European Union, and in particular Article 81(1) and (2), Article 82(1) and Article 84 thereof, Having regard to the proposal from the European Commission, After transmission of the draft legislative act to the national parliaments, Having regard to the opinion of the European Economic and Social Committee (1), Having regard to the opinion of the Committee of the Regions (2), Acting in accordance with the ordinary legislative procedure (3), Whereas: (1) The Treaty on the Functioning of the European Union (TFEU) provides for the creation of an area of freedom, security and justice, in which persons are free to move. To that end, the Union may adopt measures to develop judicial cooperation in civil and criminal matters and to promote and support the action of Member States in the field of crime prevention. Respect for fundamental rights as well as for common principles, such as non-discrimination, gender equality, effective access to justice for all, the rule of law and a well-functioning independent judicial system should be ensured in the further development of a European area of justice. (2) In the Stockholm Programme (4) the European Council reaffirmed the priority of developing an area of freedom, security and justice and specified as a political priority the achievement of a Europe of law and justice. Financing was identified as one of the important tools for the successful implementation of the Stockholm Programme's political priorities. The ambitious goals set by the Treaties and by the Stockholm Programme should be attained inter alia by establishing, for the period 2014 to 2020, a flexible and effective Justice Programme (the "Programme") which should facilitate planning and implementation. The general and specific objectives of the Programme should be interpreted in line with the relevant strategic guidelines defined by the European Council. (3) The Commission Communication of 3 March 2010 on the Europe 2020 Strategy sets out a strategy for smart, sustainable and inclusive growth. A well-functioning area of justice, where obstacles in cross-border judicial proceedings and access to justice in cross-border situations are eliminated, should be developed as a key element to support the specific objectives and flagship initiatives of the Europe 2020 Strategy and to facilitate mechanisms designed to promote growth. (4) For the purposes of this Regulation, the term "judiciary and judicial staff" should be interpreted so as to include judges, prosecutors and court officers, as well as other legal practitioners associated with the judiciary, such as lawyers, notaries, bailiffs, probation officers, mediators and court interpreters. (5) Judicial training is central to building mutual trust and improves cooperation between judicial authorities and practitioners in the various Member States. Judicial training should be seen as an essential element in promoting a genuine European judicial culture in the context of the Commission Communication of 13 September 2011 entitled "Building trust in EU-wide justice. A new dimension to European judicial training", the Council Resolution on the training of judges, prosecutors and judicial staff in the European Union (5), the Council conclusions of 27 and 28 October 2011 on European judicial training and the European Parliament resolution of 14 March 2012 on judicial training. (6) Judicial training can involve different actors, such as Member States' legal, judicial and administrative authorities, academic institutions, national bodies responsible for judicial training, European-level training organisations or networks, or networks of court coordinators of Union law. Bodies and entities pursuing a general European interest in the field of training of the judiciary, such as the European Judicial Training Network (EJTN), the Academy of European Law (ERA), the European Network of Councils for the Judiciary (ENCJ), the Association of the Councils of State and Supreme Administrative Jurisdictions of the European Union (ACA-Europe), the Network of the Presidents of Supreme Judicial Courts of the European Union (RPCSJUE) and the European Institute of Public Administration (EIPA), should continue to play their role in promoting training programmes with a genuine European dimension for the judiciary and judicial staff, and could therefore be granted adequate financial support in accordance with the procedures and the criteria set out in the annual work programmes adopted by the Commission pursuant to this Regulation. (7) The Union should facilitate training activities on the implementation of Union law by considering the salaries of participating judiciary and judicial staff incurred by the Member States' authorities as eligible costs or co-financing in kind, in accordance with Regulation (EU, Euratom) No 966/2012 of the European Parliament and of the Council (6) (the "Financial Regulation"). (8) Access to justice should include, in particular, access to courts, to alternative methods of dispute settlement and to public office-holders obliged by the law to provide parties with independent and impartial legal advice. (9) In December 2012 the Council endorsed the EU Drugs Strategy (2013-20) (7), which aims to take a balanced approach based on simultaneous reduction of drug demand and drug supply, acknowledging that drug demand reduction and drug supply reduction are mutually reinforcing elements in illicit drugs policy. That Strategy maintains as one of its main objectives the aim of contributing to a measurable reduction of drug demand, of drug dependence and of drug-related health and social risks and harms. Whereas the Drug prevention and information programme established by Decision No 1150/2007/EC of the European Parliament and of the Council (8) was based on a public health legal basis and covered those aspects, the Programme is founded on a different legal basis and should aim at the further development of a European area of justice based on mutual recognition and mutual trust, in particular by promoting judicial cooperation. Thus, in responding to the need for simplification and in line with the legal basis of each programme, the Health for Growth Programme can support measures to complement the Member's States action in attaining the objective of reducing drug-related health damage, including information and prevention. (10) Another important element of the EU Drugs Strategy (2013-20) is drug supply reduction. Whereas the Instrument for financial support for police cooperation, preventing and combating crime, and crisis management, as part of the Internal Security Fund, should support actions aimed at preventing and combating the trafficking of drugs and other types of crime, and in particular measures targeting the production, manufacture, extraction, sale, transport, importation and exportation of illegal drugs, including possession and purchase with a view to engaging in drug trafficking activities, the Programme should cover those aspects of drugs policy that are not covered by the Instrument for financial support for police cooperation, preventing and combating crime, and crisis management, as part of the Internal Security Fund, or by the Health for Growth Programme and are closely linked to its general objective. (11) In any case, the continued financing of the priorities under the 2007-2013 programming period that have been maintained as objectives under the new EU Drugs Strategy (2013-20) should be ensured, and funds should therefore be available from the Health for Growth Programme, the Instrument for financial support for police cooperation, preventing and combating crime, and crisis management, as part of the Internal Security Fund, and the Programme in accordance with their respective priorities and legal bases while avoiding any duplicate financing. (12) Pursuant to Article 3(3) of the Treaty on European Union (TEU), Article 24 of the Charter of Fundamental Rights of the European Union (the "Charter") and the 1989 United Nations Convention on the Rights of the Child, the Programme should support the protection of the rights of the child, including the right to due process, the right to understand the proceedings, the right to respect for private and family life and the right to integrity and dignity. The Programme should aim, in particular, to increase child protection within justice systems and access to justice for children, and should mainstream the promotion of the rights of the child in the implementation of all of its actions. (13) Pursuant to Articles 8 and 10 TFEU, the Programme should support the mainstreaming of equality between women and men and non-discrimination objectives in all its activities. Regular monitoring and evaluation should be carried out to assess the way in which gender equality and non-discrimination issues are addressed in the Programme's activities. (14) Experience of action at Union level has shown that achieving the objectives of the Programme in practice calls for a combination of instruments, including legal acts, policy initiatives and funding. Funding is an important tool complementing legislative measures. (15) In its conclusions of 22 and 23 September 2011 on improving the efficiency of future Union financial programmes supporting judicial cooperation, the Council stressed the important role played by Union financing programmes in the efficient implementation of the Union acquis and reiterated the need for more transparent, flexible, coherent and streamlined access to those programmes. (16) The Commission Communication of 29 June 2011 entitled 'A budget for Europe 2020' stresses the need for the rationalisation and simplification of Union funding. Especially in view of the current economic crisis, it is of the utmost importance that Union funds be structured and managed in the most diligent manner. Meaningful simplification and efficient management of funding can be achieved through a reduction in the number of programmes and through the rationalisation, simplification and harmonisation of funding rules and procedures. (17) In responding to the need for simplification, efficient management and easier access to funding, the Programme should continue and develop activities previously carried out on the basis of three programmes established by Council Decision 2007/126/JHA (9), Decision No 1149/2007/EC of the European Parliament and of the Council (10), and Decision No 1150/2007/EC. The mid-term evaluations of those programmes include recommendations aimed at improving the implementation of those programmes. The findings of those mid-term evaluations, as well as the findings of the respective ex-post evaluations, need to be taken into account in the implementation of the Programme. (18) The Commission Communication of 19 October 2010 entitled 'The EU Budget Review' and the Commission Communication of 29 June 2011 entitled 'A budget for Europe 2020' underline the importance of focusing funding on activities with clear European added value, i.e. where Union intervention can bring additional value compared to the action of Member States alone. Actions covered by this Regulation should contribute to the creation of a European area of justice by promoting the principle of mutual recognition, developing mutual trust between the Member States, increasing cross-border cooperation and networking and achieving the correct, coherent and consistent application of Union law. Funding activities should also contribute to achieving effective and better knowledge of Union law and policies by all concerned, and should provide a sound analytical basis for the support and the development of Union law and policies, in so doing contributing to their enforcement and proper implementation. Union intervention allows for those actions to be pursued consistently across the Union and brings economies of scale. Moreover, the Union is in a better position than Member States to address cross-border situations and to provide a European platform for mutual learning. (19) In selecting actions for funding under the Programme, the Commission should assess the proposals against pre-identified criteria. Those criteria should include an assessment of the European added value of the proposed actions. National projects and small-scale projects can also have European added value. (20) Bodies and entities that have access to the Programme should include national, regional and local authorities. (21) This Regulation lays down a financial envelope for the entire duration of the Programme which is to constitute the prime reference amount, within the meaning of point 17 of the Interinstitutional Agreement of 2 December 2013 between the European Parliament, the Council and the Commission on budgetary discipline, on cooperation in budgetary matters and on sound financial management (11), for the European Parliament and the Council during the annual budgetary procedure. (22) In order to ensure that the Programme is sufficiently flexible to respond to changing needs and corresponding policy priorities throughout its duration, the power to adopt acts in accordance with Article 290 TFEU should be delegated to the Commission concerning modification of the percentages set out in the Annex to this Regulation for each specific objective that would exceed those percentages by more than 5 percentage points. To assess the need for such a delegated act, those percentages should be calculated on the basis of the financial envelope of the Programme for its entire duration, and not on the basis of annual appropriations. It is of particular importance that the Commission carry out appropriate consultations during its preparatory work, including at expert level. The Commission, when preparing and drawing up delegated acts, should ensure a simultaneous, timely and appropriate transmission of relevant documents to the European Parliament and to the Council. (23) This Regulation should be implemented in full compliance with the Financial Regulation. In particular with regard to the eligibility conditions of value added tax (VAT) paid by grant beneficiaries, the eligibility of VAT should not depend on the legal status of the beneficiaries for activities which can be carried out by private and public bodies and entities under the same legal conditions. Taking into account the specific nature of the objectives and activities covered by this Regulation, it should be made clear, in calls for proposals, that, for activities which can be carried out by both public and private bodies and entities, the non-deductible VAT incurred by public bodies and entities is to be eligible, in so far as it is paid in respect of the implementation of activities, such as training or awareness-raising, which cannot be considered as the exercise of public authority. This Regulation should also make use of the simplification tools introduced by the Financial Regulation. Moreover, the criteria for identifying actions to be supported should aim at allocating the available financial resources to actions generating the highest impact in relation to the policy objective pursued. (24) In order to ensure uniform conditions for the implementation of this Regulation, implementing powers should be conferred on the Commission in respect of the adoption of annual work programmes. Those powers should be exercised in accordance with Regulation (EU) No 182/2011 of the European Parliament and of the Council (12). (25) The annual work programmes adopted by the Commission pursuant to this Regulation should ensure appropriate distribution of funds between grants and public procurement contracts. The Programme should primarily allocate funds to grants, while maintaining sufficient funding levels for procurement. The minimum percentage of annual expenditure to be allocated to grants should be established in the annual work programmes and should be not less than 65 %. To facilitate project planning and co-financing by stakeholders, the Commission should establish a clear timetable for the calls for proposals, selection of projects and award decisions. (26) In order to ensure efficient allocation of funds from the general budget of the Union, consistency, complementarity and synergies should be sought between funding programmes supporting policy areas with close links to each other, in particular between the Programme and the Rights, Equality and Citizenship Programme established by Regulation (EU) No 1381/2013 of the European Parliament and of the Council (13), the Instrument for financial support for police cooperation, preventing and combating crime, and crisis management, as part of the Internal Security Fund, the Health for Growth Programme, the Erasmus+ Programme established by Regulation (EU) No 1288/2013 of the European Parliament and of the Council (14), the Horizon 2020 Framework Programme established by Regulation (EU) No 1291/2013 of the European Parliament and of the Council (15) and the Instrument for Pre-accession Assistance (IPA II). (27) The financial interests of the Union should be protected through proportionate measures throughout the expenditure cycle, including the prevention, detection and investigation of irregularities, the recovery of funds lost, wrongly paid or incorrectly used and, where appropriate, the imposition of administrative and financial penalties in accordance with the Financial Regulation. (28) In order to implement the principle of sound financial management, this Regulation should provide for appropriate tools to assess its performance. To that end, it should define general and specific objectives. To measure the achievement of those specific objectives, a set of concrete and quantifiable indicators should be established which should remain valid for the whole duration of the Programme. The Commission should submit annually to the European Parliament and to the Council a monitoring report which should be based inter alia on the indicators set out in this Regulation and which should give information on the use of available funds. (29) The Programme should be implemented in an effective manner, respecting sound financial management, while also allowing potential applicants to have effective access to the Programme. In order to support effective access to the Programme, the Commission should use its best endeavours to simplify and harmonise the application procedures and documents, the administrative formalities and the financial management requirements, to remove administrative burdens and to encourage grant applications from entities located in Member States which are under-represented in the Programme. The Commission should publish on a dedicated webpage information about the Programme, its objectives, the various calls for proposals and their time schedules. Basic documents and guidelines relating to the calls for proposals should be available in all the official languages of the institutions of the Union. (30) In accordance with point (l) of Article 180(1) of Commission Delegated Regulation (EU) No 1268/2012 (16) ('the Rules of Application'), the grant agreements should lay down provisions governing the visibility of the Union financial support, except in duly justified cases where public display is not possible or appropriate. (31) In accordance with Article 35(2) and (3) of the Financial Regulation and Article 21 of the Rules of Application, the Commission should make available, in an appropriate and timely manner, information concerning recipients and concerning the nature and purpose of the measures financed from the general budget of the Union. That information should be made available with due observance of the requirements of confidentiality and security, in particular the protection of personal data. (32) Since the objective of this Regulation, namely to contribute to the further development of a European area of justice based on mutual recognition and mutual trust, in particular by promoting judicial cooperation in civil and criminal matters, cannot be sufficiently achieved by the Member States but can rather, by reason of its scale and effects, be better achieved at Union level, the Union may adopt measures, in accordance with the principle of subsidiarity as set out in Article 5 TEU. In accordance with the principle of proportionality, as set out in that Article, this Regulation does not go beyond what is necessary in order to achieve that objective. (33) In accordance with Article 3 of Protocol No 21 on the position of the United Kingdom and Ireland in respect of the Area of Freedom, Security and Justice, annexed to the TEU and to the TFEU, Ireland has notified its wish to take part in the adoption and application of this Regulation. (34) In accordance with Articles 1 and 2 of Protocol No 21 on the position of the United Kingdom and Ireland in respect of the Area of Freedom, Security and Justice, annexed to the TEU and to the TFEU, and without prejudice to Article 4 of that Protocol, the United Kingdom is not taking part in the adoption of this Regulation and is not bound by it or subject to its application. (35) In accordance with Articles 1 and 2 of Protocol No 22 on the position of Denmark, annexed to the TEU and to the TFEU, Denmark is not taking part in the adoption of this Regulation and is not bound by it or subject to its application. (36) In order to ensure the continuity of funding of activities previously carried out on the basis of Decision 2007/126/JHA, Decision No 1149/2007/EC and Decision No 1150/2007/EC, this Regulation should enter into force on the day following that of its publication, HAVE ADOPTED THIS REGULATION: Article 1 Establishment and duration of the Programme 1. This Regulation establishes a Justice programme ('the Programme'). 2. The Programme shall cover the period from 1 January 2014 to 31 December 2020. Article 2 European added value 1. The Programme shall finance actions with European added value which contribute to the further development of a European area of justice. To that end, the Commission shall ensure that the actions selected for funding are intended to produce results with European added value. 2. The European added value of actions, including that of small-scale and national actions, shall be assessed in the light of criteria such as their contribution to the consistent and coherent implementation of Union law and to wide public awareness about the rights deriving from it, their potential to develop mutual trust among Member States and to improve cross-border cooperation, their transnational impact, their contribution to the elaboration and dissemination of best practices or their potential to create practical tools and solutions that address cross-border or Union-wide challenges. Article 3 General objective The general objective of the Programme shall be to contribute to the further development of a European area of justice based on mutual recognition and mutual trust, in particular by promoting judicial cooperation in civil and criminal matters. Article 4 Specific objectives 1. To achieve the general objective set out in Article 3, the Programme shall have the following specific objectives: (a) to facilitate and support judicial cooperation in civil and criminal matters; (b) to support and promote judicial training, including language training on legal terminology, with a view to fostering a common legal and judicial culture; (c) to facilitate effective access to justice for all, including to promote and support the rights of victims of crime, while respecting the rights of the defence; (d) to support initiatives in the field of drugs policy as regards judicial cooperation and crime prevention aspects closely linked to the general objective of the Programme, in so far as they are not covered by the Internal security fund for financial support for police cooperation, preventing and combating crime, and crisis management or by the Health for Growth Programme; 2. The specific objectives of the Programme shall be pursued through, in particular: (a) enhancing public awareness and knowledge of Union law and policies; (b) with a view to ensuring efficient judicial cooperation in civil and criminal matters, improving knowledge of Union law, including substantive and procedural law, of judicial cooperation instruments and of the relevant case-law of the Court of Justice of the European Union, and of comparative law; (c) supporting the effective, comprehensive and consistent implementation and application of Union instruments in the Member States and the monitoring and evaluation thereof; (d) promoting cross-border cooperation, improving mutual knowledge and understanding of the civil and criminal law and the legal and judicial systems of the Member States and enhancing mutual trust; (e) improving knowledge and understanding of potential obstacles to the smooth functioning of a European area of justice; (f) improving the efficiency of judicial systems and their cooperation by means of information and communication technology, including the cross-border interoperability of systems and applications. Article 5 Mainstreaming In the implementation of all of its actions, the Programme shall seek to promote equality between women and men and to promote the rights of the child, inter alia by means of child-friendly justice. It shall also comply with the prohibition of discrimination based on any of the grounds listed in Article 21 of the Charter, in accordance with and within the limits set by Article 51 of the Charter. Article 6 Types of actions 1. The Programme shall finance inter alia the following types of actions: (a) analytical activities, such as the collection of data and statistics; the development of common methodologies and, where appropriate, indicators or benchmarks; studies, researches, analyses and surveys; evaluations; the elaboration and publication of guides, reports and educational material; workshops, seminars, experts meetings and conferences; (b) training activities, such as staff exchanges, workshops, seminars, train-the-trainer events, including language training on legal terminology, and the development of online training tools or other training modules for members of the judiciary and judicial staff; (c) mutual learning, cooperation, awareness-raising and dissemination activities, such as the identification of, and exchanges concerning, good practices, innovative approaches and experiences; the organisation of peer reviews and mutual learning; the organisation of conferences, seminars, information campaigns, including institutional communication on the political priorities of the Union as far as they relate to the objectives of the Programme; the compilation and publication of materials to disseminate information about the Programme and its results; the development, operation and maintenance of systems and tools, using information and communication technologies, including the further development of the European e-Justice portal as a tool to improve citizens' access to justice; (d) support for main actors whose activities contribute to the implementation of the objectives of the Programme, such as support for Member States in the implementation of Union law and policies, support for key European actors and European-level networks, including in the field of judicial training; and support for networking activities at European level among specialised bodies and entities as well as national, regional and local authorities and non-governmental organisations. 2. The European Judicial Training Network shall receive an operating grant to co-finance expenditure associated with its permanent work programme. Article 7 Participation 1. Access to the Programme shall be open to all bodies and entities legally established in: (a) Member States; (b) European Free Trade Association (EFTA) countries which are parties to the Agreement on the European Economic Area, in accordance with that Agreement; (c) candidate countries, potential candidates and countries acceding to the Union, in accordance with the general principles and the general terms and conditions laid down for the participation of those countries in the Union programmes established in the respective Framework Agreements and Association Council decisions, or similar agreements. 2. Bodies and entities which are profit-oriented shall have access to the Programme only in conjunction with non-profit or public organisations. 3. Bodies and entities legally established in third countries, other than those participating in the Programme in accordance with points (b) and (c) of paragraph 1, in particular countries where the European Neighbourhood Policy applies, may be associated to the actions of the Programme at their own cost, if this serves the purpose of those actions. 4. The Commission may cooperate with international organisations under the conditions laid down in the relevant annual work programme. Access to the Programme shall be open to international organisations active in the areas covered by the Programme in accordance with the Financial Regulation and the relevant annual work programme. Article 8 Budget 1. The financial envelope for the implementation of the Programme for the period 2014 to 2020 is set at EUR 377 604 000. 2. The financial allocation of the Programme may also cover expenses pertaining to preparatory, monitoring, control, audit and evaluation activities which are required for the management of the Programme and the assessment of the achievement of its objectives. The financial allocation may cover expenses relating to the necessary studies, meetings of experts, information and communication actions, including institutional communication of the political priorities of the Union, in so far as they are related to the general objectives of this Regulation, as well as expenses linked to information technology networks focusing on information processing and exchange and other technical and administrative assistance needed in connection with the management of the Programme by the Commission. 3. The annual appropriations shall be authorised by the European Parliament and the Council within the limits of the multiannual financial framework established by Council Regulation (EU, Euratom) No 1311/2013 (17). 4. Within the financial envelope for the Programme, amounts shall be allocated to each specific objective in accordance with the percentages set out in the Annex. 5. The Commission shall not depart from the allocated percentages of the financial envelope, as set out in the Annex, by more than 5 percentage points for each specific objective. Should it prove necessary to exceed that limit, the Commission shall be empowered to adopt delegated acts in accordance with Article 9 to modify each of the figures in the Annex by more than 5 and up to 10 percentage points. Article 9 Exercise of the delegation 1. The power to adopt delegated acts is conferred on the Commission subject to the conditions laid down in this Article. 2. The power to adopt delegated acts referred to in Article 8(5) shall be conferred on the Commission for the duration of the Programme. 3. The delegation of power referred to in Article 8(5) may be revoked at any time by the European Parliament or by the Council. A decision to revoke shall put an end to the delegation of the power specified in that decision. It shall take effect the day following the publication of the decision in the Official Journal of the European Union or at a later date specified therein. It shall not affect the validity of any delegated acts already in force. 4. As soon as it adopts a delegated act, the Commission shall notify it simultaneously to the European Parliament and to the Council. 5. A delegated act adopted pursuant to Article 8(5) shall enter into force only if no objection has been expressed either by the European Parliament or the Council within a period of two months of notification of that act to the European Parliament and the Council or if, before the expiry of that period, the European Parliament and the Council have both informed the Commission that they will not object. That period shall be extended by two months at the initiative of the European Parliament or the Council. Article 10 Implementing measures 1. The Commission shall implement the Programme in accordance with the Financial Regulation. 2. In order to implement the Programme, the Commission shall adopt annual work programmes in the form of implementing acts. Those implementing acts shall be adopted in accordance with the examination procedure referred to in Article 11(2). 3. Each annual work programme shall implement the objectives of the Programme by determining the following: (a) the actions to be undertaken, in accordance with the general and specific objectives set out in Article 3 and Article 4(1), including the indicative allocation of financial resources; (b) the essential eligibility, selection and award criteria to be used to select the proposals which are to receive financial contributions, in accordance with Article 84 of the Financial Regulation and with Article 94 of its Rules of Application; (c) the minimum percentage of annual expenditure to be allocated to grants. 4. Appropriate and fair distribution of financial support between different areas covered by this Regulation shall be ensured. When deciding on the allocation of funds to those areas in the annual work programmes, the Commission shall take into consideration the need to maintain sufficient funding levels for both civil justice and criminal justice, as well as for judicial training and initiatives in the field of drugs policy within the scope of the Programme. 5. Calls for proposals shall be published on an annual basis. 6. In order to facilitate judicial training activities, the costs associated with the participation of judiciary and judicial staff in those activities and incurred by the Member States' authorities shall be taken into account in accordance with the Financial Regulation when providing corresponding funding. Article 11 Committee procedure 1. The Commission shall be assisted by a committee. That committee shall be a committee within the meaning of Regulation (EU) No 182/2011. 2. Where reference is made to this paragraph, Article 5 of Regulation (EU) No 182/2011 shall apply. Article 12 Complementarity 1. The Commission, in cooperation with the Member States, shall ensure overall consistency, complementarity and synergies with other Union instruments including, inter alia, the Rights, Equality and Citizenship Programme, the Instrument for financial support for police cooperation, preventing and combating crime, and crisis management, as part of the Internal Security Fund, the Health for Growth Programme, the Erasmus+ Programme, the Horizon 2020 Framework Programme and the Instrument for Pre-accession Assistance (IPA II). 2. The Commission shall also ensure overall consistency, complementarity and synergies with the work of the Union bodies, offices and agencies operating in areas covered by the objectives of the Programme, such as Eurojust established by Council Decision 2002/187/JHA (18) and the European Monitoring Centre for Drugs and Drug Addiction (EMCDDA) established by Regulation (EC) No 1920/2006 of the European Parliament and of the Council (19). 3. The Programme may share resources with other Union instruments, in particular the Rights, Equality and Citizenship Programme, in order to implement actions meeting the objectives of both programmes. An action for which funding has been awarded from the Programme may also give rise to the award of funding from the Rights, Equality and Citizenship Programme, provided that the funding does not cover the same cost items. Article 13 Protection of the financial interests of the Union 1. The Commission shall take appropriate measures ensuring that, when actions financed under the Programme are implemented, the financial interests of the Union are protected by the application of preventive measures against fraud, corruption and any other illegal activities, by effective checks and, if irregularities are detected, by the recovery of amounts wrongly paid and, where appropriate, by effective, proportionate and dissuasive administrative and financial penalties. 2. The Commission or its representatives and the Court of Auditors shall have the power of audit, both on the basis of documents and on the spot, over all grant beneficiaries, contractors and subcontractors who have received Union funds under the Programme. 3. The European Anti-Fraud Office (OLAF) may carry out investigations, including on-the-spot checks and inspections, in accordance with the provisions and procedures laid down in Regulation (EU, Euratom) No 883/2013 of the European Parliament and of the Council (20) and in Council Regulation (Euratom, EC) No 2185/96 (21) with a view to establishing whether fraud, corruption or any other illegal activity has occurred affecting the financial interests of the Union in connection with a grant agreement or grant decision or a contract funded under the Programme. 4. Without prejudice to paragraphs 1, 2 and 3, cooperation agreements with third countries and with international organisations, grant agreements, grant decisions and contracts resulting from the implementation of the Programme shall contain provisions expressly empowering the Commission, the Court of Auditors and OLAF to conduct the audits and investigations referred to in those paragraphs, in accordance with their respective competences. Article 14 Monitoring and evaluation 1. The Commission shall monitor the Programme annually in order to follow the implementation of actions carried out under it and the achievement of the specific objectives set out in Article 4. The monitoring shall also provide a means of assessing the way in which gender equality and non-discrimination issues have been addressed across the Programme's actions. 2. The Commission shall provide the European Parliament and the Council with: (a) an annual monitoring report based on the indicators set out in Article 15(2) and on the use of the available funds; (b) an interim evaluation report by 30 June 2018; (c) an ex-post evaluation report by 31 December 2021. 3. The interim evaluation report shall assess the achievement of the Programme's objectives, the efficiency of the use of resources and the Programme's European added value with a view to determining whether funding in areas covered by the Programme should be renewed, modified or suspended after 2020. It shall also address the scope for any simplification of the Programme, its internal and external coherence, and the continued relevance of all objectives and actions. It shall take into account the results of the ex-post evaluations of the previous 2007-2013 programmes established by the Decisions referred to in Article 16. 4. The ex-post evaluation report shall assess the long-term impact of the Programme and the sustainability of the effects of the Programme, with a view to informing a decision on a subsequent programme. 5. The evaluations shall also assess the way in which gender equality and non-discrimination issues have been addressed across the Programme's actions. Article 15 Indicators 1. In accordance with Article 14, the indicators set out in paragraph 2 of this Article shall serve as a basis for monitoring and evaluating the extent to which each of the Programme's specific objectives set out in Article 4 has been achieved through the actions provided for in Article 6. They shall be measured against pre-defined baselines reflecting the situation before implementation. Where relevant, indicators shall be broken down by, inter alia, sex, age and disability. 2. The indicators referred to in paragraph 1 shall include, inter alia, the following: (a) the number and percentage of persons in a target group reached by awareness-raising activities funded by the Programme; (b) the number and percentage of members of the judiciary and judicial staff in a target group that participated in training activities, staff exchanges, study visits, workshops and seminars funded by the Programme; (c) the improvement in the level of knowledge of Union law and policies in the groups participating in activities funded by the Programme compared to the entire target group; (d) the number of cases, activities and outputs of cross-border cooperation, including cooperation by means of information technology tools and procedures established at Union level; (e) participants' assessment of the activities in which they participated and of their (expected) sustainability; (f) the geographical coverage of the activities funded by the Programme. 3. In addition to the indicators set out in paragraph 2, the interim and ex-post evaluation report of the Programme shall assess, inter alia: (a) the perceived impact of the Programme on access to justice based on qualitative and quantitative data collected at European level; (b) the number and quality of instruments and tools developed through actions funded by the Programme; (c) the European added value of the Programme, including an evaluation of the Programme's activities in the light of similar initiatives which have been developed at national or European level without support from Union funding, and their (expected) results and the advantages and/or disadvantages of Union funding compared to national funding for the type of activity in question; (d) the level of funding in relation to the outcomes achieved (efficiency); (e) the possible administrative, organisational and/or structural obstacles to the smoother, more effective and efficient implementation of the Programme (scope for simplification). Article 16 Transitional measures Actions initiated on the basis of Decision 2007/126/JHA, Decision 1149/2007/EC or Decision 1150/2007/EC shall continue to be governed by the provisions of those Decisions until their completion. In respect of those actions, reference to the committees provided for in Article 9 of Decision 2007/126/JHA, in Articles 10 and 11 of Decision 1149/2007/EC and in Article 10 of Decision 1150/2007/EC shall be interpreted as references to the committee provided for in Article 11(1) of this Regulation. Article 17 Entry into force This Regulation shall enter into force on the day following that of its publication in the Official Journal of the European Union. This Regulation shall be binding in its entirety and directly applicable in the Member States in accordance with the Treaties. Done at Brussels, 17 December 2013. For the European Parliament The President M. SCHULZ For the Council The President L. LINKEVIÃ IUS (1) OJ C 299, 4.10.2012, p. 103. (2) OJ C 277, 13.9.2012, p. 43. (3) Position of the European Parliament of 11 December 2013 (not yet published in the Official Journal) and decision of the Council of 16 December. (4) OJ C 115, 4.5.2010, p. 1. (5) OJ C 299, 22.11.2008, p. 1. (6) Regulation (EU, Euratom) No 966/2012 of the European Parliament and of the Council of 25 October 2012 on the financial rules applicable to the general budget of the Union and repealing Council Regulation (EC, Euratom) No 1605/2002 (OJ L 298, 26.10.2012, p. 1). (7) OJ C 402, 29.12.2012, p. 1. (8) Decision No 1150/2007/EC of the European Parliament and of the Council of 25 September 2007 establishing for the period 2007-2013 the Specific Programme 'Drug prevention and information' as part of the General Programme 'Fundamental Rights and Justice' (OJ L 257, 3.10.2007, p. 23). (9) Council Decision 2007/126/JHA of 12 February 2007 establishing for the period 2007-2013, as part of the General Programme on Fundamental Rights and Justice, the Specific Programme 'Criminal Justice' (OJ L 58, 24.2.2007, p. 13). (10) Decision No 1149/2007/EC of the European Parliament and of the Council of 25 September 2007 establishing for the period 2007-2013 the Specific Programme 'Civil Justice' as part of the General Programme 'Fundamental Rights and Justice (OJ L 257, 3.10.2007, p. 16). (11) OJ C 373, 20.12.2013, p. 1. (12) Regulation (EU) No 182/2011 of the European Parliament and of the Council of 16 February 2011 laying down the rules and general principles concerning mechanisms for control by Member States of the Commission's exercise of implementing powers (OJ L 55, 28.2.2011, p. 13). (13) Regulation (EU) No 1381/2013 of the European Parliament and of the Council of 17 December 2013 establishing a Rights, Equality and Citizenship Programme for the period 2014 to 2020 (see page 62 of this Official Journal). (14) Regulation (EU) No 1288/2013 of the European Parliament and of the Council of 11 December 2013 establishing "Erasmus+": the Union programme for education, training, youth and sport and repealing Decisions No 1719/2006/EC, No 1720/2006/EC and No 1298/2008/EC (OJ L 347, 20.12.2013, p. 50). (15) Regulation (EU) No 1291/2013 of the European Parliament and of the Council of 11 December 2013 establishing Horizon 2020 - the Framework Programme for Research and Innovation (2014-2020) and repealing Decision No 1982/2006/EC (OJ L 347, 20.12.2013, p. 104). (16) Commission Delegated Regulation (EU) No 1268/2012 of 29 October 2012 on the rules of application of Regulation (EU, Euratom) No 966/2012 of the European Parliament and of the Council on the financial rules applicable to the general budget of the Union (OJ L 362, 31.12.2012, p. 1). (17) Council Regulation (EU, Euratom) No 1311/2013 of 2 December 2013 laying down the multiannual financial framework for the years 2014-2020 (OJ L 347, 20.12.2013, p. 884). (18) Council Decision 2002/187/JHA of 28 February 2002 setting up Eurojust with a view to reinforcing the fight against serious crime (OJ L 63, 6.3.2002, p. 1). (19) Regulation (EC) No 1920/2006 of the European Parliament and of the Council of 12 December 2006 on the European Monitoring Centre for Drugs and Drug Addiction (OJ L 376, 27.12.2006, p. 1). (20) Regulation (EU, Euratom) No 883/2013 of the European Parliament and of the Council of 11 September 2013 concerning investigations conducted by the European Anti-Fraud Office (OLAF) and repealing Regulation (EC) No 1073/1999 of the European Parliament and of the Council and Council Regulation (Euratom) No 1074/1999 (OJ L 248, 18.9.2013, p. 1). (21) Council Regulation (Euratom, EC) No 2185/96 of 11 November 1996 concerning on-the-spot checks and inspections carried out by the Commission in order to protect the European Communities' financial interests against fraud and other irregularities (OJ L 292, 15.11.1996, p. 2). ANNEX ALLOCATION OF FUNDS Within the financial envelope for the Programme, amounts shall be allocated as follows to each specific objective set out in Article 4(1): Specific objectives Share of the financial envelope (in %) (a) to facilitate and support judicial cooperation in civil and criminal matters 30 % (b) to support and promote judicial training, including language training on legal terminology, with a view to fostering a common legal and judicial culture 35 % (c) to facilitate effective access to justice for all, including to promote and support the rights of victims of crime, while respecting the rights of the defence 30 % (d) to support initiatives in the field of drugs policy as regards judicial cooperation and crime prevention aspects closely linked to the general objective of the Programme, in so far as they are not covered by the Instrument for financial support for police cooperation, preventing and combating crime, and crisis management, as part of the Internal Security Fund, or by the Health for Growth Programme 5 %.
============================== "END OF DOC" ==============================
        

In [6]:
import pandas as pd

In [7]:
laws = pd.read_csv('dataset/act_raw_text_with_4meta.csv')

In [17]:
laws.columns


Index(['Unnamed: 0', 'CELEX', 'Status', 'Act_type', 'Treaty', 'act_raw_text'], dtype='str')

In [8]:
def get__embeddings(enhanced_query):
    query_embedding = embedding_model.embed_query(enhanced_query)

    return query_embedding

In [9]:
query_embedding = get__embeddings("driving without license penalty")

In [10]:
weaviate_client.is_live()

True

In [28]:
eu = weaviate_client.collections.use("Euro_Laws")
response = eu.query.near_vector(
    near_vector= query_embedding, 
    limit=5
)

for obj in response.objects:
    print(json.dumps(obj.properties, indent=10))  # Inspect the results

KeyboardInterrupt: 

In [11]:
vectorstore = WeaviateVectorStore(
    client = weaviate_client,
    index_name = "Euro_Laws",
    text_key="text",
    embedding = embedding_model
)

In [26]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
docs = retriever.invoke("driving without license penalty")
docs

ValueError: Error during query: Query call with protocol GRPC search failed with message Deadline Exceeded.

```
'celex': , 'status': 

'act_type' , 'treaty':

```

In [15]:
def get_celex_ids(docs):
    celex_ids = []
    for i in docs:
        celex_ids.append(i.metadata['celex'])

    celex_ids = list(set(celex_ids))

    return celex_ids

get_celex_ids(docs)  

['32015L0413', '32006L0126', '32009R1072', '31980L1263']

In [22]:
laws[laws['CELEX'] == '32015L0413']['Status'].iloc[0]

'In Force'

In [18]:
def search_docs(query):
    celex_ids = []
    full_doc_info = """ """

    retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
    docs = retriever.invoke(query)

    for doc in docs:
        celex_ids.append(doc.metadata['celex'])

    celex_ids = list(set(celex_ids))    

    for i , celex_id in enumerate(celex_ids):

        full_doc_info += f"""

        doc {i} :

        'celex': {laws[laws['CELEX'] == celex_id]['CELEX'].iloc[0]}
        'status': {laws[laws['CELEX'] == celex_id]['Status'].iloc[0]}
        'act_type': {laws[laws['CELEX'] == celex_id]['Act_type'].iloc[0]}
        'treaty': {laws[laws['CELEX'] == celex_id]['Treaty'].iloc[0]}

        full_doc :

        {laws[laws['CELEX'] == celex_id]['act_raw_text'].iloc[0]}
{"=="*15} "END OF DOC" {"=="*15}
        """

    return full_doc_info    



In [ ]:
display(Markdown(search_docs("driving without license penalty"))) 

 

        doc 0 :

        'celex': 31980L1263
        'status': Not in Force
        'act_type': Directive
        'treaty': TEEC

        full_doc :

        Avis juridique important|31980L1263First Council Directive 80/1263/EEC of 4 December 1980 on the introduction of a Community driving licence Official Journal L 375 , 31/12/1980 P. 0001 - 0015 Finnish special edition: Chapter 7 Volume 2 P. 0171 Greek special edition: Chapter 13 Volume 10 P. 0089 Swedish special edition: Chapter 7 Volume 2 P. 0171 Spanish special edition: Chapter 07 Volume 2 P. 0259 Portuguese special edition Chapter 07 Volume 2 P. 0259 FIRST COUNCIL DIRECTIVE of 4 December 1980 on the introduction of a Community driving licence (80/1263/EEC) THE COUNCIL OF THE EUROPEAN COMMUNITIES, Having regard to the Treaty establishing the European Economic Community, and in particular Article 75 (1) (c) thereof, Having regard to the proposal from the Commission, Having regard to the opinion of the European Parliament (1), Having regard to the opinion of the Economic and Social Committee (2), Whereas, for the purposes of the common transport policy, as a contribution to improving road traffic safety, and to assist the movement of persons settling in a Member State other than that in which they have passed a driving test, or moving within the Community, it is desirable that a Community driving licence be introduced; Whereas the introduction of a Community driving licence presupposes the harmonization of existing national driving test arrangements, which can only be achieved gradually ; whereas the first stage of this harmonization could culminate in the establishment of a Community model national licence and the mutual recognition by Member States of national driving licences and the exchange of licences by holders transferring their place of residence or place of employment from one Member State to another; Whereas the Community model national licence should be based on that defined by the Final Act of the Convention on Road Traffic drawn up in Vienna in November 1968 by the United Nations Road Traffic Conference; Whereas the mutual recognition of driving licences issued by the different Member States and the exchange of a licence by a holder moving from one Community country to reside or work in another will only be possible further to an initial harmonization of the regulations governing the issue and validity of licences; Whereas, without prejudice to the final provisions to be adopted by the Council on vehicle categories, it is necessary to establish common standards in respect of the validity of the licence for driving the different categories of vehicles, so that the Community model licence can be issued throughout the Community under comparable conditions; Whereas, however, at this initial harmonization stage and pending the introduction of the final system, Member States should be allowed to lay down the conditions with regard to age and the period of validity of licences arid also, under certain specific conditions, to derogate from the categories, speeds and conditions of validity laid down by this (1)OJ No C 238, 11.10.1976, p. 43. (2)OJ No C 197, 23.8.1976, p. 32. Directive ; and, where appropriate, to check the additional conditions laid down for the exchange of driving licences of certain categories of vehicles; Whereas it is desirable that the standards for testing drivers and issuing licences should be further harmonized as soon as possible, HAS ADOPTED THIS DIRECTIVE: Article 1 The Member States shall introduce a national driving licence based on the Community model provided for in Article 2. A Community model driving licence shall, subject to Article 8, entitle the holder to drive, both on national and international journeys, vehicles of the categories for which it has been granted. Community model driving licences shall be issued by the Member States in accordance with this Directive. Article 2 The driving licence provided for in Article 1 shall conform to the model in Annex I. The oval on page 1 of the model shall contain the distinguishing sign of the State issuing the licence. After consulting the Commission, Member States may adapt the model in the Annex in any way necessary to enable them to: - process the driving licence by computer, - enter in the licence any categories of vehicle which, pursuant to Article 9, differ from those provided for in Article 3. Member States shall take all necessary steps to avoid any risk of forgery of driving licences. Article 3 1. Without prejudice to the final provisions to be adopted by the Council concerning vehicle categories, the driving licence provided for in Article I shall authorize the driving on public roads of vehicles in the following categories: category A : motorcycles with or without side-car; category B : motor vehicles, other than those in category A, with a permissible maximum weight not exceeding 3 500 kg and not more than eight seats in addition to the driver's seat; category C : motor vehicles used for the carriage of goods and whose permissible maximum weight exceeds 3 500 kg; category D : motor vehicles used for the carriage of passengers, with more than eight seats in addition to the driver's seat: category E : combinations of vehicles of which the tractor vehicle is in a category or categories for which the driver is licensed (B and/or C and/or D), but which are not themselves in that category or categories. 2. For the purposes of paragraph 1: (a) a trailer with a permissible maximum weight not exceeding 750 kg may be coupled to a motor vehicle in category B above ; a trailer with a permissible maximum weight exceeding 750 kg may likewise be coupled to such vehicle, provided that the following two conditions are fulfilled: - the permissible maximum weight of the trailer does not exceed the unladen weight of the motor vehicle, and - the total permissible maximum weight of the combination of vehicles does not exceed 3 500 kg; (b) a motor vehicle in category C or D may be coupled to a trailer the permissible maximum weight of which does not exceed 750 kg. 3. For the purposes of this Article: - "motorcycle" means any two or three-wheeled vehicle with a maximum design speed exceeding 50 kph (33 mph) or, if it is powered by a heat propulsion engine, with a cylinder capacity exceeding 50 cc. In addition, in the case of a three-wheeled vehicle, the unladen weight shall not exceed 400 kg; - "power-driven vehicle" means any self-propelled vehicle running on a road, other than a railborne vehicle; - "motor vehicle" means any power-driven vehicle, other than a motorcycle, which is normally used for carrying persons or goods by road or for drawing, on the road, vehicles used for the carriage of persons or goods. This term shall include trolley buses, i.e. vehicles connected to an electric conductor and not rail borne. It shall not include agricultural or forestry tractors; - "agricultural or forestry tractor" means any power-driven vehicle running on wheels or tracks, having at least two axles, the principal function of which lies in its tractive power, which is specially designed to pull, push, carry or operate certain tools, machines or trailers used in connection with agricultural or forestry operations, and the use of which for carrying persons or goods by road or for drawing, on the road, vehicles used for the carriage or persons or goods is only a secondary function. Article 4 1. The validity of the driving licence provided for in Article 1 shall be determined as follows: (a) licences granted for categories C and D shall also be valid for the driving of vehicles in category B; (b) licences granted for category E shall, without prejudice to the provisions of (c), be valid for the driving of combinations of vehicles; (c) licences for category E shall be granted only to drivers already entitled to drive vehicles in category B, C or D. 2. Licences issued to disabled drivers shall specifically mention the conditions under which such drivers are entitled to drive. Article 5 1. Without prejudice to Article 5 of Council Regulation (EEC) No 543/69 of 25 March 1969 on the harmonization of certain social legislation relating to road transport (1), Member States shall fix the minimum age at which driving licences may be issued. 2. Member States may refuse to recognize the validity on their territory of driving licences issued to drivers under the age of 18 years. Article 6 1. A driving licence shall, moreover, be issued only to those applicants: (a) who have passed a practical and theoretical test and who meet medical standards, the minimum requirements of which may not be substantially less stringent than those set out in Annexes II and III; (b) who have their normal residence in the territory of the Member State issuing the licence, if the legislation of the Member State concerned so requires. 2. Member States may apply to the issue of driving licences the provisions of their national legislation relating thereto which are concerned with conditions other than those referred to in paragraph 1. Article 7 Without prejudice to the provisions which may be adopted by the Council in this regard, each Member State shall retain the right to fix, on the basis of national criteria, the period of validity of the driving licences (Community model) which it issues or exchanges pursuant to Article 8. Article 8 1. The Member States shall provide that, if the holder of a valid national driving licence or valid Community model licence issued by a Member State takes up normal residence in another Member State his licence shall remain valid there for up to a maximum of a year following the taking up of residence At the request of the holder within that period, and against surrender of his licence, the State in which he has taken up normal residence shall issue him with a driving licence (Community model) for the corresponding category or categories without subjecting him to the conditions laid down in Article 6. However, that Member State may refuse to exchange the licence if its national regulations, including medical standards, preclude the issue of the licence. The exchange must be preceded by the submission of a statement by the applicant to the effect that his (1)OJ No L 77, 29.3.1969, p. 49. driving licence is currently valid. It shall be for the Member State effecting the exchange to check the veracity of his statement if necessary. The Member State effecting the exchange shall return the old licence to the authorities of the Member State which issued it. 2. Member States which, pursuant to Article 9, do not apply categories C, D and E as defined in Article 3 (1) may: - exchange category C, D and E driving licences in accordance with paragraph 1 of this Article or, - require the applicant to furnish proof of driving experience and in this case issue a licence entitling him to drive vehicles in the national category in respect of which he furnished proof of adequate experience, or vehicles in a lower category. In any event, such States shall issue to the applicant at least a licence to drive vehicles in the lowest of the national categories corresponding to categories C, D and E as defined in Article 3 (1). During the year following the taking up of residence by drivers who have not applied for a licence exchange, such States shall recognize such drivers' licences as being equivalent at least to licences for the lowest relevant national category. 3. Where a Member State exchanges a licence, issued by a third country, for a Community model driving licence, such exchange shall be recorded in the licence, as shall any subsequent renewal or replacement of that licence. In the event of subsequent exchange of the said licence, Member States shall not be obliged to apply paragraph 1. A Community model driving licence may in any event be issued only if the licence issued by the third country has been surrendered to the competent authorities of the Member State issuing the Community licence. Article 9 After consulting the Commission, Member States may, pending introduction of the final system and provided that the fact is recorded on the licence, derogate from: - the categories defined in Article 3 (1); - the speeds indicated in the first indent of Article 3 (3), provided that the speeds which they prescribe are lower; - the conditions of validity provided for in Article 4. Furthermore, Member States shall, pursuant to the procedure laid down in Article 12, establish equivalent definitions in so far as their national categories differ. Article 10 The Council, acting on a proposal from the Commission, shall carry out as soon as possible a more detailed harmonization of the standards for driving tests and licensing with a view to inter alia subsequent improvements in road safety throughout the Community. Article 11 The Member States shall determine the arrangements for replacing currently valid national driving licences issued by them with Community model driving licences for the corresponding category or categories. This operation shall take place without the need for the tests provided for under Article 6, on submission of and in exchange for the old licences. Article 12 1. After consulting the Commission, Member States shall, in good time and at the latest by 30 June 1982, adopt the laws, regulations or administrative provisions necessary for the implementation of this Directive from 1 January 1983. 2. However, a Member State may, without prejudice to the application of the other provisions in this Directive, decide not to issue Community model driving licences until a later date, which may not be later than 1 January 1986. 3. Member States shall assist one another in the implementation of this Directive. Article 13 This Directive is addressed to the Member States. Done at Brussels, 4 December 1980. For the Council The President J. BARTHEL ANNEX I >PIC FILE= "T0014238">Comments on the model driving licence shown on page 1 1. The colour of the Community driving licence shall be pink. 2. On the cover page: - mention of the name of the Member State issuing the licence shall be optional, - the distinguishing sign of the Member State issuing the licence shall be entered in the oval, - the words "driving licence" shall be printed in large type in the language or languages of the Member State issuing the licence. They shall appear, alter a suitable space, in small type in the other languages of the European Communities, - the words "European Communities model" shall be printed in the language or languages of the Member State issuing the licence. 3. The printed entries on the other pages shall be in the language or languages of the Member State issuing the licence. 4. The page entitled "Additional information" is designed for details of any restriction or extension of the conditions governing the validity of the licence. This page may also be used for showing the period of validity of the licence where this varies. >PIC FILE= "T0014239"> 5. Other comments may be entered on the remaining blank pages. Where appropriate, Member States may enter on them categories of vehicles not covered by this Directive or may subdivide categories A, B, C, D and E in the corresponding page. 6. Member States shall have the right to: - dispense with the photograph requirement; - replace the permanent place of residence by the postal address; - delete the date of issue and indicate the date of commencement of validity of the licence. SPECIMEN COMMUNITY MODEL LICENCE : BELGIAN LICENCE (FOR INFORMATION) >PIC FILE= "T0014240"> ANNEX II MINIMUM REQUIREMENTS FOR DRIVING TESTS THEORETICAL TEST Form 1. The form chosen shall be such as to establish whether the candidate has the required knowledge and understanding of the subjects listed in paragraphs 2 and 3 of this Annex. Content 2. Knowledge and understanding of the regulations, and more especially of the rules applicable to the use of vehicles of the category corresponding to the type of licence applied for: 2.1. Knowledge and understanding of traffic rules and regulations, signs, signals and road markings and of their meaning; 2.2. Basic knowledge and understanding of the technical regulations relating to vehicle safety in traffic; 2.3. Knowledge and understanding of rules relating to the driver, in so far as they concern road safety, including, for drivers of category C and D vehicles only, rules relating to hours of work and rest periods; 2.4. Knowledge and understanding of the rules on what a driver should do in the event of an accident. 3. Knowledge and understanding of other subjects: 3.1. Adequate knowledge and understanding of the importance of road safety matters, and especially of the following accident factors: 3.1.1. Driving hazards, such as the danger of overtaking, misjudgement of speed (effects on braking and safety distances), influence of the weather (snow, rain, fog, side-winds, aquaplaning), behaviour of other road users, and in particular of elderly people and children; 3.1.2. Factors likely to reduce the driver's vigilance and his physical and mental fitness to drive, such as fatigue, illness, alcohol and other drugs, etc.; 3.1.3. Safety factors relating to vehicle loading and to passengers carried. 3.2. Category A and B vehicles only : basic knowledge of those items of the vehicle which are vital to the protection of its occupants and to road safety, such as brakes, tyres, oil levels, safety belts, etc.; Category C, D and E vehicles only : knowledge of the function and simple maintenance of the items mentioned above and of all other vehicle parts and devices of particular importance to safety; 3.3. Knowledge of the action which may be required in order to assist road accident victims. PRACTICAL TEST The vehicle and its equipment 4. If a candidate takes the test on a vehicle with automatic transmission, this shall be recorded on any licence issued on the basis of such a test; - Category C vehicles : the permissible maximum weight shall be not less than 7 000 kg; - Category D vehicles : the vehicle shall have not less than 28 seats and shall be not less than 7 m in length; - Category E vehicles : when the towing vehicle belongs to category C, and except in the case of a semi-trailer, the trailer shall have at least two axles, the distance between which shall be greater than 1 m. Contents 5. The principal manoeuvres to be carried out to check the candidate's ability to control the vehicle are as follows: 5.1. Starting on upgrades; 5.2. Category B, C, D and E vehicles only : reversing and reverse turning: 5.3. Braking and stopping at various speeds, including emergency stops if road and traffic conditions permit; 5.4. Category B, C, D and E vehicles only : oblique parking, parking on upgrades and down-grades; 5.5. Turning in a restricted space; 5.6. Category A vehicles only : riding at a slow speed. 6. Behaviour in traffic The main checks to which the candidate will be subjected are: 6.1. Correct positioning on the carriageway; 6.2. Proper negotiation of right and left-hand bends; 6.3. Correct execution of the manoeuvres of changing lanes and turning off at junctions; 6.4. Alertness to other traffic; 6.5. Correct behaviour at intersections, taking due account of all movements of other road users, with special regard to right-of-way; 6.6. Driving at appropriate speeds; 6.7. Use of rear-view mirrors; 6.8. Correct signalling of intended manoeuvres; 6.9. Correct operation of vehicle lighting and warning devices and other ancillary controls; 6.10. Driving with due care and consideration for pedestrians and other road users; 6.11. Correct behaviour with regard to public transport vehicles; 6.12. Compliance with traffic-light signals and instructions given by authorized officials on point duty; 6.13. Appropriate reaction to legally specified signals given by other road users; 6.14. Observance of traffic signs and signals, road markings and pedestrian crossings; 6.15. Observance of appropriate following and lateral distances; 6.16. Correct overtaking; 6.17 Correct use of safety belts if national legislation requires that they be fitted to the vehicle. Sequence of the parts of the test 7. Whenever possible, the part of the test described in paragraph 5 should be carried out before the part described in paragraph 6. Duration of the test 8. The duration of the test and the distance covered shall be sufficient for the checks prescribed in paragraphs 5 and 6 to be carried out. The duration of the part of the test described in paragraph 6 should be more than 30 minutes, but shall not in any case be less than 20 minutes. Location of the test 9. The part of the test described in paragraph 5 may be conducted on a special testing ground, in which case precise criteria should be laid down for measuring objectively the candidate's ability to handle the vehicle. The part of the test described in paragraph 6 shall, wherever possible, be conducted on roads outside built-up areas and on motorways as well as in urban traffic. ANNEX III MINIMUM STANDARDS OF PHYSICAL AND MENTAL FITNESS DEFINITIONS 1. For the purpose of this Annex, drivers are classified into two groups: 1.1. Group 1 : drivers of vehicles of categories A and B; 1.2. Group 2 : drivers of vehicles of categories C, D and E. 2. Similarly, applicants for a first driving licence or for the renewal of a driving licence are classified in the group to which they will belong once the licence has been granted or renewed. MEDICAL EXAMINATIONS 3. Group 1 : applicants shall be required to undergo a medical examination if it becomes apparent, when the necessary formalities are being completed or during the tests which they have to undergo prior to obtaining a driving licence, that they have one or more of the medical disabilities mentioned in this Annex in respect of this group. 4. Group 2 : applicants shall undergo a medical examination before a driving licence is first granted to them and thereafter drivers shall undergo such periodic examinations as may be prescribed by national laws. Eyesight 5. An examination conducted by suitably trained personnel shall be undergone by all applicants for a driving licence. In doubtful cases the applicant shall be referred to a competent medical authority. At the medical examination, attention should be paid to visual acuity, field of vision, night vision, progressive eye diseases, etc. When the wearing of corrective lenses is recognized by the issuing authority as necessary for driving, this shall be recorded on the driving licence. 6. Group 1 : drivers in this group should have their eyesight tested not later than at the age of 70 and preferably earlier, and thereafter at appropriate intervals. If applicants or drivers aged 40 years or more have sub-normal vision after correction but nevertheless meet the minimum requirements given in paragraphs 6.1 and 6.2 below, the cause of loss of vision shall be investigated before driving licences are granted or renewed. Where a disease of the eye is discovered or suspected, the periodic tests should be frequent. 6.1. Applicants for a driving licence or for the renewal of such a licence shall have a visual acuity, with corrective lenses if necessary, of at least 0 74, and preferably of a higher standard in the better eye or of at least 0 75 in both eyes together and, on medical examination, of at least 0 72 in the worse eye. Driving licences shall not be granted or renewed if, on examination, it is shown that there is more than 20 º loss in the temporal part of the applicant's or the driver's field of vision, or if the applicant or driver has diplopia or defective binocular vision. 6.2. Applicants or drivers with sight only in one eye may obtain a driving licence or the renewal of such a licence if the monocular vision is certified by a competent medical authority as having existed for sufficient time to allow adaptation and the visual acuity, with corrective lenses if necessary, is at least 0 78. Such persons must have unrestricted field of vision in their good eye. 7. Group 2 : applicants or drivers in this group shall have their eyesight tested on application for a driving licence and preferably periodically thereafter. If applicants or drivers aged 40 years or more have sub-normal vision after correction but nevertheless meet the minimum requirements given in paragraph 7.1 below, the cause of visual loss shall be investigated before driving licences are granted or renewed. 7.1. Applicants for a driving licence or for the renewal of such a licence must have binocular vision with a visual acuity, with corrective lenses if necessary, of at least 0 775 in the better eye and of at least 0 75 in the worse eye. If corrective lenses are used, the uncorrected vision must not be less than 0 71 and the correction must be tolerated. Driving licences shall not be granted or renewed if the applicant or driver has a restricted field of vision or if he has diplopia or defective binocular vision. 7.2. The use of contact lenses by drivers in this group may be permitted if approved by a competent medical authority. Hearing 8. Driving licences shall not be granted or renewed for applicants or drivers in group 2 if their hearing is so bad that it interferes with the proper discharge of their duties. General physique and physical disabilities 9. Group 1 : unrestricted driving licences shall not be granted or renewed for physically disabled applicants or drivers, unless a driving test has established their ability to operate vehicles with conventional controls. 9.1. Restricted driving licences may be granted or renewed for physically disabled applicants or drivers if the vehicles they drive are adapted to suit the requirements of their disablement. Any restriction on the driving licence shall state the adaptation required on the vehicle. 9.2. In cases of doubt, a practical test shall be made of driving abilities after medical examination by a competent authority and, where appropriate, a driving licence for a limited duration may be issued so as to keep a case under observation. The assessment of physical disablement shall primarily be based on mechanical considerations which make it possible to ascertain whether the disablement is likely to interfere for prolonged periods with efficient and rapid manoeuvring and the handling of controls under all driving conditions, especially in an emergency. 10. Group 2 : driving licences shall not be granted or renewed for applicants or drivers who have any disablement which is likely to prevent proper and safe control of a vehicle. 10.1. Medical examination of applicants or drivers shall cover the full range of body movements - strength, control and coordination - and, in particular, movements of the upper and lower limbs. 10.2. If disablement which is likely to hinder proper and safe control of a vehicle occurs after a driving licence has been granted, the disabled person must give up driving and undergo an examination by a competent medical authority. Cardiovascular diseases 11. Driving licences shall not be granted or renewed for applicants or drivers with cardiovascular diseases unless their request is supported by authorized medical opinion. 12. With regard to applicants or drivers in group 2, the competent medical authority shall give due consideration to the additional risks and dangers involved in the driving of vehicles covered by the definition of this group. Endocrine disorders 13. In cases of severe endocrine disorders other than diabetes, appropriate provisions in respect of the granting or renewal of driving licences shall be established by the laws of the Member States. 14. Group 1 : driving licences shall not be granted or renewed for applicants or drivers suffering from diabetes who are affected by ocular, nervous or cardiovascular complications or uncompensated acidosis. 14.1. Driving licences may be granted or renewed for a restricted period for applicants or drivers suffering from diabetes who are not affected by any of the complications mentioned in paragraph 14 above, subject to their remaining under authorized medical supervision. 15. Group 2 : driving licences shall not be granted or renewed for applicants or drivers who are diabetics needing insulin treatment. Diseases of the nervous system 16. Driving licences shall not be granted or renewed for applicants or drivers suffering from (a) encephalitis, multiple sclerosis, myasthenia gravis or hereditary diseases of the nervous system associated with progressive muscular atrophy and congenital myotonic disorders; (b) diseases of the peripheral nervous system ; or (c) trauma of the central or peripheral nervous system, unless their application is supported by authorized medical opinion and they are able to handle the controls of a vehicle safely and to comply with traffic regulations. Such cases shall be reviewed at regular intervals. 17. Group 1 : driving licences shall not be granted or renewed for applicants or drivers suffering from epilepsy. National legislation may provide that, subject to authorized medical opinion, licences be granted to persons who have suffered from epilepsy in the past but who have been free from attacks for a long time (e.g. two years). 17.1. Driving licences shall not be granted or renewed for applicants or drivers suffering from cerebrovascular diseases, unless their application is supported by authorized medical opinion and provided that, where necessary, the controls of the vehicle they drive are suitably re-arranged or modified, or that suitable special types of vehicles are used. The duration of the validity of driving licences granted or renewed in such cases shall be limited in accordance with authorized medical opinion. 17.2. Driving licences shall not be granted or renewed for applicants or drivers who have suffered a lesion with damage to the spinal cord and resultant paraplegia unless the vehicle they drive is fitted with special controls. 18. Group 2 : driving licences shall not be granted or renewed for applicants or drivers who suffer or have suffered in the past from epilepsy, a cerebrovascular disease or a lesion with damage to the spinal cord and resulting paraplegia. Mental disorders 19. Driving licences shall not be granted or renewed for applicants or drivers who: (a) suffer from mental disturbance due to disease or trauma of, or operations upon, the central nervous system; (b) suffer from severe mental retardation; (c) suffer from psychosis, which in particular has caused general paralysis ; or (d) suffer from psychoneurosis or personality disorders. unless their application is supported by authorized medical opinion. 20. With regard to applicants or drivers in group 2, the authorized medical authority shall give due consideration to the additional risks and dangers involved in driving the vehicles covered by this group. Alcohol 21. Driving licences shall not be granted or renewed for applicants or drivers who suffer from chronic alcoholism. If the application is supported by an authorized medical opinion, driving licences may be granted or renewed for a limited period for applicants or drivers who suffered from chronic alcoholism in the past. Such cases shall be reviewed at regular intervals. 22. With regard to applicants or drivers in group 2, the authorized medical authority shall give due consideration to the additional risks and dangers involved in driving the vehicles covered by this group. Drugs and medicaments 23. Drug abuse : driving licences shall not be granted or renewed for applicants or drivers who are dependant on psycho-active drugs. 24. Drugs or medicaments taken on a regular basis : driving licences shall not be granted or renewed for applicants or drivers who regularly take drugs or medicaments which can hamper the ability to drive safely, unless their application is supported by authorized medical opinion. 24.1. With regard to applicants or drivers in group 2, the authorized medical authority shall give due consideration to the additional risks and dangers involved in driving the vehicles covered by this group. Diseases of the blood 25. Driving licences shall not be granted or renewed for applicants or drivers suffering from serious diseases of the blood unless the application is supported by authorized medical opinion. Diseases of the genito-urinary system 26. Driving licences shall not be granted or renewed for applicants or drivers suffering from severe renal deficiency. WITHDRAWAL OF DRIVING LICENCES 27. National laws shall include provisions to the effect that, subject to authorized medical opinion, a driving licence shall be withdrawn where the authorities concerned have become aware that the holder's state of health is such that his application for a licence or for its renewal would have been refused. OTHER PROVISIONS (i) The provisions of the Annex shall not prevent a Member State from providing that a driver who has obtained a driving licence before 1 January 1983 under less stringent conditions than those provided for herein may have this licence regularly renewed under the conditions pertaining when he obtained it. (ii) Member States may derogate from the provisions of the Annex where the development of medical science makes such derogations fully compatible with the standards laid down herein. These derogations shall apply only to applicants who have undergone a medical examination and whose application is supported by authorized medical opinion.
============================== "END OF DOC" ==============================
        

        doc 1 :

        'celex': 32009R1072
        'status': In Force
        'act_type': Regulation
        'treaty': TEC (1992)

        full_doc :

        14.11.2009 EN Official Journal of the European Union L 300/72 REGULATION (EC) No 1072/2009 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL of 21 October 2009 on common rules for access to the international road haulage market (recast) (Text with EEA relevance) THE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE EUROPEAN UNION, Having regard to the Treaty establishing the European Community, and in particular Article 71 thereof, Having regard to the proposal from the Commission, Having regard to the opinion of the European Economic and Social Committee (1), After consulting the Committee of the Regions, Acting in accordance with the procedure laid down in Article 251 of the Treaty (2), Whereas: (1) A number of substantial changes are to be made to Council Regulation (EEC) No 881/92 of 26 March 1992 on access to the market in the carriage of goods by road within the Community to or from the territory of a Member State or passing across the territory of one or more Member States (3), to Council Regulation (EEC) No 3118/93 of 25 October 1993 laying down the conditions under which non-resident carriers may operate national road haulage services within a Member State (4), and to Directive 2006/94/EC of the European Parliament and of the Council of 12 December 2006 on the establishment of common rules for certain types of carriage of goods by road (5). In the interests of clarity and simplification, those legal acts should be recast and incorporated into one single regulation. (2) The establishment of a common transport policy entails, inter alia, laying down common rules applicable to access to the market in the international carriage of goods by road within the territory of the Community, as well as laying down the conditions under which non-resident hauliers may operate transport services within a Member State. Those rules must be laid down in such a way as to contribute to the smooth operation of the internal transport market. (3) To ensure a coherent framework for international road haulage throughout the Community, this Regulation should apply to all international carriage on Community territory. Carriage from Member States to third countries is still largely covered by bilateral agreements between the Member States and those third countries. Therefore, this Regulation should not apply to that part of the journey within the territory of the Member State of loading or unloading as long as the necessary agreements between the Community and the third countries concerned have not been concluded. It should, however, apply to the territory of a Member State crossed in transit. (4) The establishment of a common transport policy implies the removal of all restrictions against the person providing transport services on the grounds of nationality or the fact that he is established in a different Member State from the one in which the services are to be provided. (5) In order to achieve this smoothly and flexibly, provision should be made for a transitional cabotage regime as long as harmonisation of the road haulage market has not yet been completed. (6) The gradual completion of the single European market should lead to the elimination of restrictions on access to the domestic markets of Member States. Nevertheless, this should take into account the effectiveness of controls and the evolution of employment conditions in the profession, the harmonisation of the rules in the fields of, inter alia, enforcement and road user charges, and social and safety legislation. The Commission should closely monitor the market situation as well as the harmonisation mentioned above and propose, if appropriate, the further opening of domestic road transport markets, including cabotage. (7) Under Directive 2006/94/EC, a certain number of types of carriage are exempt from Community authorisation and from any other carriage authorisation. Within the framework of the organisation of the market provided for by this Regulation, a system of exemption from the Community licence and from any other carriage authorisation should be maintained for some of those types of carriage, because of their special nature. (8) Under Directive 2006/94/EC, the carriage of goods with vehicles of a maximum laden weight of between 3,5 tonnes and 6 tonnes was exempt from the requirement for a Community licence. Community rules in the field of road transport of goods, however, apply in general to vehicles with a maximum laden mass of more than 3,5 tonnes. Thus, the provisions of this Regulation should be aligned with the general scope of application of Community road transport rules and should only provide for an exemption for vehicles with a maximum laden mass of up to 3,5 tonnes. (9) The international carriage of goods by road should be conditional on the possession of a Community licence. Hauliers should be required to carry a certified true copy of the Community licence aboard each of their vehicles in order to facilitate effective controls by enforcement authorities, especially those outside the Member State in which the haulier is established. To this end, it is necessary to lay down more detailed specifications as regards the layout and other features of the Community licence and the certified copies. (10) Roadside checks should be carried out without direct or indirect discrimination on grounds of the nationality of the road transport operator or the country of establishment of the road transport operator or of registration of the vehicle. (11) The conditions governing the issue and withdrawal of Community licences and the types of carriage to which they apply, their periods of validity and the detailed rules for their use should be determined. (12) A driver attestation should also be established in order to allow Member States to check effectively whether drivers from third countries are lawfully employed or at the disposal of the haulier responsible for a given transport operation. (13) Hauliers who are holders of Community licences provided for in this Regulation and hauliers authorised to operate certain categories of international haulage service should be permitted to carry out national transport services within a Member State on a temporary basis in conformity with this Regulation, without having a registered office or other establishment therein. When such cabotage operations are performed, they should be subject to Community legislation such as Regulation (EC) No 561/2006 of the European Parliament and of the Council of 15 March 2006 on the harmonisation of certain social legislation relating to road transport (6) and to national law in force in specified areas in the host Member State. (14) Provisions should be adopted to allow action to be taken in the event of serious disturbance of the transport markets affected. For that purpose it is necessary to introduce a suitable decision-making procedure and for the required statistical data to be collected. (15) Without prejudice to the provisions of the Treaty on the right of establishment, cabotage operations consist of the provision of services by hauliers within a Member State in which they are not established and should not be prohibited as long as they are not carried out in a way that creates a permanent or continuous activity within that Member State. To assist the enforcement of this requirement, the frequency of cabotage operations and the period in which they can be performed should be more clearly defined. In the past, such national transport services were permitted on a temporary basis. In practice, it has been difficult to ascertain which services are permitted. Clear and easily enforceable rules are thus needed. (16) This Regulation is without prejudice to the provisions concerning the incoming or outgoing carriage of goods by road as one leg of a combined transport journey as laid down in Council Directive 92/106/EEC of 7 December 1992 on the establishment of common rules for certain types of combined transport of goods between Member States (7). National journeys by road within a host Member State which are not part of a combined transport operation as laid down in Directive 92/106/EEC fall within the definition of cabotage operations and should accordingly be subject to the requirements of this Regulation. (17) The provisions of Directive 96/71/EC of the European Parliament and of the Council of 16 December 1996 concerning the posting of workers in the framework of the provision of services (8) apply to transport undertakings performing a cabotage operation. (18) In order to perform efficient controls of cabotage operations, the enforcement authorities of the host Member States should, at least, have access to data from consignment notes and from recording equipment, in accordance with Council Regulation (EEC) No 3821/85 of 20 December 1985 on recording equipment in road transport (9). (19) Member States should grant each other mutual assistance with a view to the sound application of this Regulation. (20) Administrative formalities should be reduced as far as possible without abandoning the controls and penalties that guarantee the correct application and effective enforcement of this Regulation. To this end, the existing rules on the withdrawal of the Community licence should be clarified and strengthened. The current rules should be adapted to allow the effective sanctioning of serious infringements committed in a host Member State. Penalties should be non-discriminatory and proportionate to the seriousness of the infringements. It should be possible to lodge an appeal in respect of any penalties imposed. (21) Member States should enter in their national electronic register of road transport undertakings all serious infringements committed by hauliers which have led to the imposition of a penalty. (22) In order to facilitate and strengthen the exchange of information between national authorities, Member States should exchange the relevant information through the national contact points set up pursuant to Regulation (EC) No 1071/2009 of the European Parliament and of the Council of 21 October 2009 establishing common rules concerning the conditions to be complied with to pursue the occupation of road transport operator (10). (23) The measures necessary for the implementation of this Regulation should be adopted in accordance with Council Decision 1999/468/EC of 28 June 1999 laying down the procedures for the exercise of implementing powers conferred on the Commission (11). (24) In particular, the Commission should be empowered to adapt Annexes I, II and III to this Regulation to technical progress. Since those measures are of general scope and are designed to amend non-essential elements of this Regulation, they must be adopted in accordance with the regulatory procedure with scrutiny provided for in Article 5a of Decision 1999/468/EC. (25) Member States should take the necessary measures to implement this Regulation, in particular as regards effective, proportionate and dissuasive penalties. (26) Since the objective of this Regulation, namely to ensure a coherent framework for international road haulage throughout the Community, cannot be sufficiently achieved by the Member States and can therefore, by reason of its scale and effects, be better achieved at Community level, the Community may adopt measures, in accordance with the principle of subsidiarity as set out in Article 5 of the Treaty. In accordance with the principle of proportionality, as set out in that Article, this Regulation does not go beyond what is necessary in order to achieve that objective, HAVE ADOPTED THIS REGULATION: CHAPTER I GENERAL PROVISIONS Article 1 Scope 1. This Regulation shall apply to the international carriage of goods by road for hire or reward for journeys carried out within the territory of the Community. 2. In the event of carriage from a Member State to a third country and vice versa, this Regulation shall apply to the part of the journey on the territory of any Member State crossed in transit. It shall not apply to that part of the journey on the territory of the Member State of loading or unloading, as long as the necessary agreement between the Community and the third country concerned has not been concluded. 3. Pending the conclusion of the agreements referred to in paragraph 2, this Regulation shall not affect: (a) provisions relating to the carriage from a Member State to a third country and vice versa included in bilateral agreements concluded by Member States with those third countries; (b) provisions relating to the carriage from a Member State to a third country and vice versa included in bilateral agreements concluded between Member States which, under either bilateral authorisations or liberalisation arrangements, allow loading and unloading in a Member State by hauliers not established in that Member State. 4. This Regulation shall apply to the national carriage of goods by road undertaken on a temporary basis by a non-resident haulier as provided for in Chapter III. 5. The following types of carriage and unladen journeys made in conjunction with such carriage shall not require a Community licence and shall be exempt from any carriage authorisation: (a) carriage of mail as a universal service; (b) carriage of vehicles which have suffered damage or breakdown; (c) carriage of goods in motor vehicles the permissible laden mass of which, including that of trailers, does not exceed 3,5 tonnes; (d) carriage of goods in motor vehicles provided the following conditions are fulfilled: (i) the goods carried are the property of the undertaking or have been sold, bought, let out on hire or hired, produced, extracted, processed or repaired by the undertaking; (ii) the purpose of the journey is to carry the goods to or from the undertaking or to move them, either inside or outside the undertaking for its own requirements; (iii) motor vehicles used for such carriage are driven by personnel employed by, or put at the disposal of, the undertaking under a contractual obligation; (iv) the vehicles carrying the goods are owned by the undertaking, have been bought by it on deferred terms or have been hired provided that in the latter case they meet the conditions of Directive 2006/1/EC of the European Parliament and of the Council of 18 January 2006 on the use of vehicles hired without drivers for the carriage of goods by road (12); and (v) such carriage is no more than ancillary to the overall activities of the undertaking; (e) carriage of medicinal products, appliances, equipment and other articles required for medical care in emergency relief, in particular for natural disasters. Point (d)(iv) of the first subparagraph shall not apply to the use of a replacement vehicle during a short breakdown of the vehicle normally used. 6. The provisions of paragraph 5 shall not affect the conditions under which a Member State authorises its nationals to engage in the activities referred to in that paragraph. Article 2 Definitions For the purposes of this Regulation: 1. vehicle means a motor vehicle registered in a Member State, or a coupled combination of vehicles the motor vehicle of which at least is registered in a Member State, used exclusively for the carriage of goods; 2. international carriage means: (a) a laden journey undertaken by a vehicle the point of departure and the point of arrival of which are in two different Member States, with or without transit through one or more Member States or third countries; (b) a laden journey undertaken by a vehicle from a Member State to a third country or vice versa, with or without transit through one or more Member States or third countries; (c) a laden journey undertaken by a vehicle between third countries, with transit through the territory of one or more Member States; or (d) an unladen journey in conjunction with the carriage referred to in points (a), (b) and (c); 3. host Member State means a Member State in which a haulier operates other than the hauliers Member State of establishment; 4. non-resident haulier means a road haulage undertaking which operates in a host Member State; 5. driver means any person who drives the vehicle even for a short period, or who is carried in a vehicle as part of his duties to be available for driving if necessary; 6. cabotage operations means national carriage for hire or reward carried out on a temporary basis in a host Member State, in conformity with this Regulation; 7. serious infringement of Community road transport legislation means an infringement which may lead to the loss of good repute in accordance with Article 6(1) and (2) of Regulation (EC) No 1071/2009 and/or to the temporary or permanent withdrawal of a Community licence. CHAPTER II INTERNATIONAL CARRIAGE Article 3 General principle International carriage shall be carried out subject to possession of a Community licence and, if the driver is a national of a third country, in conjunction with a driver attestation. Article 4 Community licence 1. The Community licence shall be issued by a Member State, in accordance with this Regulation, to any haulier carrying goods by road for hire or reward who: (a) is established in that Member State in accordance with Community legislation and the national legislation of that Member State; and (b) is entitled in the Member State of establishment, in accordance with Community legislation and the national legislation of that Member State concerning admission to the occupation of road haulage operator, to carry out the international carriage of goods by road. 2. The Community licence shall be issued by the competent authorities of the Member State of establishment for renewable periods of up to 10 years. Community licences and certified copies issued before the date of application of this Regulation shall remain valid until their date of expiry. The Commission shall adapt the period of validity of the Community licence to technical progress, in particular the national electronic registers of road transport undertakings as provided for in Article 16 of Regulation (EC) No 1071/2009. Those measures, designed to amend non-essential elements of this Regulation, shall be adopted in accordance with the regulatory procedure with scrutiny referred to in Article 15(2). 3. The Member State of establishment shall issue the holder with the original of the Community licence, which shall be kept by the haulier, and the number of certified true copies corresponding to the number of vehicles at the disposal of the holder of the Community licence, whether those vehicles are wholly owned or, for example, held under a hire purchase, hire or leasing contract. 4. The Community licence and the certified true copies shall correspond to the model set out in Annex II, which also lays down the conditions governing its use. They shall contain at least two of the security features listed in Annex I. The Commission shall adapt Annexes I and II to technical progress. Those measures, designed to amend non-essential elements of this Regulation, shall be adopted in accordance with the regulatory procedure with scrutiny referred to in Article 15(2). 5. The Community licence and the certified true copies thereof shall bear the seal of the issuing authority as well as a signature and a serial number. The serial numbers of the Community licence and of the certified true copies shall be recorded in the national electronic register of road transport undertakings as part of the data relating to the haulier. 6. The Community licence shall be issued in the name of the haulier and shall be non-transferable. A certified true copy of the Community licence shall be kept in each of the hauliers vehicles and shall be presented at the request of any authorised inspecting officer. In the case of a coupled combination of vehicles, the certified true copy shall accompany the motor vehicle. It shall cover the coupled combination of vehicles even where the trailer or semi-trailer is not registered or authorised to use the roads in the name of the licence holder or where it is registered or authorised to use the roads in another State. Article 5 Driver attestation 1. A driver attestation shall be issued by a Member State, in accordance with this Regulation, to any haulier who: (a) is the holder of a Community licence; and (b) in that Member State, either lawfully employs a driver who is neither a national of a Member State nor a long-term resident within the meaning of Council Directive 2003/109/EC of 25 November 2003 concerning the status of third-country nationals who are long-term residents (13), or lawfully uses a driver who is neither a national of a Member State nor a long-term resident within the meaning of that Directive and who is put at the disposal of that haulier in accordance with the conditions of employment and of vocational training laid down in that Member State: (i) by laws, regulations or administrative provisions; and, as appropriate; (ii) by collective agreements, in accordance with the rules applicable in that Member State. 2. The driver attestation shall be issued by the competent authorities of the Member State of establishment of the haulier, at the request of the holder of the Community licence, for each driver who is neither a national of a Member State nor a long-term resident within the meaning of Directive 2003/109/EC whom that haulier lawfully employs, or for each driver who is neither a national of a Member State nor a long-term resident within the meaning of that Directive and who is put at the disposal of the haulier. Each driver attestation shall certify that the driver named therein is employed in accordance with the conditions laid down in paragraph 1. 3. The driver attestation shall correspond to the model set out in Annex III. It shall contain at least two of the security features listed in Annex I. 4. The Commission shall adapt Annex III to technical progress. Those measures, designed to amend non-essential elements of this Regulation, shall be adopted in accordance with the regulatory procedure with scrutiny referred to in Article 15(2). 5. The driver attestation shall bear the seal of the issuing authority as well as a signature and a serial number. The serial number of the driver attestation may be recorded in the national electronic register of road transport undertakings as part of the data relating to the haulier who puts it at the disposal of the driver designated therein. 6. The driver attestation shall belong to the haulier, who puts it at the disposal of the driver designated therein when that driver drives a vehicle using a Community licence issued to that haulier. A certified true copy of the driver attestation issued by the competent authorities of the hauliers Member State of establishment shall be kept at the hauliers premises. The driver attestation shall be presented at the request of any authorised inspecting officer. 7. A driver attestation shall be issued for a period to be determined by the issuing Member State, subject to a maximum validity of 5 years. Driver attestations issued before the date of application of this Regulation shall remain valid until their date of expiry. The driver attestation shall be valid only as long as the conditions under which it was issued are satisfied. Member States shall take appropriate measures to ensure that if those conditions are no longer met, the haulier returns the attestation immediately to the issuing authorities. Article 6 Verification of conditions 1. Whenever an application for a Community licence or an application for renewal of a Community licence in accordance with Article 4(2) is lodged, the competent authorities of the Member State of establishment shall verify whether the haulier satisfies or continues to satisfy the conditions laid down in Article 4(1). 2. The competent authorities of the Member State of establishment shall regularly verify, by carrying out checks each year covering at least 20 % of the valid driver attestations issued in that Member State, whether the conditions, referred to in Article 5(1), under which a driver attestation has been issued are still satisfied. Article 7 Refusal to issue and withdrawal of Community licence and driver attestation 1. If the conditions laid down in Article 4(1) or those referred to in Article 5(1) are not satisfied, the competent authorities of the Member State of establishment shall reject an application for the issue or renewal of a Community licence or the issue of a driver attestation, by means of a reasoned decision. 2. The competent authorities shall withdraw a Community licence or a driver attestation where the holder: (a) no longer satisfies the conditions laid down in Article 4(1) or those referred to in Article 5(1); or (b) has supplied incorrect information in relation to an application for a Community licence or for a driver attestation. CHAPTER III CABOTAGE Article 8 General principle 1. Any haulier for hire or reward who is a holder of a Community licence and whose driver, if he is a national of a third country, holds a driver attestation, shall be entitled, under the conditions laid down in this Chapter, to carry out cabotage operations. 2. Once the goods carried in the course of an incoming international carriage have been delivered, hauliers referred to in paragraph 1 shall be permitted to carry out, with the same vehicle, or, in the case of a coupled combination, the motor vehicle of that same vehicle, up to three cabotage operations following the international carriage from another Member State or from a third country to the host Member State. The last unloading in the course of a cabotage operation before leaving the host Member State shall take place within 7 days from the last unloading in the host Member State in the course of the incoming international carriage. Within the time limit referred to in the first subparagraph, hauliers may carry out some or all of the cabotage operations permitted under that subparagraph in any Member State under the condition that they are limited to one cabotage operation per Member State within 3 days of the unladen entry into the territory of that Member State. 3. National road haulage services carried out in the host Member State by a non-resident haulier shall only be deemed to conform with this Regulation if the haulier can produce clear evidence of the incoming international carriage and of each consecutive cabotage operation carried out. Evidence referred to in the first subparagraph shall comprise the following details for each operation: (a) the name, address and signature of the sender; (b) the name, address and signature of the haulier; (c) the name and address of the consignee as well as his signature and the date of delivery once the goods have been delivered; (d) the place and the date of taking over of the goods and the place designated for delivery; (e) the description in common use of the nature of the goods and the method of packing, and, in the case of dangerous goods, their generally recognised description, as well as the number of packages and their special marks and numbers; (f) the gross mass of the goods or their quantity otherwise expressed; (g) the number plates of the motor vehicle and trailer. 4. No additional document shall be required in order to prove that the conditions laid down in this Article have been met. 5. Any haulier entitled in the Member State of establishment, in accordance with that Member States legislation, to carry out the road haulage operations for hire or reward specified in Article 1(5)(a), (b) and (c) shall be permitted, under the conditions set out in this Chapter, to carry out, as the case may be, cabotage operations of the same kind or cabotage operations with vehicles in the same category. 6. Permission to carry out cabotage operations, within the framework of the types of carriage referred to in Article 1(5)(d) and (e), shall be unrestricted. Article 9 Rules applicable to cabotage operations 1. The performance of cabotage operations shall be subject, save as otherwise provided in Community legislation, to the laws, regulations and administrative provisions in force in the host Member State with regard to the following: (a) the conditions governing the transport contract; (b) the weights and dimensions of road vehicles; (c) the requirements relating to the carriage of certain categories of goods, in particular dangerous goods, perishable foodstuffs and live animals; (d) the driving time and rest periods; (e) the value added tax (VAT) on transport services. The weights and dimensions referred to in point (b) of the first subparagraph may, where appropriate, exceed those applicable in the hauliers Member State of establishment, but they may under no circumstances exceed the limits set by the host Member State for national traffic or the technical characteristics mentioned in the proofs referred to in Article 6(1) of Council Directive 96/53/EC of 25 July 1996 laying down for certain road vehicles circulating within the Community the maximum authorised dimensions in national and international traffic and the maximum authorised weights in international traffic (14). 2. The laws, regulations and administrative provisions referred to in paragraph 1 shall be applied to non-resident hauliers under the same conditions as those imposed on hauliers established in the host Member State, so as to prevent any discrimination on grounds of nationality or place of establishment. Article 10 Safeguard procedure 1. In the event of serious disturbance of the national transport market in a given geographical area due to, or aggravated by, cabotage, any Member State may refer the matter to the Commission with a view to the adoption of safeguard measures and shall provide the Commission with the necessary information and notify it of the measures it intends to take as regards resident hauliers. 2. For the purposes of paragraph 1: serious disturbance of the national transport market in a given geographical area means the existence on the market of problems specific to it, such that there is a serious and potentially enduring excess of supply over demand, implying a threat to the financial stability and survival of a significant number of hauliers, geographical area means an area covering all or part of the territory of a Member State or extending to all or part of the territory of other Member States. 3. The Commission shall examine the situation on the basis in particular of the relevant data and, after consulting the committee referred to in Article 15(1), shall decide within 1 month of receipt of the Member States request whether or not safeguard measures are necessary and shall adopt them if they are necessary. Such measures may involve the temporary exclusion of the area concerned from the scope of this Regulation. Measures adopted in accordance with this Article shall remain in force for a period not exceeding 6 months, renewable once within the same limits of validity. The Commission shall without delay notify the Member States and the Council of any decision taken pursuant to this paragraph. 4. If the Commission decides to adopt safeguard measures concerning one or more Member States, the competent authorities of the Member States involved shall be required to take measures of equivalent scope in respect of resident hauliers and shall inform the Commission thereof. Those measures shall be applied at the latest as from the same date as the safeguard measures adopted by the Commission. 5. Any Member State may refer to the Council a decision taken by the Commission pursuant to paragraph 3 within 30 days of its notification. The Council, acting by a qualified majority may, within 30 days of that referral, or, if there are referrals by several Member States, of the first referral, take a different decision. The limits of validity laid down in the third subparagraph of paragraph 3 shall apply to the Councils decision. The competent authorities of the Member States concerned shall be required to take measures of equivalent scope in respect of resident hauliers, and shall inform the Commission thereof. If the Council takes no decision within the period referred to in the first subparagraph, the Commission decision shall become final. 6. Where the Commission considers that the measures referred to in paragraph 3 need to be prolonged, it shall submit a proposal to the Council, which shall take a decision by qualified majority. CHAPTER IV MUTUAL ASSISTANCE AND PENALTIES Article 11 Mutual assistance Member States shall assist one another in ensuring the application and monitoring of this Regulation. They shall exchange information via the national contact points established pursuant to Article 18 of Regulation (EC) No 1071/2009. Article 12 Sanctioning of infringements by the Member State of establishment 1. In the event of a serious infringement of Community road transport legislation committed or ascertained in any Member State, the competent authorities of the Member State of establishment of the haulier who has committed such infringement shall take the appropriate action which may include a warning, if provided for by national law, to pursue the matter which may lead, inter alia, to the imposition of the following administrative penalties: (a) temporary or permanent withdrawal of some or all of the certified true copies of the Community licence; (b) temporary or permanent withdrawal of the Community licence. These penalties may be determined after the final decision on the matter has been taken and shall have regard to the seriousness of the infringement committed by the holder of the Community licence and to the total number of certified true copies of that licence that he holds in respect of international traffic. 2. In the event of a serious infringement regarding any misuse whatsoever of driver attestations, the competent authorities of the Member State of establishment of the haulier who committed such infringement shall impose appropriate penalties, such as: (a) suspending the issue of driver attestations; (b) withdrawing driver attestations; (c) making the issue of driver attestations subject to additional conditions in order to prevent misuse; (d) withdrawing, temporarily or permanently, some or all of the certified true copies of the Community licence; (e) withdrawing, temporarily or permanently, the Community licence. These penalties may be determined after the final decision on the matter has been taken and shall have regard to the seriousness of the infringement committed by the holder of the Community licence. 3. The competent authorities of the Member State of establishment shall communicate to the competent authorities of the Member State in which the infringement was ascertained, as soon as possible and at the latest within 6 weeks of their final decision on the matter, which, if any, of the penalties provided for in paragraphs 1 and 2 have been imposed. If such penalties are not imposed, the competent authorities of the Member State of establishment shall state the reasons therefor. 4. The competent authorities shall ensure that the penalties imposed on the haulier concerned are, as a whole, proportionate to the infringement or infringements which gave rise to such penalties, taking into account any penalty for the same infringement imposed in the Member State in which the infringement was ascertained. 5. The competent authorities of the hauliers Member State of establishment may also, pursuant to national law, bring proceedings against the haulier before a competent national court or tribunal. They shall inform the competent authority of the host Member State of any decisions taken to this effect. 6. Member States shall ensure that hauliers have the right to appeal against any administrative penalty imposed on them pursuant to this Article. Article 13 Sanctioning of infringements by the host Member State 1. Where the competent authorities of a Member State are aware of a serious infringement of this Regulation or of Community road transport legislation attributable to a non-resident haulier, the Member State within the territory of which the infringement is ascertained shall transmit to the competent authorities of the hauliers Member State of establishment, as soon as possible and at the latest within 6 weeks of their final decision on the matter, the following information: (a) a description of the infringement and the date and time when it was committed; (b) the category, type and seriousness of the infringement; and (c) the penalties imposed and the penalties executed. The competent authorities of the host Member State may request the competent authorities of the Member State of establishment to impose administrative penalties in accordance with Article 12. 2. Without prejudice to any criminal prosecution, the competent authorities of the host Member State shall be empowered to impose penalties on a non-resident haulier who has committed infringements of this Regulation or of national or Community road transport legislation in their territory during a cabotage operation. They shall impose such penalties on a non-discriminatory basis. These penalties may, inter alia, consist of a warning, or, in the event of a serious infringement, a temporary ban on cabotage operations on the territory of the host Member State where the infringement was committed. 3. Member States shall ensure that hauliers have the right to appeal against any administrative penalty imposed on them pursuant to this Article. Article 14 Entry in the national electronic registers Member States shall ensure that serious infringements of Community road transport legislation committed by hauliers established in their territory, which have led to the imposition of a penalty by any Member State, as well as any temporary or permanent withdrawal of the Community licence or of the certified true copy thereof, are recorded in the national electronic register of road transport undertakings. Entries in the register which concern a temporary or permanent withdrawal of a Community licence shall remain in the database for 2 years from the time of the expiry of the period of withdrawal, in the case of temporary withdrawal, or from the date of withdrawal, in the case of permanent withdrawal. CHAPTER V IMPLEMENTATION Article 15 Committee procedure 1. The Commission shall be assisted by the committee established by Article 18(1) of Regulation (EEC) No 3821/85. 2. Where reference is made to this paragraph, Article 5a(1) to (4) and Article 7 of Decision 1999/468/EC shall apply, having regard to the provisions of Article 8 thereof. Article 16 Penalties Member States shall lay down the rules on penalties applicable to infringements of the provisions of this Regulation, and shall take all the measures necessary to ensure that they are implemented. The penalties provided for must be effective, proportionate and dissuasive. Member States shall notify those provisions to the Commission by 4 December 2011, and shall notify it without delay of any subsequent amendment affecting them. Member States shall ensure that all such measures are taken without discrimination as to the nationality or place of establishment of the haulier. Article 17 Reporting 1. Every 2 years Member States shall inform the Commission of the number of hauliers possessing Community licences on 31 December of the previous year and of the number of certified true copies corresponding to the vehicles in circulation at that date. 2. Member States shall also inform the Commission of the number of driver attestations issued in the previous calendar year as well as the number of driver attestations in circulation on 31 December of that same year. 3. The Commission shall draw up a report on the state of the Community road transport market by the end of 2013. The report shall contain an analysis of the market situation, including an evaluation of the effectiveness of controls and the evolution of employment conditions in the profession, as well as an assessment as to whether harmonisation of the rules in the fields, inter alia, of enforcement and road user charges, as well as social and safety legislation, has progressed to such an extent that the further opening of domestic road transport markets, including cabotage, could be envisaged. CHAPTER VI FINAL PROVISIONS Article 18 Repeals Regulations (EEC) No 881/92 and (EEC) No 3118/93 and Directive 2006/94/EC are hereby repealed. References to the repealed Regulations and Directive shall be construed as references to this Regulation and shall be read in accordance with the correlation table set out in Annex IV. Article 19 Entry into force This Regulation shall enter into force on the 20th day following its publication in the Official Journal of the European Union. It shall apply from 4 December 2011, with the exception of Articles 8 and 9, which shall apply from 14 May 2010. This Regulation shall be binding in its entirety and directly applicable in all Member States. Done at Strasbourg, 21 October 2009. For the European Parliament The President J. BUZEK For the Council The President C. MALMSTRÃ M (1) OJ C 204, 9.8.2008, p. 31. (2) Opinion of the European Parliament of 21 May 2008 (not yet published in the Official Journal), Council Common Position of 9 January 2009 (OJ C 62 E, 17.3.2009, p. 46), Position of the European Parliament of 23 April 2009 (not yet published in the Official Journal) and Council Decision of 24 September 2009. (3) OJ L 95, 9.4.1992, p. 1. (4) OJ L 279, 12.11.1993, p. 1. (5) OJ L 374, 27.12.2006, p. 5. (6) OJ L 102, 11.4.2006, p. 1. (7) OJ L 368, 17.12.1992, p. 38. (8) OJ L 18, 21.1.1997, p. 1. (9) OJ L 370, 31.12.1985, p. 8. (10) See page 51 of this Official Journal. (11) OJ L 184, 17.7.1999, p. 23. (12) OJ L 33, 4.2.2006, p. 82. (13) OJ L 16, 23.1.2004, p. 44. (14) OJ L 235, 17.9.1996, p. 59. ANNEX I Security features of the Community licence and the driver attestation The Community licence and the driver attestation must have at least two of the following security features:  a hologram,  special fibres in the paper which become visible under UV-light,  at least one microprint line (printing visible only with a magnifying glass and not reproduced by photocopying machines),  tactile characters, symbols or patterns,  double numbering: serial number of the Community licence, of the certified copy thereof or of the driver attestation as well as, in each case, the issue number,  a security design background with fine guilloche patterns and rainbow printing. ANNEX II Community licence model EUROPEAN COMMUNITY (a) (Colour Pantone light blue, format DIN A4 cellulose paper 100 g/m2 or more) (First page of the licence) (Text in (one of) the official language(s) of the Member State issuing the licence) (b) (Second page of the licence) (Text in (one of) the official language(s) of the Member State issuing the licence) GENERAL PROVISIONS This licence is issued under Regulation (EC) No 1072/2009. It entitles the holder to engage in the international carriage of goods by road for hire or reward by any route for journeys or parts of journeys carried out within the territory of the Community and, where appropriate, subject to the conditions laid down herein:  where the point of departure and the point of arrival are situated in two different Member States, with or without transit through one or more Member States or third countries,  from a Member State to a third country or vice versa, with or without transit through one or more Member States or third countries,  between third countries with transit through the territory of one or more Member States, and unladen journeys in connection with such carriage. In the case of carriage from a Member State to a third country or vice versa, this licence is valid for that part of the journey carried out within the territory of the Community. It shall be valid in the Member State of loading or unloading only after the conclusion of the necessary agreement between the Community and the third country in question in accordance with Regulation (EC) No 1072/2009. The licence is personal to the holder and is non-transferable. It may be withdrawn by the competent authority of the Member State which issued it, notably where the holder has:  not complied with all the conditions for using the licence,  supplied incorrect information with regard to the data needed for the issue or extension of the licence. The original of the licence must be kept by the haulage undertaking. A certified copy of the licence must be kept in the vehicle (1). In the case of a coupled combination of vehicles it must accompany the motor vehicle. It covers the coupled combination of vehicles even if the trailer or semi-trailer is not registered or authorised to use the roads in the name of the licence holder or if it is registered or authorised to use the roads in another State. The licence must be presented at the request of any authorised inspecting officer. Within the territory of each Member State, the holder must comply with the laws, regulations and administrative provisions in force in that State, in particular with regard to transport and traffic. (1) Vehicle means a motor vehicle registered in a Member State, or a coupled combination of vehicles the motor vehicle of which at least is registered in a Member State, used exclusively for the carriage of goods. ANNEX III Driver attestation model EUROPEAN COMMUNITY (a) (Colour Pantone pink, format DIN A4 cellulose paper 100g/m2 or more) (First page of the attestation) (Text in (one of) the official language(s) of the Member State issuing the attestation) (b) (Second page of the attestation) (Text in (one of) the official language(s) of the Member State issuing the attestation) GENERAL PROVISIONS This attestation is issued under Regulation (EC) No 1072/2009. It certifies that the driver named therein is employed, in accordance with the laws, regulations or administrative provisions and, as appropriate, the collective agreements, in accordance with the rules applicable in the Member State mentioned on the attestation, on the conditions of employment and of vocational training of drivers applicable in that Member State to carry out road operations in that State. The driver attestation shall belong to the haulier, who puts it at the disposal of the driver designated therein when that driver drives a vehicle (1) engaged in carriage using a Community licence issued to that haulier. The driver attestation is not transferable. The driver attestation shall be valid only as long as the conditions under which it was issued are still satisfied and must be returned immediately by the haulier to the issuing authorities if these conditions are no longer met. It may be withdrawn by the competent authority of the Member State which issued it, in particular where the holder has:  not complied with all the conditions for using the attestation,  supplied incorrect information with regard to the data needed for the issue or extension of the attestation. A certified true copy of the attestation must be kept by the haulage undertaking. An original attestation must be kept in the vehicle and must be presented by the driver at the request of any authorised inspecting officer. (1) Vehicle means a motor vehicle registered in a Member State, or a coupled combination of vehicles the motor vehicle of which at least is registered in a Member State, used exclusively for the carriage of goods. ANNEX IV Correlation Table Regulation (EEC) No 881/92 Regulation (EEC) No 3118/93 Directive 2006/94/EC This Regulation Article 1(1) Article 1(1) Article 1(2) Article 1(2) Article 1(3) Article 1(3) Annex II Article 1(1) and (2), Annex I; Article 2 Article 1(5) Article 2 Article 1(6) Article 2 Article 2 Article 3(1) Article 3 Article 3(2) Article 4(1) Article 3(3) Article 5(1) Article 4 Article 5(1) Article 4(2) Article 5(2) Article 4(3) Article 5(3) Article 4(4) Article 4(5) Article 5(4), Annex I Article 4(6) Article 5(5) Article 4(2) Article 6(1) Article 5(2) Article 6(2) Article 5(2) Article 6(3) Article 5(3) Article 6(4) Article 5(6) Article 6(5) Article 5(7) Article 7 Article 6 Article 8(1) Article 7(1) Article 8(2) Article 7(2) Article 8(3) Article 12(1) Article 8(4) Article 12(2) Article 9(1) and (2) Article 12(6) Article 1(1) Article 8(1) Article 1(2) Article 8(5) Article 1(3) and (4) Article 8(6) Article 2 Article 3 Article 4 Article 5 Article 6(1) Article 9(1) Article 6(2) Article 6(3) Article 9(2) Article 6(4) Article 7 Article 10 Article 10 Article 17(1) Article 11(1) Article 8(1) Article 11 Article 11(2) Article 13(1) Article 11(3) Article 12(4) Article 11a Article 8(2) and (3) Article 13(2) Article 8(4), first and third subparagraphs Article 8(4), second subparagraph Article 12(4) Article 8(4), fourth and fifth subparagraphs Article 12(5) Article 9 Article 13(3) Article 12 Article 18 Article 13 Article 14 Article 10 Article 11 Article 15 Article 12 Article 4 Article 19 Article 3 Article 5 Annex II, III Annex I Annex II Annex III Annex III Annex I Annex II Annex III Annex IV
============================== "END OF DOC" ==============================
        

        doc 2 :

        'celex': 32015L0413
        'status': In Force
        'act_type': Directive
        'treaty': TFEU (2008)

        full_doc :

        13.3.2015 EN Official Journal of the European Union L 68/9 DIRECTIVE (EU) 2015/413 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL of 11 March 2015 facilitating cross-border exchange of information on road-safety-related traffic offences (Text with EEA relevance) THE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE EUROPEAN UNION, Having regard to the Treaty on the Functioning of the European Union, and in particular Article 91(1)(c) thereof, Having regard to the proposal from the European Commission, After transmission of the draft legislative act to the national parliaments, Having regard to the opinion of the European Economic and Social Committee (1), After consulting the Committee of the Regions, Acting in accordance with the ordinary legislative procedure (2), Whereas: (1) Improving road safety is a prime objective of the Union's transport policy. The Union is pursuing a policy to improve road safety with the objective of reducing fatalities, injuries and material damage. An important element of that policy is the consistent enforcement of sanctions for road traffic offences committed in the Union which considerably jeopardise road safety. (2) However, due to a lack of appropriate procedures and notwithstanding existing possibilities under Council Decision 2008/615/JHA (3) and Council Decision 2008/616/JHA (4) (the PrÃ ¼m Decisions), sanctions in the form of financial penalties for certain road traffic offences are often not enforced if those offences are committed with a vehicle which is registered in a Member State other than the Member State where the offence took place. This Directive aims to ensure that even in such cases, the effectiveness of the investigation of road-safety-related traffic offences should be ensured. (3) In its communication of 20 July 2010 entitled Towards a European road safety area: policy orientations on road safety 2011-2020, the Commission emphasised that enforcement of road traffic rules remains a key factor in creating the conditions for a considerable reduction in the number of deaths and injuries. In its conclusions of 2 December 2010 on road safety, the Council called for consideration of the need for further strengthening of enforcement of road traffic rules by Member States and, where appropriate, at Union level. It invited the Commission to examine the possibilities of harmonising traffic rules at Union level where appropriate and adopting further measures on facilitating cross-border enforcement with regard to road traffic offences, in particular those related to serious traffic accidents. (4) On 19 March 2008, the Commission adopted a proposal for a Directive of the European Parliament and of the Council facilitating cross-border enforcement in the field of road safety on the basis of Article 71(1)(c) of the Treaty establishing the European Community (now Article 91 of Treaty on the Functioning of the European Union (TFEU)). Directive 2011/82/EU of the European Parliament and of the Council (5) was, however, adopted on the basis of Article 87(2) TFEU. The judgment of the Court of Justice of 6 May 2014 in Case C-43/12 (6) annulled Directive 2011/82/EU on the grounds that it could not validly be adopted on the basis of Article 87(2) TFEU. The judgment maintained the effects of Directive 2011/82/EU until the entry into force within a reasonable period of time  which is not to exceed 12 months as from the date of delivery of the judgment  of a new directive based on Article 91(1)(c) TFEU. Therefore a new Directive should be adopted on the basis of that Article. (5) Greater convergence of control measures between Member States should be encouraged and the Commission should examine in this respect the need for developing common standards for automatic checking equipment for road safety controls. (6) The awareness of Union citizens should be raised as regards the road safety traffic rules in force in different Member States and as regards the implementation of this Directive, in particular through appropriate measures guaranteeing the provision of sufficient information on the consequences of not respecting the road safety traffic rules when travelling in a Member State other than the Member State of registration. (7) In order to improve road safety throughout the Union and to ensure equal treatment of drivers, namely resident and non-resident offenders, enforcement should be facilitated irrespective of the Member State of registration of the vehicle. To this end, a system of cross-border exchange of information should be used for certain identified road-safety-related traffic offences, regardless of their administrative or criminal nature under the law of the Member State concerned, granting the Member State of the offence access to vehicle registration data (VRD) of the Member State of registration. (8) A more efficient cross-border exchange of VRD, which should facilitate the identification of persons suspected of committing a road-safety-related traffic offence, might increase the deterrent effect and induce more cautious behaviour by the driver of a vehicle that is registered in a Member State other than the Member State of the offence, thereby preventing casualties due to road traffic accidents. (9) The road-safety-related traffic offences covered by this Directive are not subject to homogeneous treatment in the Member States. Some Member States qualify such offences under national law as administrative offences while others qualify them as criminal offences. This Directive should apply regardless of how those offences are qualified under national law. (10) Member States should grant each other the right of access to their VRD in order to improve the exchange of information and to speed up the procedures in force. To this end, the provisions concerning the technical specifications and the availability of automated data exchange set out in the PrÃ ¼m Decisions should, as far as possible, be included in this Directive. (11) Decision 2008/616/JHA specifies the security features for existing software applications and the related technical requirements for the exchange of vehicle registration data. Without prejudice to the general applicability of that Decision, those security features and technical requirements should, for reasons of regulatory and practical efficiency, be used for the purposes of this Directive. (12) Existing software applications should be the basis for the data exchange under this Directive and should, at the same time, also facilitate the reporting by Member States to the Commission. Such applications should provide for the expeditious, secure and confidential exchange of specific VRD between Member States. Advantage should be taken of the European Vehicle and Driving Licence Information System (Eucaris) software application, which is mandatory for Member States under the PrÃ ¼m Decisions as regards VRD. The Commission should assess and report on the functioning of the software applications used for the purposes of this Directive. (13) The scope of those software applications should be limited to the processes used in the exchange of information between the national contact points in the Member States. Procedures and automated processes in which the information is to be used are outside the scope of such applications. (14) The Information Management Strategy for EU internal security aims to find the simplest and most easily traceable and cost-effective solutions for data exchange. (15) Member States should be able to contact the owner, the holder of the vehicle or the otherwise identified person suspected of committing the road-safety-related traffic offence in order to keep the person concerned informed of the applicable procedures and the legal consequences under the law of the Member State of the offence. In doing so, Member States should consider sending the information concerning road-safety-related traffic offences in the language of the registration documents, or in the language most likely to be understood by the person concerned, to ensure that that person has a clear understanding of the information which is being shared with the person concerned. Member States should apply the appropriate procedures to ensure that only the person concerned is informed and not a third party. To that effect, Member States should use detailed arrangements similar to those adopted for following up such offences including means such as, where appropriate, registered delivery. This will allow that person to respond to the information letter in an appropriate way, in particular by asking for more information, by settling the fine or by exercising his/her rights of defence, especially in the case of mistaken identity. Further proceedings are covered by applicable legal instruments, including instruments on mutual assistance and on mutual recognition, for example Council Framework Decision 2005/214/JHA (7). (16) Member States should provide equivalent translation with respect to the information letter sent by the Member State of the offence, as provided for in Directive 2010/64/EU of the European Parliament and of the Council (8). (17) With a view to pursuing a road safety policy that aims to provide a high level of protection for all road users in the Union, and taking into account the widely differing circumstances pertaining within the Union, Member States should act, without prejudice to more restrictive policies and laws, in order to ensure greater convergence of road traffic rules and of their enforcement between Member States. In the framework of its report to the European Parliament and to the Council on the application of this Directive, the Commission should examine the need to develop common standards in order to establish comparable methods, practices and minimum standards at Union level taking into account international cooperation and existing agreements in the field of road safety, in particular the Vienna Convention on Road Traffic of 8 November 1968. (18) In its report to the European Parliament and to the Council on the application of this Directive by the Member States, the Commission should examine the need for common criteria for follow-up procedures by Member States in the event of non-payment of a financial penalty, in accordance with Member States' laws and procedures. In that report, the Commission should address issues such as the procedures between the competent authorities of the Member States for the transmission of the final decision to impose a sanction and/or financial penalty as well as the recognition and enforcement of the final decision. (19) In preparing the review of this Directive, the Commission should consult the relevant stakeholders, such as road safety and law enforcement authorities or competent bodies, victims' associations and other non-governmental organisations active in the field of road safety. (20) Closer cooperation between law enforcement authorities should go hand in hand with respect for fundamental rights, in particular the right to respect for privacy and to the protection of personal data, guaranteed by special data protection arrangements. Those arrangements should take particular account of the specific nature of cross-border online access to databases. It is necessary that the software applications to be set up enable the exchange of information to be carried out in secure conditions and ensure the confidentiality of the data transmitted. The data collected under this Directive should not be used for purposes other than those of this Directive. Member States should comply with the obligations on the conditions of use and of temporary storage of the data. (21) The processing of personal data provided by this Directive is appropriate for attaining the legitimate aims pursued by this Directive in the field of road safety, namely to ensure a high level of protection for all road users in the Union by facilitating the cross-border exchange of information on road-safety-related traffic offences and, thereby, the enforcement of sanctions, and does not exceed what is appropriate and necessary in order to achieve those objectives. (22) Data relating to the identification of an offender are personal data. Directive 95/46/EC of the European Parliament and of the Council (9) should apply to the processing activities carried out in application of this Directive. Without prejudice to the procedural requirements for appeal and the redress mechanisms of the Member State concerned, the data subject should accordingly be informed, when notified of the offence, of the right to access and the right to rectification and deletion of personal data, as well as of the maximum legal storage period of the data. In this context, the data subject should also have the right to obtain the correction of any inaccurate personal data or the immediate deletion of any data recorded unlawfully. (23) In the framework of the PrÃ ¼m Decisions, the processing of VRD containing personal data is subject to the specific provisions on data protection set out in Decision 2008/615/JHA. In that respect, Member States have the possibility to apply those specific provisions to personal data which are also processed for the purposes of this Directive provided that they ensure that the processing of data related to all of the offences covered by this Directive complies with the national provisions implementing Directive 95/46/EC. (24) It should be possible for third countries to participate in the exchange of VRD provided that they have concluded an agreement with the Union to this effect. Such an agreement would have to include necessary provisions on data protection. (25) This Directive upholds the fundamental rights and principles recognised by the Charter of Fundamental Rights of the European Union, including the respect for private and family life, the protection of personal data, the right to a fair trial, the presumption of innocence and the right of defence. (26) In order to achieve the objective of the exchange of information between Member States through interoperable means, the power to adopt acts in accordance with Article 290 TFEU should be delegated to the Commission in respect of the taking into account of relevant changes to PrÃ ¼m Decisions or where required by legal acts of the Union directly relevant for the updating of Annex I. It is of particular importance that the Commission follow its usual practice and carry out appropriate consultations during its preparatory work, including at expert level. The Commission, when preparing and drawing up delegated acts, should ensure a simultaneous, timely and appropriate transmission of relevant documents to the European Parliament and to the Council. (27) The Commission should analyse the application of this Directive with a view to identifying further effective and efficient measures to improve road safety. Without prejudice to obligations to transpose this Directive, Denmark, Ireland and the United Kingdom should also cooperate with the Commission in this work, where appropriate, to ensure timely and complete reporting on this matter. (28) Since the objective of this Directive, namely to ensure a high level of protection for all road users in the Union by facilitating the cross-border exchange of information on road-safety-related traffic offences, where they are committed with a vehicle registered in a Member State other than the Member State where the offence took place, cannot be sufficiently achieved by the Member States, but can rather, by reason of the scale and effects of the action, be better achieved at Union level, the Union may adopt measures, in accordance with the principle of subsidiarity, as set out in Article 5 of the Treaty on European Union. In accordance with the principle of proportionality, as set out in that Article, this Directive does not go beyond what is necessary in order to achieve that objective. (29) Given that Denmark, Ireland and the United Kingdom were not subject to Directive 2011/82/EU and therefore have not transposed it, it is appropriate to allow those Member States sufficient additional time to do so. (30) The European Data Protection Supervisor was consulted in accordance with Article 28(2) of Regulation (EC) No 45/2001 of the European Parliament and of the Council (10) and delivered an opinion on 3 October 2014, HAVE ADOPTED THIS DIRECTIVE: Article 1 Objective This Directive aims to ensure a high level of protection for all road users in the Union by facilitating the cross-border exchange of information on road-safety-related traffic offences, and thereby facilitating the enforcement of sanctions, where those offences are committed with a vehicle registered in a Member State other than the Member State in which the offence took place. Article 2 Scope This Directive applies to the following road-safety-related traffic offences: (a) speeding; (b) failing to use a seat-belt; (c) failing to stop at a red traffic light; (d) drink-driving; (e) driving while under the influence of drugs; (f) failing to wear a safety helmet; (g) the use of a forbidden lane; (h) illegally using a mobile telephone or any other communication devices while driving. Article 3 Definitions For the purposes of this Directive, the following definitions apply: (a) vehicle means any power-driven vehicle, including motorcycles, which is normally used for carrying persons or goods by road; (b) Member State of the offence means the Member State where the offence was committed; (c) Member State of registration means the Member State where the vehicle with which the offence was committed is registered; (d) speeding means exceeding speed limits in force in the Member State of offence for the road or type of vehicle concerned; (e) failing to use a seat-belt means not complying with the requirement to wear a seat-belt or to use a child restraint in accordance with Council Directive 91/671/EEC (11) and the law of the Member State of the offence; (f) failing to stop at a red traffic light means driving through a red traffic light or any other relevant stop signal, as defined in the law of the Member State of the offence; (g) drink-driving means driving while impaired by alcohol, as defined in the law of the Member State of the offence; (h) driving under the influence of drugs means driving while impaired by drugs or other substances having a similar effect, as defined in the law of the Member State of the offence; (i) failing to wear a safety helmet means not wearing a safety helmet, as defined in the law of the Member State of the offence; (j) use of a forbidden lane means illegally using part of a road section, such as an emergency lane, public transport lane or temporary closed lane for reasons of congestion or road works, as defined in the law of the Member State of the offence; (k) illegally using a mobile telephone or any other communication devices while driving means illegally using a mobile telephone or any other communication devices while driving, as defined in the law of the Member State of the offence; (l) national contact point means a designated competent authority for the exchange of VRD; (m) automated search means an online access procedure for consulting the databases of one, more than one, or all of the Member States or of the participating countries; (n) holder of the vehicle means the person in whose name the vehicle is registered, as defined in the law of the Member State of registration. Article 4 Procedure for the exchange of information between Member States 1. For the investigation of the road-safety-related traffic offences referred to in Article 2, the Member State shall grant other Member States' national contact points, referred to in paragraph 2 of this Article, access to the following national VRD, with the power to conduct automated searches thereon: (a) data relating to vehicles; and (b) data relating to owners or holders of the vehicle. The data elements referred to in points (a) and (b) which are necessary to conduct a search shall be in compliance with Annex I. 2. For the purposes of the exchange of data referred to in paragraph 1, each Member State shall designate a national contact point. The powers of the national contact points shall be governed by the applicable law of the Member State concerned. 3. When conducting a search in the form of an outgoing request, the national contact point of the Member State of the offence shall use a full registration number. Those searches shall be conducted in compliance with the procedures as described in Chapter 3 of the Annex to Decision 2008/616/JHA, except for point 1 of Chapter 3 of the Annex to Decision 2008/616/JHA, for which Annex I to this Directive shall apply. The Member State of the offence shall, under this Directive, use the data obtained in order to establish who is personally liable for road-safety-related traffic offences listed in Article 2 of this Directive. 4. Member States shall take all necessary measures to ensure that the exchange of information is carried out by interoperable electronic means without exchange of data involving other databases which are not used for the purposes of this Directive. Member States shall ensure that such exchange of information is conducted in a cost-efficient and secure manner. Member States shall ensure the security and protection of the data transmitted, as far as possible using existing software applications such as the one referred to in Article 15 of Decision 2008/616/JHA and amended versions of those software applications, in compliance with Annex I to this Directive and with points 2 and 3 of Chapter 3 of the Annex to Decision 2008/616/JHA. The amended versions of the software applications shall provide for both online real-time exchange mode and batch exchange mode, the latter allowing for the exchange of multiple requests or responses within one message. 5. Each Member State shall bear its own costs arising from the administration, use and maintenance of the software applications referred to in paragraph 4. Article 5 Information letter on the road-safety-related traffic offences 1. The Member State of the offence shall decide whether or not to initiate follow-up proceedings in relation to the road-safety-related traffic offences listed in Article 2. Where the Member State of the offence decides to initiate such proceedings, that Member State shall, in accordance with its national law, inform the owner, the holder of the vehicle or the otherwise identified person suspected of committing the road-safety-related traffic offence. This information shall, as applicable under national law, include the legal consequences thereof within the territory of the Member State of the offence under the law of that Member State. 2. When sending the information letter to the owner, the holder of the vehicle or to the otherwise identified person suspected of committing the road-safety-related traffic offence, the Member State of the offence shall, in accordance with its law, include any relevant information, notably the nature of this road-safety-related traffic offence, the place, date and time of the offence, the title of the texts of the national law infringed and the sanction and, where appropriate, data concerning the device used for detecting the offence. For that purpose, the Member State of the offence may use the template set out in Annex II. 3. Where the Member State of the offence decides to initiate follow-up proceedings in relation to the road-safety-related traffic offences listed in Article 2, the Member State of the offence, for the purpose of ensuring the respect of fundamental rights, sends the information letter in the language of the registration document of the vehicle, if available, or in one of the official languages of the Member State of registration. Article 6 Reporting by Member States to the Commission Each Member State shall send a comprehensive report to the Commission by 6 May 2016 and every two years thereafter. The comprehensive report shall indicate the number of automated searches conducted by the Member State of the offence addressed to the national contact point of the Member State of registration, following offences committed on its territory, together with the type of offences for which requests were addressed and the number of failed requests. The comprehensive report shall also include a description of the situation at national level in relation to the follow-up given to the road-safety-related traffic offences, based on the proportion of such offences which have been followed up by information letters. Article 7 Data protection 1. The provisions on data protection set out in Directive 95/46/EC shall apply to personal data processed under this Directive. 2. In particular, each Member State shall ensure that personal data processed under this Directive are, within an appropriate time period, rectified if inaccurate, or erased or blocked when they are no longer required, in accordance with Articles 6 and 12 of Directive 95/46/EC, and that a time limit for the storage of data is established in accordance with Article 6 of that Directive. Member States shall ensure that all personal data processed under this Directive are only used for the objective set out in Article 1 of this Directive, and that the data subjects have the same rights to information, to access, to rectification, erasure and blocking, to compensation and to judicial redress as those adopted under national law in implementation of the relevant provisions of Directive 95/46/EC. 3. Any person concerned shall have the right to obtain information on which personal data recorded in the Member State of registration were transmitted to the Member State of the offence, including the date of the request and the competent authority of the Member State of the offence. Article 8 Information for road users in the Union 1. The Commission shall make available on its website a summary in all official languages of the institutions of the Union of the rules in force in Member States in the field covered by this Directive. Member States shall provide information on these rules to the Commission. 2. Member States shall provide road users with the necessary information about the rules applicable in their territory and the measures implementing this Directive in association with, among other organisations, road safety bodies, non-governmental organisations active in the field of road safety and automobile clubs. Article 9 Delegated acts The Commission shall be empowered to adopt delegated acts, in accordance with Article 10, updating Annex I in the light of technical progress to take into account relevant changes to PrÃ ¼m Decisions or where this is required by legal acts of the Union directly relevant to the updating of Annex I. Article 10 Exercise of the delegation 1. The power to adopt delegated acts is conferred on the Commission subject to the conditions laid down in this Article. 2. The power to adopt delegated acts referred to in Article 9 shall be conferred on the Commission for a period of five years from 13 March 2015. The Commission shall draw up a report in respect of the delegation of power not later than nine months before the end of the five-year period. The delegation of power shall be tacitly extended for periods of an identical duration, unless the European Parliament or the Council opposes such extension not later than three months before the end of each period. 3. The delegation of power referred to in Article 9 may be revoked at any time by the European Parliament or by the Council. A decision to revoke shall put an end to the delegation of the power specified in that decision. It shall take effect on the day following the publication of the decision in the Official Journal of the European Union or at a later date specified therein. It shall not affect the validity of any delegated acts already in force. 4. It is of particular importance that the Commission follow its usual practice and carry out consultations with experts, including Member States' experts, before adopting those delegated acts. As soon as it adopts a delegated act, the Commission shall notify it simultaneously to the European Parliament and to the Council. 5. A delegated act adopted pursuant to Article 9 shall enter into force only if no objection has been expressed either by the European Parliament or the Council within a period of two months of notification of that act to the European Parliament and the Council or if, before the expiry of that period, the European Parliament and the Council have both informed the Commission that they will not object. That period shall be extended by two months at the initiative of the European Parliament or of the Council. Article 11 Revision of the Directive Without prejudice to the provisions laid down in the second subparagraph of Article 12(1), the Commission shall, by 7 November 2016, submit a report to the European Parliament and to the Council on the application of this Directive by the Member States. In its report, the Commission shall focus in particular on, and shall, as appropriate, make proposals to cover, the following aspects:  an assessment of whether other road-safety-related traffic offences should be added to the scope of this Directive,  an assessment of the effectiveness of this Directive on the reduction in the number of fatalities on Union roads,  an assessment of the need for developing common standards for automatic checking equipment and for procedures. In this context, the Commission is invited to develop at Union level road safety guidelines within the framework of the common transport policy in order to ensure greater convergence of the enforcement of road traffic rules by Member States through comparable methods and practices. These guidelines may cover at least the offences listed in points (a) to (d) of Article 2,  an assessment of the need to strengthen the enforcement of sanctions with regard to road-safety-related traffic offences and to propose common criteria concerning the follow-up procedures in the case of non-payment of a financial penalty, within the framework of all relevant Union policies, including the common transport policy,  the possibilities for harmonising traffic rules where appropriate,  an assessment of the software applications as referred to in Article 4(4), with a view to ensuring proper implementation of this Directive as well as guaranteeing an effective, expeditious, secure and confidential exchange of specific VRD. Article 12 Transposition 1. Member States shall bring into force the laws, regulations and administrative provisions necessary to comply with this Directive by 6 May 2015. They shall forthwith communicate to the Commission the text of those provisions. When Member States adopt those provisions, they shall contain a reference to this Directive or be accompanied by such a reference on the occasion of their official publication. Member States shall determine how such reference is to be made. By way of derogation from the first subparagraph, the Kingdom of Denmark, Ireland and the United Kingdom of Great Britain and Northern Ireland may postpone the deadline referred to in the first subparagraph until 6 May 2017. 2. Member States shall communicate to the Commission the text of the main provisions of national law which they adopt in the field covered by this Directive. Article 13 Entry into force This Directive shall enter into force on the fourth day following that of its publication in the Official Journal of the European Union. Article 14 Addressees This Directive is addressed to the Member States. Done at Strasbourg, 11 March 2015. For the European Parliament The President M. SCHULZ For the Council The President Z. KALNIÃ A-LUKAÃ EVICA (1) OJ C 12, 15.1.2015, p. 115. (2) Position of the European Parliament of 11 February 2015 (not yet published in the Official Journal) and Decision of the Council of 2 March 2015. (3) Council Decision 2008/615/JHA of 23 June 2008 on the stepping up of cross-border cooperation, particularly in combating terrorism and cross-border crime (OJ L 210, 6.8.2008, p. 1). (4) Council Decision 2008/616/JHA of 23 June 2008 on the implementation of Decision 2008/615/JHA on the stepping up of cross-border cooperation, particularly in combating terrorism and cross-border crime (OJ L 210, 6.8.2008, p. 12). (5) Directive 2011/82/EU of the European Parliament and of the Council of 25 October 2011 facilitating the cross-border exchange of information on road safety related traffic offences (OJ L 288, 5.11.2011, p. 1). (6) Judgment in Commission v Parliament and Council, C-43/12, EU:C:2014:298. (7) Council Framework Decision 2005/214/JHA of 24 February 2005 on the application of the principle of mutual recognition to financial penalties (OJ L 76, 22.3.2005, p. 16). (8) Directive 2010/64/EU of the European Parliament and of the Council of 20 October 2010 on the right to interpretation and translation in criminal proceedings (OJ L 280, 26.10.2010, p. 1). (9) Directive 95/46/EC of the European Parliament and of the Council of 24 October 1995 on the protection of individuals with regard to the processing of personal data and on the free movement of such data (OJ L 281, 23.11.1995, p. 31). (10) Regulation (EC) No 45/2001 of the European Parliament and of the Council of 18 December 2000 on the protection of individuals with regard to the processing of personal data by the Community institutions and bodies and on the free movement of such data (OJ L 8, 12.1.2001, p. 1). (11) Council Directive 91/671/EEC of 16 December 1991 relating to the compulsory use of safety belts and child-restraint systems in vehicles (OJ L 373, 31.12.1991, p. 26). ANNEX I Data elements necessary to conduct the search referred to in Article 4(1) Item M/O (1) Remarks Data relating to the vehicle M Member State of registration M Registration number M (A (2)) Data relating to the offence M Member State of the offence M Reference date of the offence M Reference time of the offence M Purpose of the search M Code indicating the type of offence as listed in Article 2 1. = Speeding 2. = Drink-driving 3. = Failing to use a seat belt 4. = Failing to stop at a red traffic light 5. = Use of a forbidden lane 10. = Driving under the influence of drugs 11. = Failing to wear a safety helmet 12. = Illegally using a mobile phone or any other communication devices while driving Data elements provided as a result of the search conducted pursuant to Article 4(1) Part I. Data relating to vehicles Item M/O (3) Remarks Registration number M Chassis number/VIN M Member State of registration M Make M (D.1 (4)) e.g. Ford, Opel, Renault Commercial type of the vehicle M (D.3) e.g. Focus, Astra, Megane EU Category Code M (J) e.g. mopeds, motorbikes, cars Part II. Data relating to owners or holders of the vehicles Item M/O (5) Remarks Data relating to holders of the vehicle (C.1 (6)) The data refer to the holder of the specific registration certificate. Registration holders' (company) name M (C.1.1) Separate fields shall be used for surname, infixes, titles, etc., and the name in printable format shall be communicated. First name M (C.1.2) Separate fields for first name(s) and initials shall be used, and the name in printable format shall be communicated. Address M (C.1.3) Separate fields shall be used for street, house number and annex, post code, place of residence, country of residence, etc., and the address in printable format shall be communicated. Gender O Male, female Date of birth M Legal entity M Individual, association, company, firm, etc. Place of Birth O ID Number O An identifier that uniquely identifies the person or the company. Data relating to owners of the vehicle (C.2) The data refer to the owner of the vehicle. Owners' (company) name M (C.2.1) First name M (C.2.2) Address M (C.2.3) Gender O Male, female Date of birth M Legal entity M Individual, association, company, firm, etc. Place of Birth O ID Number O An identifier that uniquely identifies the person or the company. In case of scrap vehicles, stolen vehicles or number plates, or outdated vehicle registration no owner/holder information shall be provided. Instead, the message Information not disclosed shall be returned. (1) M = mandatory when available in national register, O = optional. (2) Harmonised code, see Council Directive 1999/37/EC of 29 April 1999 on the registration documents for vehicles (OJ L 138, 1.6.1999, p. 57). (3) M = mandatory when available in national register, O = optional. (4) Harmonised code, see Directive 1999/37/EC. (5) M = mandatory when available in national register, O = optional. (6) Harmonised code, see Directive 1999/37/EC. ANNEX II Text of image TEMPLATE FOR THE INFORMATION LETTER referred to in Article 5 [Cover page] [Name, address and telephone number of sender] [Name and address of addressee] INFORMATION LETTER regarding a road-safety-related traffic offence committed in [name of the Member State of the offence] Text of image Page 2 On a road-safety-related traffic offence committed with the vehicle with registration [date] number make model was detected by [name of the responsible body] [Option 1] (1) You are registered as the holder of the registration certificate of the abovementioned vehicle. [Option 2] (1) The holder of the registration certificate of the abovementioned vehicle indicated that you were driving that vehicle when the road-safety-related traffic offence was committed. The relevant details of the offence are described on page 3 below. The amount of the financial penalty due for this offence is EUR/national currency. Deadline for the payment is You are advised to complete the attached reply form (page 4) and send it to the address shown, if you do not pay this financial penalty. This letter shall be processed in accordance with the national law of [name of the Member State of the offence]. Text of image Page 3 Relevant details concerning the offence (a) Data concerning the vehicle with which the offence was committed: Registration number: Member State of registration: Make and model: (b) Data concerning the offence: Place, date and time where the offence was committed: Nature and legal classification of the offence: speeding, failing to use a seatbelt, failing to stop at a red traffic light, drink-driving, driving under the influence of drugs, failing to wear a safety helmet, use of a forbidden lane, illegally using a mobile telephone or any other communication devices while driving (1) Detailed description of the offence: Reference to the relevant legal provision(s): Description of or reference to the evidence for the offence: Text of image (c) Data concerning the device that was used for detecting the offence (2): Type of device for detection of speeding, failing to use a seatbelt, failing to stop at a red traffic light, drink-driving, driving under the influence of drugs, failing to wear a safety helmet, use of a forbidden lane, illegally using a mobile telephone or any other communication devices while driving (1): Specification of the device: Identification number of the device: Expiry date for the last gauging: (d) The result of the application of the device: [example for speeding; other offences to be added:] The maximum speed: The measured speed: The measured speed corrected for margin of error: (1) Delete if not applicable. (2) Not applicable if no device has been used. Text of image Page 4 Reply form (please complete using block capitals) A. Identity of the driver:  Full name:  Place and date of birth:  Number of driving licence: delivered (date): and at (place):  Address: B. List of questions: 1. Is the vehicle, make , registration number , registered in your name? yes/no (1) If not, the holder of the registration certificate is: (name, first name, address) 2. Do you acknowledge that you committed the offence? yes/no (1) 3. If you do not acknowledge this, please explain why: Please send the completed form within 60 days from the date of this information letter to the following authority: at the following address: INFORMATION This case will be examined by the competent authority of [name of the Member State of the offence] If this case is not pursued, you will be informed within 60 days after receipt of the reply form. (1) Delete if not applicable. Text of image If this case is pursued, the following procedure applies: [to be filled in by the Member State of the offence  what the further procedure will be, including details of the possibility and procedure of appeal against the decision to pursue the case. These details shall in any event include: name and address of the authority in charge of pursuing the case; deadline for payment; name and address of the body of appeal concerned; deadline for appeal]. This letter as such does not lead to legal consequences.
============================== "END OF DOC" ==============================
        

        doc 3 :

        'celex': 32006L0126
        'status': In Force
        'act_type': Directive
        'treaty': TEC (1992)

        full_doc :

        30.12.2006 EN Official Journal of the European Union L 403/18 DIRECTIVE 2006/126/EC OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL of 20 December 2006 on driving licences (Recast) (Text with EEA relevance) THE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE EUROPEAN UNION, Having regard to the Treaty establishing the European Community, and in particular Article 71 thereof, Having regard to the proposal from the Commission, Having regard to the opinion of the European Economic and Social Committee (1), After consulting the Committee of the Regions, Acting in accordance with the procedure laid down in Article 251 of the Treaty (2), Whereas: (1) Council Directive 91/439/EEC of 29 July 1991 on driving licences (3) has been significantly amended on several occasions. Now that new amendments are being made to the said Directive, it is desirable, in order to clarify matters, that the provisions in question should be recast. (2) The rules on driving licences are essential elements of the common transport policy, contribute to improving road safety, and facilitate the free movement of persons taking up residence in a Member State other than the one issuing the licence. Given the importance of individual means of transport, possession of a driving licence duly recognised by a host Member State promotes free movement and freedom of establishment of persons. Despite the progress achieved with harmonising the rules on driving licences, significant differences have persisted between Member States in the rules on periodicity of licences renewal and on subcategories of vehicles, which needed to be harmonised more fully, in order to contribute to the implementation of Community policies. (3) The possibility of laying down national provisions with regard to the period of validity provided for in Directive 91/439/EEC leads to the co-existence of different rules in different Member States and over 110 different models of driving licences valid in the Member States. This creates problems of transparency for citizens, police forces and the administrations responsible for the administration of driving licences and leads to the falsification of documents which sometimes date back several decades. (4) In order to prevent the single European driving licence model from becoming an additional model to the 110 already in circulation, Member States should take all necessary measures to issue this single model to all licence holders. (5) This Directive should not prejudice existing entitlements to drive granted or acquired before its date of application. (6) Driving licences are mutually recognised. Member States should be able to apply the period of validity prescribed by this Directive to a licence without a limited administrative validity issued by another Member State and whose holder has resided on their territory for more than two years. (7) The introduction of a period of administrative validity for new driving licences should make it possible to apply at the time of periodic renewal the most recent counter-falsification measures and the medical examinations or other measures provided for by the Member States. (8) On road safety grounds, the minimum requirements for the issue of a driving licence should be laid down. Standards for driving tests and licensing need to be harmonised. To this end the knowledge, skills and behaviour connected with driving motor vehicles should be defined, the driving test should be based on these concepts and the minimum standards of physical and mental fitness for driving such vehicles should be redefined. (9) Proof of fulfilment of compliance with minimum standards of physical and mental fitness for driving by drivers of vehicles used for the transport of persons or goods should be provided when the driving licence is issued and periodically thereafter. Such regular control in accordance with national rules of compliance with minimum standards will contribute to the free movement of persons, avoid distortions of competition and better take into account the specific responsibility of drivers of such vehicles. Member States should be allowed to impose medical examinations as a guarantee of compliance with the minimum standards of physical and mental fitness for driving other motor vehicles. For reasons of transparency, such examinations should coincide with a renewal of driving licences and therefore be determined by the period of validity of the licence. (10) It is necessary to strengthen further the principle of progressive access to the categories of two-wheeled vehicles and to the categories of vehicles used for the transport of passengers and goods. (11) Nevertheless, Member States should be allowed to set a higher age limit for the driving of certain categories of vehicles in order to further promote road safety; Member States should in exceptional circumstances be allowed to set lower age limits in order to take account of national circumstances. (12) The definitions of the categories should reflect to a greater extent the technical characteristics of the vehicles concerned and the skills needed to drive a vehicle. (13) Introducing a category of driving licences for mopeds will, in particular, increase road safety as regards the youngest drivers who, according to the statistics, are the hardest hit by road accidents. (14) Specific provisions should be adopted to make it easier for physically disabled persons to drive vehicles. (15) For reasons connected with road safety, Member States should be able to apply their national provisions on the withdrawal, suspension, renewal and cancellation of driving licences to all licence holders having acquired normal residence in their territory. (16) The model driving licence as set out in Directive 91/439/EEC should be replaced by a single model in the form of a plastic card. At the same time, this model driving licence needs to be adapted on account of the introduction of a new category of driving licences for mopeds and of a new category of driving licences for motorcycles. (17) The introduction of an optional microchip in the new plastic card model driving licence should enable the Member States to further improve the level of anti-fraud protection. Member States should have flexibility to include national data on the chip provided that it does not interfere with commonly accessible data. The technical requirements for the microchip should be determined by the Commission, assisted by the committee on driving licences. (18) Minimum standards concerning access to the profession of examiner and examiner training requirements should be established in order to improve the knowledge and skills of examiners thereby ensuring a more objective evaluation of driving licence applicants and achieving greater harmonisation of driving tests. (19) The Commission should be allowed to undertake the adaptation of Annexes I to VI to scientific and technical progress. (20) The measures necessary for the implementation of this Directive should be adopted in accordance with Council Decision 1999/468/EC of 28 June 1999 laying down the procedures for the exercise of implementing powers conferred on the Commission (4). (21) In particular, the Commission should be empowered to establish the criteria necessary for the application of this Directive. Since those measures are of general scope and are designed to amend non-essential elements of this Directive, they should be adopted in accordance with the regulatory procedure with scrutiny provided for in Article 5a of Decision 1999/468/EC. (22) Since the objectives of this Directive cannot be sufficiently achieved by the Member States and can therefore, by reason of their scale and their effects, be better achieved at Community level, the Community may adopt measures, in accordance with the principle of subsidiarity as set out in Article 5 of the Treaty. In accordance with the principle of proportionality, as set out in that Article, this Directive does not go beyond what is necessary in order to achieve those objectives. (23) This Directive should not prejudice the obligations of the Member States relating to the deadlines for transposition into national law and application of the Directives listed in Annex VII, Part B, HAVE ADOPTED THIS DIRECTIVE: Article 1 Model licence 1. Member States shall introduce a national driving licence based on the Community model set out in Annex I, in accordance with the provisions of this Directive. The emblem on page 1 of the Community model driving licences shall contain the distinguishing sign of the Member State issuing the licence. 2. Without prejudice to data protection rules, Member States may introduce a storage medium (microchip) as part of the driving licence, as soon as the requirements concerning the microchip referred to in Annex I, which are designed to amend non-essential elements of this Directive, by supplementing it, are laid down by the Commission in accordance with the procedure referred to in Article 9(2). These requirements shall provide for EC type-approval, which shall only be granted when the ability to resist attempts to tamper with or alter data is demonstrated. 3. The microchip shall incorporate the harmonised driving licence data specified in Annex I. After consulting the Commission, Member States may store additional data, provided that it does not in any way interfere with the implementation of this Directive. In accordance with the procedure referred to in Article 9(2), the Commission may amend Annex I in order to guarantee future interoperability. 4. With the agreement of the Commission, Member States may make to the model set out in Annex I such adjustments as are necessary for computer processing of the driving licence. Article 2 Mutual recognition 1. Driving licences issued by Member States shall be mutually recognised. 2. When the holder of a valid national driving licence without the administrative validity period set out in Article 7(2) takes up normal residence in a Member State other than that which issued the driving licence, the host Member State may apply to the licence the administrative validity periods set out in that Article by renewing the driving licence, as from 2 years after the date on which the holder has taken up normal residence on its territory. Article 3 Anti-forgery measures 1. Member States shall take all necessary steps to avoid any risk of forgery of driving licences, including that of model driving licences issued before the entry into force of this Directive. They shall inform the Commission thereof. 2. The material used for the driving licence, as set out in Annex I, shall be made secure against forgery in application of specifications designed to amend non-essential elements of this Directive, by supplementing it, which are to be laid down by the Commission in accordance with the procedure referred to in Article 9(2). Member States are free to introduce additional security features. 3. Member States shall ensure that, by 19 January 2033, all driving licences issued or in circulation fulfil all the requirements of this Directive. Article 4 Categories, definitions and minimum ages 1. The driving licence provided for in Article 1 shall authorise the driving of power-driven vehicles in the categories defined hereafter. It may be issued from the minimum age indicated for each category. A power-driven vehicle means any self-propelled vehicle running on a road under its own power, other than a rail-borne vehicle. 2. mopeds: Category AM:  Two-wheel vehicles or three-wheel vehicles with a maximum design speed of not more than 45 km/h, as defined in Article 1(2)(a) of Directive 2002/24/EC of the European Parliament and of the Council of 18 March 2002 relating to the type-approval of two or three-wheel motor vehicles (5) (excluding those with a maximum design speed under or equal to 25 km/h), and light quadricycles as defined in Article 1(3)(a) of Directive 2002/24/EC,  the minimum age for category AM is fixed at 16 years; 3. motorcycles with or without a sidecar and motor tricycles:  motorcycle means two-wheel vehicles with or without a sidecar, as defined in Article 1(2)(b) of Directive 2002/24/EC,  motor tricycle means vehicles with three symmetrically arranged wheels, as defined in Article 1(2)(c) of Directive 2002/24/EC; (a) Category A1:  motorcycles with a cylinder capacity not exceeding 125 cubic centimetres, of a power not exceeding 11 kW and with a power/weight ratio not exceeding 0,1 kW/kg,  motor tricycles with a power not exceeding 15 kW,  the minimum age for category A1 is fixed at 16 years; (b) Category A2:  motorcycles of a power not exceeding 35 kW and with a power/weight ratio not exceeding 0,2 kW/kg and not derived from a vehicle of more than double its power,  the minimum age for category A2 is fixed at 18 years; (c) Category A: (i) motorcycles  The minimum age for category A is fixed at 20 years. However, access to the driving of motorcycles of this category shall be subject to a minimum of two years' experience on motorcycles under an A2 licence. This requirement as to previous experience may be waived if the candidate is at least 24 years old. (ii) motor tricycles with a power exceeding 15 kW  The minimum age for motor tricycles exceeding 15 kW is fixed at 21 years. 4. motor vehicles:  motor vehicle means any power-driven vehicle, which is normally used for carrying persons or goods by road or for drawing, on the road, vehicles used for the carriage of persons or goods. This term shall include trolleybuses, i.e. vehicles connected to an electric conductor and not rail-borne. It shall not include agricultural or forestry tractors,  Agricultural or forestry tractor means any power-driven vehicle running on wheels or tracks, having at least two axles, the principal function of which lies in its tractive power, which is specially designed to pull, push, carry or operate certain tools, machines or trailers used in connection with agricultural or forestry operations, and the use of which for carrying persons or goods by road or drawing, on the road, vehicles used for the carriage of persons or goods is only a secondary function; (a) Category B1:  quadricycles, as defined in Article 1(3)(b) of Directive 2002/24/EC,  the minimum age for category B1 is fixed at 16 years,  category B1 is optional; in Member States which do not introduce this category of driving licence, a driving licence for category B shall be required to drive such vehicles; (b) Category B: motor vehicles with a maximum authorised mass not exceeding 3 500 kg and designed and constructed for the carriage of no more than eight passengers in addition to the driver; motor vehicles in this category may be combined with a trailer having a maximum authorised mass which does not exceed 750 kg. Without prejudice to the provisions of type-approval rules for the vehicles concerned, motor vehicles in this category may be combined with a trailer with a maximum authorised mass exceeding 750 kg, provided that the maximum authorised mass of this combination does not exceed 4 250 kg. In case such a combination exceeds 3 500 kg, Member States shall, in accordance with the provisions of Annex V, require that this combination shall only be driven after:  a training has been completed, or  a test of skills and behaviour has been passed. Member States may also require both such a training and the passing of a test of skills and behaviour. Member States shall indicate the entitlement to drive such a combination on the driving licence by means of the relevant Community code. The minimum age for category B is fixed at 18 years; (c) Category BE:  without prejudice to the provisions of type-approval rules for the vehicles concerned, combination of vehicles consisting of a tractor vehicle in category B and a trailer or semi-trailer where the maximum authorised mass of the trailer or semi-trailer does not exceed 3 500 kg,  the minimum age for category BE is fixed at 18 years; (d) Category C1: motor vehicles other than those in categories D1 or D, the maximum authorised mass of which exceeds 3 500 kg, but does not exceed 7 500 kg, and which are designed and constructed for the carriage of no more than eight passengers in addition to the driver; motor vehicles in this category may be combined with a trailer having a maximum authorised mass not exceeding 750 kg; (e) Category C1E:  without prejudice to the provisions of type-approval rules for the vehicles concerned, combinations of vehicles where the tractor vehicle is in category C1 and its trailer or semi-trailer has a maximum authorised mass of over 750 kg provided that the authorised mass of the combination does not exceed 12 000 kg,  without prejudice to the provisions of type-approval rules for the vehicles concerned, combinations of vehicles where the tractor vehicle is in category B and its trailer or semi-trailer has an authorised mass of over 3 500 kg, provided that the authorised mass of the combination does not exceed 12 000 kg,  the minimum age for categories C1 and C1E is fixed at the age of 18 years, without prejudice to the provisions for the driving of such vehicles in Directive 2003/59/EC of the European Parliament and of the Council of 15 July 2003 on the initial qualification and periodic training of drivers of certain road vehicles for the carriage of goods or passengers (6); (f) Category C: motor vehicles other than those in categories D1 or D, whose maximum authorised mass is over 3 500 kg and which are designed and constructed for the carriage of no more than eight passengers in addition to the driver; motor vehicles in this category may be combined with a trailer having a maximum authorised mass which does not exceed 750 kg; (g) Category CE:  without prejudice to the provisions of type-approval rules for the vehicles concerned, combinations of vehicles where the tractor vehicle is in category C and its trailer or semi-trailer has a maximum authorised mass of over 750 kg,  the minimum age for categories C and CE is fixed at 21 years, without prejudice to the provisions for the driving of such vehicles in Directive 2003/59/EC; (h) Category D1: motor vehicles designed and constructed for the carriage of no more than 16 passengers in addition to the driver and with a maximum length not exceeding 8 m; motor vehicles in this category may be combined with a trailer having a maximum authorised mass not exceeding 750 kg; (i) Category D1E:  without prejudice to the provisions of type-approval rules for the vehicles concerned, combinations of vehicles where the tractor vehicle is in category D1 and its trailer has a maximum authorised mass of over 750 kg,  the minimum age for categories D1 and D1E is fixed at 21 years, without prejudice to the provisions for the driving of such vehicles in Directive 2003/59/EC; (j) Category D: motor vehicles designed and constructed for the carriage of more than eight passengers in addition to the driver; motor vehicles which may be driven with a category D licence may be combined with a trailer having a maximum authorised mass which does not exceed 750 kg; (k) Category DE:  without prejudice to the provisions of type-approval rules for the vehicles concerned, combinations of vehicles where the tractor vehicle is in category D and its trailer has a maximum authorised mass of over 750 kg,  the minimum age for categories D and DE is fixed at 24 years, without prejudice to the provisions for the driving of such vehicles in Directive 2003/59/EC; 5. With the agreement of the Commission, Member States may exclude from the application of this Article certain specific types of power-driven vehicle such as special vehicles for disabled persons. Member States may exclude from the application of this Directive vehicles used by, or under the control of, the armed forces and civil defence. 6. Member States may raise or lower the minimum age for issuing a driving licence: (a) for category AM down to 14 years or up to 18 years; (b) for category B1 up to 18 years; (c) for category A1 up to 17 or 18 years,  if there is a two years difference between the minimum age for category A1 and the minimum age for category A2, and  there is a requirement of a minimum of two years experience on motorcycles of category A2 before access to the driving of motorcycles for category A can be granted, as referred to in Article 4(3)(c)(i); (d) for categories B and BE down to 17 years. Member States may lower the minimum age for category C to 18 years and for category D to 21 years with regard to: (a) vehicles used by the fire service and vehicles used for maintaining public order; (b) vehicles undergoing road tests for repair or maintenance purposes. Driving licences issued to persons at a lower age than set out in paragraphs 2 to 4 in accordance with this paragraph shall only be valid on the territory of the issuing Member State until the licence holder has reached the minimum age limit set out in paragraphs 2 to 4. Member States may recognise the validity on their territory of driving licences issued to drivers under the minimum ages set out in paragraphs 2 to 4. Article 5 Conditions and restrictions 1. Driving licences shall state the conditions under which the driver is authorised to drive. 2. If, because of a physical disability, driving is authorised only for certain types of vehicle or for adapted vehicles, the test of skills and behaviour provided for in Article 7 shall be taken in such a vehicle. Article 6 Staging and equivalences between categories 1. The issue of driving licences shall be subject to the following conditions: (a) licences for categories C1, C, D1 and D shall be issued only to drivers already entitled to drive vehicles in category B; (b) licences for categories BE, C1E, CE, D1E and DE shall be issued only to drivers already entitled to drive vehicles in categories B, C1, C, D1 and D respectively. 2. The validity of driving licences shall be determined as follows: (a) licences granted for categories C1E, CE, D1E or DE shall be valid for combinations of vehicles in category BE; (b) licences granted for category CE shall be valid for category DE as long as their holders are entitled to drive vehicles in category D; (c) licences granted for category CE and DE shall be valid for combinations of vehicles in categories C1E and D1E respectively; (d) licences granted for any category shall be valid for vehicles in category AM. However, for driving licences issued on its territory, a Member State may limit the equivalences for category AM to categories A1, A2 and A, if that Member State imposes a practical test as a condition for obtaining category AM; (e) licences issued for category A2 shall also be valid for category A1; (f) licences granted for categories A, B, C or D shall be valid for categories A1, A2, B1, C1, or D1 respectively. 3. For driving on their territory, Member States may grant the following equivalences: (a) motor tricycles under a licence for category B, for motor tricycles with a power exceeding 15 kW provided that the holder of the licence for category B is at least 21 years old; (b) category A1 motorcycles under a licence for category B. As this paragraph is only valid on their territories, Member States shall not indicate on the driving licence that a holder is entitled to drive these vehicles. 4. Member States may, after consulting the Commission, authorise the driving on their territory of: (a) vehicles of category D1 (with a maximum authorised mass of 3 500 kg, excluding any specialised equipment intended for the carriage of disabled passengers) by holders over 21 years old of a driving licence for category B which was obtained at least two years earlier provided that the vehicles are being used by non-commercial bodies for social purposes and that the driver provides his services on a voluntary basis; (b) vehicles of a maximum authorised mass exceeding 3 500 kg by holders over 21 years old of a driving licence for category B which was obtained at least two years before, provided that the main purpose of the vehicles is to be used only when stationary as an instructional or recreational area, and that they are being used by non-commercial bodies for social purposes and that vehicles have been modified so that they may not be used either for the transport of more than nine persons or for the transport of any goods other than those strictly necessary for their purposes. Article 7 Issue, validity and renewal 1. Driving licences shall be issued only to those applicants: (a) who have passed a test of skills and behaviour and a theoretical test and who meet medical standards, in accordance with the provisions of Annexes II and III; (b) who have passed a theory test only as regards category AM; Member States may require applicants to pass a test of skills and behaviour and a medical examination for this category. For tricycles and quadricycles within this category, Member States may impose a distinctive test of skills and behaviour. For the differentiation of vehicles in category AM, a national code may be inserted on the driving licence; (c) who have, as regards category A2 or category A, on the condition of having acquired a minimum of 2 years' experience on a motorcycle in category A1 or in category A2 respectively, passed a test of skills and behaviour only, or completed a training pursuant to Annex VI; (d) who have completed a training or passed a test of skills and behaviour, or completed a training and passed a test of skills and behaviour pursuant to Annex V as regards category B for driving a vehicle combination as defined in the second subparagraph of Article 4(4)(b); (e) who have their normal residence in the territory of the Member State issuing the licence, or can produce evidence that they have been studying there for at least six months. 2. (a) As from 19 January 2013, licences issued by Member States for categories AM, A1, A2, A, B, B1 and BE shall have an administrative validity of 10 years. A Member State may choose to issue such licences with an administrative validity of up to 15 years; (b) As from 19 January 2013, licences issued by Member States for categories C, CE, C1, C1E, D, DE, D1, D1E shall have an administrative validity of 5 years; (c) The renewal of a driving licence may trigger a new administrative validity period for another category or categories the licence holder is entitled to drive, insofar as this is in conformity with the conditions laid down in this Directive; (d) The presence of a microchip pursuant to Article 1 shall not be a prerequisite for the validity of a driving licence. The loss or unreadability of the microchip, or any other damage thereto, shall not affect the validity of the document. 3. The renewal of driving licences when their administrative validity expires shall be subject to: (a) continuing compliance with the minimum standards of physical and mental fitness for driving set out in Annex III for driving licences in categories C, CE, C1, C1E, D, DE, D1, D1E; and (b) normal residence in the territory of the Member State issuing the licence, or evidence that applicants have been studying there for at least six months. Member States may, when renewing driving licences in categories AM, A, A1, A2, B, B1 and BE, require an examination applying the minimum standards of physical and mental fitness for driving set out in Annex III. Member States may limit the period of administrative validity set out in paragraph 2 of driving licences issued to novice drivers for any category in order to apply specific measures to such drivers, aiming at improving road safety. Member States may limit the period of administrative validity of the first licence issued to novice drivers for categories C and D to 3 years in order to be able to apply specific measures to such drivers, so as to improve their road safety. Member States may limit the period of administrative validity set out in paragraph 2 of individual driving licences for any category in case it is found necessary to apply an increased frequency of medical checks or other specific measures such as restrictions for traffic offenders. Member States may reduce the period of administrative validity set out in paragraph 2 of driving licences of holders residing on their territory having reached the age of 50 years in order to apply an increased frequency of medical checks or other specific measures such as refresher courses. This reduced period of administrative validity can only be applied upon renewing the driving licence. 4. Without prejudice to national criminal and police laws, Member States may, after consulting the Commission, apply to the issuing of driving licences the provisions of their national rules relating to conditions other than those referred to in this Directive. 5. (a) No person may hold more than one driving licence; (b) A Member State shall refuse to issue a licence where it establishes that the applicant already holds a driving licence; (c) Member States shall take the necessary measures pursuant to point (b). The necessary measures as regards the issue, replacement, renewal or exchange of a driving licence shall be to verify with other Member States where there are reasonable grounds to suspect that the applicant is already the holder of another driving licence; (d) In order to facilitate the checks pursuant to point (b), Member States shall use the EU driving licence network once it is operational. Without prejudice to Article 2, a Member State issuing a licence shall apply due diligence to ensure that a person fulfils the requirements set out in paragraph 1 of this Article and shall apply its national provisions on the cancellation or withdrawal of the right to drive if it is established that a licence has been issued without the requirements having been met. Article 8 Adaptation to scientific and technical progress The amendments necessary to adapt Annexes I to VI to scientific and technical progress shall be adopted in accordance with the procedure referred to in Article 9(2). Article 9 Committee 1. The Commission shall be assisted by the committee on driving licences. 2. Where reference is made to this paragraph, Article 5a(1) to (4), and Article 7 of Decision 1999/468/EC shall apply, having regard to the provisions of Article 8 thereof. Article 10 Examiners From the entry into force of this Directive, driving examiners shall meet the minimum standards set out in Annex IV. Driving examiners already working in that capacity before 19 January 2013 shall be subject only to the requirements concerning quality assurance and regular periodic training measures. Article 11 Various provisions concerning the exchange, the withdrawal, the replacement and the recognition of driving licences 1. Where the holder of a valid national driving licence issued by a Member State has taken up normal residence in another Member State, he may request that his driving licence be exchanged for an equivalent licence. It shall be for the Member State effecting the exchange to check for which category the licence submitted is in fact still valid. 2. Subject to observance of the principle of territoriality of criminal and police laws, the Member State of normal residence may apply its national provisions on the restriction, suspension, withdrawal or cancellation of the right to drive to the holder of a driving licence issued by another Member State and, if necessary, exchange the licence for that purpose. 3. The Member State effecting the exchange shall return the old licence to the authorities of the Member State which issued it and give the reasons for doing so. 4. A Member State shall refuse to issue a driving licence to an applicant whose driving licence is restricted, suspended or withdrawn in another Member State. A Member State shall refuse to recognise the validity of any driving licence issued by another Member State to a person whose driving licence is restricted, suspended or withdrawn in the former State's territory. A Member State may also refuse to issue a driving licence to an applicant whose licence is cancelled in another Member State. 5. A replacement for a driving licence which has, for example, been lost or stolen may only be obtained from the competent authorities of the Member State in which the holder has his normal residence; those authorities shall provide the replacement on the basis of the information in their possession or, where appropriate, proof from the competent authorities of the Member State which issued the original licence. 6. Where a Member State exchanges a driving licence issued by a third country for a Community model driving licence, such exchange shall be recorded on the Community model driving licence as shall any subsequent renewal or replacement. Such an exchange may occur only if the licence issued by the third country has been surrendered to the competent authorities of the Member State making the exchange. If the holder of this licence transfers his normal residence to another Member State, the latter need not apply the principle of mutual recognition set out in Article 2. Article 12 Normal residence For the purpose of this Directive, normal residence means the place where a person usually lives, that is for at least 185 days in each calendar year, because of personal and occupational ties, or, in the case of a person with no occupational ties, because of personal ties which show close links between that person and the place where he is living. However, the normal residence of a person whose occupational ties are in a different place from his personal ties and who consequently lives in turn in different places situated in two or more Member States shall be regarded as being the place of his personal ties, provided that such person returns there regularly. This last condition need not be met where the person is living in a Member State in order to carry out a task of a definite duration. Attendance at a university or school shall not imply transfer of normal residence. Article 13 Equivalences between non-Community model licences 1. With the agreement of the Commission, Member States shall establish equivalences between entitlements obtained before the implementation of this Directive and the categories defined in Article 4. After consulting the Commission, Member States may make to their national legislation such adjustments as are necessary for the purpose of implementing the provisions of Article 11(4), (5) and (6). 2. Any entitlement to drive granted before 19 January 2013 shall not be removed or in any way qualified by the provisions of this Directive. Article 14 Review The Commission shall report on the implementation of this Directive, including its impact on road safety, not earlier than 19 January 2018. Article 15 Mutual Assistance Member States shall assist one another in the implementation of this Directive and shall exchange information on the licences they have issued, exchanged, replaced, renewed or revoked. They shall use the EU driving licence network set up for these purposes, once this network is operational. Article 16 Transposition 1. Member States shall adopt and publish, not later than 19 January 2011, the laws, regulations and administrative provisions necessary to comply with Article 1(1), Article 3, Article 4(1), (2), (3) and (4)(b) to (k), Article 6(1), (2)(a), (c), (d) and (e), Article 7(1)(b), (c) and (d), (2), (3) and (5), Article 8, Article 10, Article 13, Article 14, Article 15, and Annexes I, point 2, II, point 5.2 concerning categories A1, A2 and A, IV, V and VI. They shall forthwith communicate to the Commission the text of those provisions. 2. They shall apply those provisions as from 19 January 2013. 3. When Member States adopt those provisions, they shall contain a reference to this Directive or shall be accompanied by such reference on the occasion of their official publication. They shall also contain an indication that references made, in the laws, regulations or administrative provisions in force, to the repealed Directive shall be construed as being made to this Directive. The methods of making such reference, and its wording, shall be laid down by Member States. 4. Member States shall communicate to the Commission the text of the main provisions of national law which they adopt in the field covered by this Directive. Article 17 Repeal Directive 91/439/EEC shall be repealed with effect from 19 January 2013, without prejudice to the obligations of the Member States with regard to the deadlines indicated in Annex VII, Part B for transposing that Directive into national law. Article 2(4) of Directive 91/439/EEC shall be repealed on 19 January 2007. References made to the repealed Directive shall be construed as being made to this Directive and should be read in accordance with the correlation table in Annex VIII. Article 18 Entry into force This Directive shall enter into force on the twentieth day following that of its publication in the Official Journal of the European Union. Article 2(1), Article 5, Article 6(2)(b), Article 7(1)(a), Article 9, Article 11(1), (3), (4), (5) and (6), Article 12, and Annexes I, II and III shall apply from 19 January 2009. Article 19 Addressees This Directive is addressed to the Member States. Done at Brussels, 20 December 2006. For the European Parliament The President J. BORRELL FONTELLES For the Council The President J. KORKEAOJA (1) OJ C 112, 30.4.2004, p. 34. (2) Opinion of the European Parliament of 23 February 2005 (OJ C 304 E, 1.12.2005, p. 202), Council Common Position of 18 September 2006 (OJ C 295 E, 5.12.2006, p. 1) and Position of the European Parliament of 14 December 2006 (not yet published in the Official Journal). Council Decision of 19 December 2006. (3) OJ L 237, 24.8.1991, p. 1. Directive as last amended by Regulation (EC) No 1882/2003 of the European Parliament and of the Council (OJ L 284, 31.10.2003, p. 1). (4) OJ L 184, 17.7.1999, p. 23. Decision as amended by Decision 2006/512/EC (OJ L 200, 22.7.2006, p. 11). (5) OJ L 124, 9.5.2002, p. 1. Directive as last amended by Commission Directive 2005/30/EC (OJ L 106, 27.4.2005, p. 17). (6) OJ L 226, 10.9.2003, p. 4. Directive as amended by Council Directive 2004/66/EC (OJ L 168, 1.5.2004, p. 35). ANNEX I PROVISIONS CONCERNING THE COMMUNITY MODEL DRIVING LICENCE 1. The physical characteristics of the card of the Community model driving licence shall be in accordance with ISO 7810 and ISO 7816-1. The card shall be made of polycarbonate. Methods for testing the characteristics of driving licences for the purpose of confirming their compliance with the international standards shall be in accordance with ISO 10373. 2. Physical security of driving licences The threats to the physical security of driving licences are:  production of false cards: creating a new object which bears great resemblance to the document, either by making it from scratch or by copying an original document,  material alteration: changing a property of an original document, e.g. modifying some of the data printed on the document; The overall security lies in the system in its entirety, consisting of the application process, the transmission of data, the card body material, the printing technique, a minimum set of different security features and the personalisation process. (a) The material used for driving licences shall be made secure against forgery by using the following techniques (mandatory security features):  card bodies shall be UV dull,  a security background pattern designed to be resistant to counterfeit by scanning, printing or copying, using rainbow printing with multicolour security inks and positive and negative guilloche printing. The pattern shall not be composed of the primary colours (CMYK), shall contain complex pattern designs in a minimum of two special colours and shall include micro lettering,  optical variable elements providing adequate protection against copying and tampering of the photograph,  laser engraving,  in the area of the photograph the security design background and photograph should overlap on at least its border (weakening pattern). (b) In addition, the material used for driving licences shall be made secure against forgery by using at least three of the following techniques (additional security features):  colour-shifting inks*,  termochromic ink*,  custom holograms*,  variable laser images*,  ultraviolet fluorescent ink, visible and transparent,  iridescent printing,  digital watermark in the background,  infrared or phosphorescent pigments,  tactile characters, symbols or patterns*. (c) Member States are free to introduce additional security features. As a basis, the techniques indicated with an asterisk are to be preferred as they enable the law enforcement officers to check the validity of the card without any special means. 3. The licence shall have two sides. Page 1 shall contain: (a) the words Driving Licence printed in large type in the language or languages of the Member State issuing the licence; (b) the name of the Member State issuing the licence (optional); (c) the distinguishing sign of the Member State issuing the licence, printed in negative in a blue rectangle and encircled by twelve yellow stars; the distinguishing signs shall be as follows: B : Belgium CZ : Czech Republic DK : Denmark D : Germany EST : Estonia GR : Greece E : Spain F : France IRL : Ireland I : Italy CY : Cyprus LV : Latvia LT : Lithuania L : Luxembourg H : Hungary M : Malta NL : The Netherlands A : Austria PL : Poland P : Portugal SLO : Slovenia SK : Slovakia FIN : Finland S : Sweden UK : The United Kingdom; (d) information specific to the licence issued, numbered as follows: 1. surname of the holder; 2. other name(s) of the holder; 3. date and place of birth; 4. (a) date of issue of the licence; (b) date of expiry of the licence or a dash if the licence is valid indefinitely under the provision of Article 7(2)(c); (c) the name of the issuing authority (may be printed on page 2); (d) a different number from the one under heading 5, for administrative purposes (optional); 5. number of the licence; 6. photograph of the holder; 7. signature of the holder; 8. permanent place of residence, or postal address (optional); 9. category of vehicle(s) the holder is entitled to drive (national categories shall be printed in a different type from harmonised categories); (e) the words European Communities model in the language(s) of the Member State issuing the licence and the words Driving Licence in the other languages of the Community, printed in pink to form the background of the licence: Permiso de ConducciÃ ³n Ã idiÃ skÃ ½ prÃ ¯kaz KÃ ¸rekort FÃ ¼hrerschein Juhiluba Ã Ã ´Ã µÃ ¹Ã ± Ã Ã ´Ã ®Ã ³Ã ·Ã Ã ·Ã  Driving Licence Permis de conduire CeadÃ ºas TiomÃ ¡na Patente di guida VadÃ «tÃ ja apliecÃ «ba Vairuotojo paÃ ¾ymÃ jimas VezetÃ i engedÃ ©ly LiÃ enzja tas-Sewqan Rijbewijs Prawo Jazdy Carta de ConduÃ §Ã £o VodiÃ skÃ ½ preukaz VozniÃ ¡ko dovoljenje Ajokortti KÃ ¶rkort; (f) Colour references:  blue: Pantone Reflex Blue,  yellow: Pantone Yellow. Page 2 shall contain: (a) 9. category of vehicle(s) the holder is entitled to drive (national categories shall be printed in a different type from harmonised categories); 10. date of first issue of each category (this date must be repeated on the new licence in the event of subsequent replacement or exchange); 11. date of expiry of each category; 12. additional information/restriction(s), in code form, facing the (sub)category affected. The codes shall be as follows:  codes 01 to 99 : harmonised Community codes DRIVER (Medical reasons) 01. Sight correction and/or protection 01.01 Glasses 01.02 Contact lense(s) 01.03 Protective glass 01.04 Opaque lense 01.05 Eye cover 01.06 Glasses or contact lenses 02. Hearing aid/communication aid 02.01 Hearing aid for one ear 02.02 Hearing aid for two ears 03. Prosthesis/orthosis for the limbs 03.01 Upper limb prosthesis/orthosis 03.02 Lower limb prosthesis/orthosis 05. Limited use (subcode use obligatory, driving subject to restrictions for medical reasons) 05.01 Limited to day time journeys (for example: one hour after sunrise and one hour before sunset) 05.02 Limited to journeys within a radius of ¦ km from holder's place of residence or only inside city/region 05.03 Driving without passengers 05.04 Limited to journeys with a speed not greater than ¦ km/h 05.05 Driving authorised solely when accompanied by a holder of a driving licence 05.06 Without trailer 05.07 No driving on motorways 05.08 No alcohol VEHICLE ADAPTATIONS 10. Modified transmission 10.01 Manual transmission 10.02 Automatic transmission 10.03 Electronically operated transmission 10.04 Adjusted gear-shift lever 10.05 Without secondary gearbox 15. Modified clutch 15.01 Adjusted gear-shift lever 15.02 Manual clutch 15.03 Automatic clutch 15.04 Partitioning in front of/fold away/detached clutch pedal 20. Modified braking systems 20.01 Adjusted brake pedal 20.02 Enlarged brake pedal 20.03 Brake pedal suitable for use by left foot 20.04 Brake pedal by sole 20.05 Tilted brake pedal 20.06 Manual (adapted) service brake 20.07 Maximum use of reinforced service brake 20.08 Maximum use of emergency brake integrated in the service brake 20.09 Adjusted parking brake 20.10 Electrically operated parking brake 20.11 (Adjusted) foot operated parking brake 20.12 Partitioning in front of/fold away/detached brake pedal 20.13 Brake operated by knee 20.14 Electrically operated service brake 25. Modified accelerator systems 25.01 Adjusted accelerator pedal 25.02 Accelerator pedal by sole 25.03 Tilted accelerator pedal 25.04 Manual accelerator 25.05 Accelerator at knee 25.06 Servo accelerator (electronic, pneumatic, etc.) 25.07 Accelerator pedal on the left of brake pedal 25.08 Accelerator pedal on the left 25.09 Partitioning in front of/fold away/detached accelerator pedal 30. Modified combined braking and accelerator systems 30.01 Parallel pedals 30.02 Pedals at (or almost at) the same level 30.03 Accelerator and brake with sliding 30.04 Accelerator and brake with sliding and orthesis 30.05 Fold away/detached accelerator and brake pedals 30.06 Raised floor 30.07 Partitioning on the side of the brake pedal 30.08 Partitioning for prosthesis on the side of the brake pedal 30.09 Partitioning in front of the accelerator and brake pedals 30.10 Heel/leg support 30.11 Electrically operated accelerator and brake 35. Modified control layouts (Lights switches, windscreen wiper/washer, horn, direction indicators, etc.) 35.01 Control devices operable without negative influence on the steering and handling 35.02 Control devices operable without releasing the steering wheel and accessories (knob, fork, etc.) 35.03 Control devices operable without releasing the steering wheel and accessories (knob, fork, etc.) with the left hand 35.04 Control devices operable without releasing the steering wheel and accessories (knob, fork, etc.) with the right hand 35.05 Control devices operable without releasing the steering wheel and accessories (knob, fork, etc.) and the combined accelerator and braking mechanismss 40. Modified steering 40.01 Standard assisted steering 40.02 Reinforced assisted steering 40.03 Steering with backup system 40.04 Lengthened steering column 40.05 Adjusted steering wheel (Larger and/or thicker steering wheel section, reduced diameter steering wheel, etc.) 40.06 Tilted steering wheel 40.07 Vertical steering wheel 40.08 Horizontal steering wheel 40.09 Foot operated driving 40.10 Alternative adjusted steering (joy-stick, etc.) 40.11 Knob on the steering wheel 40.12 Hand orthesis on the steering wheel 40.13 With orthesis tenodese 42. Modified rearview mirror(s) 42.01 External (left or) right-side rear-view mirror 42.02 External rear-view mirror set on the wing 42.03 Additional inside rear-view mirror permitting view of traffic 42.04 Panoramic inside rear-view mirror 42.05 Blind spot rear-view mirror 42.06 Electrically operated outside rear-view mirror(s) 43. Modified driver seat 43.01 Driver seat at a good viewing height and in normal distance from the steering wheel and the pedal 43.02 Driver seat adjusted to body shape 43.03 Driver seat with lateral support for good sitting stability 43.04 Driver seat with armrest 43.05 Lengthening of sliding driver's seat 43.06 Seat-belt adjustment 43.07 Harness-type seat-belt 44. Modifications to motorcycles (subcode use obligatory) 44.01 Single operated brake 44.02 (Adjusted) hand operated brake (front wheel) 44.03 (Adjusted) foot operated brake (back wheel) 44.04 (Adjusted) accelerator handle 44.05 (Adjusted) manual transmission and manual clutch 44.06 (Adjusted) rear-view mirror(s) 44.07 (Adjusted) commands (direction indicators, braking light, ¦) 44.08 Seat height allowing the driver, in sitting position, to have two feet on the road at the same time 45. Motorcycle with side-car only 50. Restricted to a specific vehicle/chassis number (vehicle identification number, VIN) 51. Restricted to a specific vehicle/registration plate (vehicle registration number, VRN) ADMINISTRATIVE MATTERS 70. Exchange of licence No ¦ issued by ¦ (EU/UN distinguishing sign in the case of a third country; e.g: 70.0123456789.NL) 71. Duplicate of licence No ¦ (EU/UN distinguishing sign in the case of a third country; e.g: 71.987654321.HR) 72. Restricted to category A vehicles having a maximum cylinder capacity of 125 cc and maximum power of 11 KW (A1) 73. Restricted to category B vehicles of the motor tricycle or quadricycle type (B1) 74. Restricted to category C vehicles the maximum authorised mass of which does not exceed 7 500 kg (C1) 75. Restricted to category D vehicles with not more than 16 seats, excluding the driver's seat (D1) 76. Restricted to category C vehicles the maximum authorised mass of which does not exceed 7 500 kg (C1), attached to a trailer the maximum authorised mass of which exceeds 750 kg, provided that the maximum authorised mass of the vehicle train thus formed does not exceed 12 000 kg, and that the maximum authorised mass of the trailer does not exceed the unladen mass of the drawing vehicle (C1E) 77. Restricted to category D vehicles with not more than 16 passenger seats, excluding the driver's seat (D1), attached to a trailer the maximum authorised mass of which exceeds 750 kg provided that (a) the maximum authorised mass of the vehicle train thus formed does not exceed 12 000 kg and the maximum authorised mass of the trailer does not exceed the unladen mass of the drawing vehicle and (b) the trailer is not used to carry passengers (D1E) 78. Restricted to vehicles with automatic transmission 79. ( ¦) Restricted to vehicles which comply with the specifications indicated in brackets, in the context of the application of Article 10(1) of Directive 91/439/EEC 90.01 : to the left 90.02 : to the right 90.03 : left 90.04 : right 90.05 : hand 90.06 : foot 90.07 : usable 95. Driver holding CPC meeting the obligation of professional aptitude provided for by Directive 2003/59/EC until ¦ [e.g.: 95.01.01.2012] 96. Driver having completed training or having passed a test of skills and behaviour in accordance with the provisions of Annex V.  codes 100 and above: : national codes valid only for driving in the territory of the Member State which issued the licence. Where a code applies to all categories for which the licence is issued, it may be printed under headings 9, 10 and 11; 13. in implementation of section 4(a) of this Annex, a space reserved for the possible entry by the host Member State of information essential for administering the licence; 14. a space reserved for the possible entry by the Member State which issues the licence of information essential for administering the licence or related to road safety (optional). If the information relates to one of the headings defined in this Annex, it should be preceded by the number of the heading in question. With the specific written agreement of the holder, information which is not related to the administration of the driving licence or road safety may also be added in this space; such addition shall not alter in any way the use of the model as a driving licence; (b) an explanation of the numbered items which appear on pages 1 and 2 of the licence (at least items 1, 2, 3, 4 (a), 4 (b), 4 (c), 5, 10, 11 and 12) If a Member State wishes to make the entries in a national language other than one of the following languages: Czech, Danish, Dutch, English, Estonian, Finnish, French, German, Greek, Hungarian, Italian, Latvian, Lithuanian, Maltese, Polish, Portuguese, Slovak, Slovenian, Spanish or Swedish, it shall draw up a bilingual version of the licence using one of the aforementioned languages, without prejudice to the other provisions of this Annex; (c) a space shall be reserved on the Community model licence to allow for the possible introduction of a microchip or similar computer device. 4. Special provisions (a) Where the holder of a driving licence issued by a Member State in accordance with this Annex has his normal place of residence in another Member State, that Member State may enter in the licence such information as is essential for administering it, provided that it also enters this type of information in the licences which it issues and provided that there remains enough space for the purpose. (b) After consulting the Commission, Member States may add colours or markings, such as bar codes and national symbols, without prejudice to the other provisions of this Annex. In the context of mutual recognition of licences, the bar code may not contain information other than what can already be read on the driving licence or which is essential to the process of issuing the licence. COMMUNITY MODEL DRIVING LICENCE Page 1 DRIVING LICENCE [MEMBER STATE] Page 2 1. Name 2. First name 3. Date and place of birth 4a. Date of issue of driving licence 4b. Official date of expiry 4c. Issued by 5. Serial number of licence 8. Place of residence 9. Category (1) 10. Date of issue, by category 11. Date of expiry, by category 12. Restrictions SPECIMEN MODEL LICENCE BELGIAN LICENCE (for information) (1) Note: a pictogram and a line for category AM will be added. Note: the term A2 will be added to the section on motorcycle categories. ANNEX II I. MINIMUM REQUIREMENTS FOR DRIVING TESTS Member States shall take the necessary measures to ensure that applicants for driving licences possess the knowledge and skills and exhibit the behaviour required for driving a motor vehicle. The tests introduced to this effect must consist of:  a theory test, and then  a test of skills and behaviour. The conditions under which these tests shall be conducted are set out below. A. THEORY TEST 1. Form The form chosen shall be such as to make sure that the applicant has the required knowledge of the subjects listed on points 2, 3 and 4. Any applicant for a licence in one category who has passed a theory test for a licence in a different category may be exempt from the common provisions of points 2, 3 and 4. 2. Content of the theory test concerning all vehicle categories 2.1. Questions must be asked on each of the points listed below, the content and form of the questions being left to the discretion of each Member State: 2.1.1. Road traffic regulations:  in particular as regards road signs, markings and signals, rights of way and speed limits; 2.1.2. The driver:  importance of alertness and of attitude to other road users,  perception, judgement and decision-taking, especially reaction time, as well as changes in driving behaviour due to the influence of alcohol, drugs and medicinal products, state of mind and fatigue; 2.1.3. The road:  the most important principles concerning the observance of a safe distance between vehicles, braking distances and road holding under various weather and road conditions,  driving risk factors related to various road conditions, in particular as they change with the weather and the time of day or night,  characteristics of various types of road and the related statutory requirements; 2.1.4. Other road users:  specific risk factors related to the lack of experience of other road users and the most vulnerable categories of users such as children, pedestrians, cyclists and people whose mobility is reduced,  risks involved in the movement and driving of various types of vehicles and of the different fields of view of their drivers; 2.1.5. General rules and regulations and other matters:  rules concerning the administrative documents required for the use of vehicles,  general rules specifying how the driver must behave in the event of an accident (setting warning devices and raising the alarm) and the measures which he can take to assist road accident victims where necessary,  safety factors relating to the vehicle, the load and persons carried; 2.1.6. Precautions necessary when alighting from the vehicle; 2.1.7. Mechanical aspects with a bearing on road safety; applicants must be able to detect the most common faults, in particular in the steering, suspension and braking systems, tyres, lights and direction indicators, reflectors, rear-view mirrors, windscreen and wipers, the exhaust system, seat-belts and the audible warning device; 2.1.8. Vehicle safety equipment and, in particular, the use of seat-belts, head restraints and child safety equipment; 2.1.9. Rules regarding vehicle use in relation to the environment (appropriate use of audible warning devices, moderate fuel consumption, limitation of pollutant emissions, etc.). 3. Specific provisions concerning categories A1, A2 and A 3.1. Compulsory check of general knowledge on: 3.1.1. Use of protective outfit such as gloves, boots, clothes and safety helmet; 3.1.2. Visibility of motorcycle riders for other road users; 3.1.3. Risk factors related to various road conditions as laid down above with additional attention to slippery parts such as drain covers, road markings such as lines and arrows, tram rails; 3.1.4. Mechanical aspects with a bearing on road safety as laid down above with additional attention to the emergency stop switch, the oil levels and the chain. 4. Specific provisions concerning categories C, CE, C1, C1E, D, DE, D1 and D1E 4.1. Compulsory check of general knowledge on: 4.1.1. Rules on driving hours and rest periods as defined by Council Regulation (EEC) No 3820/85 of 20 December 1985 on the harmonisation of certain social legislation relating to road transport (1); use of the recording equipment as defined by Council Regulation (EEC) No 3821/85 of 20 December 1985 on recording equipment in road transport (2), 4.1.2. Rules concerning the type of transport concerned: goods or passengers; 4.1.3. Vehicle and transport documents required for the national and international carriage of goods and passengers; 4.1.4. How to behave in the event of an accident; knowledge of measures to be taken after an accident or similar occurrence, including emergency action such as evacuation of passengers and basic knowledge of first aid; 4.1.5. The precautions to be taken during the removal and replacement of wheels; 4.1.6. Rules on vehicle weights and dimensions; rules on speed limiters; 4.1.7. Obstruction of the field of view caused by the characteristics of their vehicles; 4.1.8. Reading a road map, route planning, including the use of electronic navigation systems (optional); 4.1.9. Safety factors relating to vehicle loading: controlling the load (stowing and fastening), difficulties with different kinds of load (e.g. liquids, hanging loads, ¦), loading and unloading goods and the use of loading equipment (categories C, CE, C1, C1E only); 4.1.10. The driver's responsibility in respect to the carriage of passengers; comfort and safety of passengers; transport of children; necessary checks before driving away; all sorts of buses should be part of the theory test (public service buses and coaches, buses with special dimensions, ¦) (categories D, DE, D1, D1E only). 4.2. Compulsory check of general knowledge on the following additional provisions concerning categories C, CE, D and DE: 4.2.1. The principles of the construction and functioning of: internal combustion engines, fluids (e.g. engine oil, coolant, washer fluid), the fuel system, the electrical system, the ignition system, the transmission system (clutch, gearbox, etc.); 4.2.2. Lubrication and antifreeze protection; 4.2.3. The principles of the construction, the fitting, correct use and care of tyres; 4.2.4. The principles of the types, operation, main parts, connection, use and day-to-day maintenance of brake fittings and speed governors, and use of anti-lock brakes; 4.2.5. The principles of the types, operation, main parts, connection, use and day-to-day maintenance of coupling systems (categories CE, DE only); 4.2.6. Methods of locating causes of breakdowns; 4.2.7. Preventive maintenance of vehicles and necessary running repairs; 4.2.8. The driver's responsibility in respect of the receipt, carriage and delivery of goods in accordance with the agreed conditions (categories C, CE only). B. TEST OF SKILLS AND BEHAVIOUR 5. The vehicle and its equipment 5.1. The driving of a vehicle with manual transmission shall be subject to the passing of a skills and behaviour test taken on a vehicle with manual transmission. If an applicant takes the test of skills and behaviour on a vehicle with automatic transmission this shall be recorded on any licence issued on the basis of such a test. Licences with this indication shall be used only for driving vehicles with automatic transmission. Vehicle with automatic transmission means a vehicle in which the gear ratio between the engine and the wheels can be varied by use only of the accelerator or the brakes 5.2. The vehicles used in tests of skills and behaviour shall comply with the minimum criteria given below. Member States may make provisions for more stringent criteria or add others. Category A1: Category A1 motorcycle without sidecar, with a cubic capacity of at least 120 cm3, and capable of a speed of at least 90 km/h Category A2: Motorcycle without sidecar, with a cylinder capacity of at least 400 cm3, and an engine power of at least 25 kW Category A Motorcycle without sidecar, with a cylinder capacity of at least 600 cm3, and an engine power of at least 40 kW Category B: A four-wheeled category B vehicle capable of a speed of at least 100 km/h; Category BE: A combination, made up of a category B test vehicle and a trailer with a maximum authorised mass of at least 1 000 kg, capable of a speed of at least 100 km/h, which does not fall within category B; the cargo compartment of the trailer shall consist of a closed box body which is at least as wide and as high as the motor vehicle; the closed box body may also be slightly less wide than the motor vehicle provided that the view to the rear is only possible by use of the external rear-view mirrors of the motor vehicle; the trailer shall be presented with a minimum of 800 kg real total mass; Category B1: A motor-powered quadricycle capable of a speed of at least 60 km/h; Category C: A category C vehicle with a maximum authorised mass of at least 12 000 kg, a length of at least 8 m, a width of at least 2,40 m and capable of a speed of at least 80 km/h; fitted with anti-lock brakes, equipped with a gearbox having at least eight forward ratios and recording equipment as defined by Regulation (EEC) No 3821/85; the cargo compartment shall consist of a closed box body which is at least as wide and as high as the cab; the vehicle shall be presented with a minimum of 10 000 kg real total mass; Category CE: either an articulated vehicle or a combination of a category C test vehicle and a trailer of at least 7,5 m in length; both the articulated vehicle and the combination shall have a maximum authorised mass of at least 20 000 kg, a length of at least 14 m and a width of at least 2,40 m, shall be capable of a speed of at least 80 km/h, fitted with anti-lock brakes, equipped with a gearbox having at least eight forward ratios and with recording equipment as defined by Regulation (EEC) No 3821/85; the cargo compartment shall consist of a closed box body which is at least as wide and as high as the cab; both the articulated vehicle and the combination shall be presented with a minimum of 15 000 kg real total mass; Category C1: A subcategory C1 vehicle with a maximum authorised mass of at least 4 000 kg, with a length of at least 5 m and capable of a speed of at least 80 km/h; fitted with anti-lock brakes and equipped with recording equipment as defined by Regulation (EEC) No 3821/85; the cargo compartment shall consist of a closed box body which is at least as wide and as high as the cab; Category C1E: A combination made up of a subcategory C1 test vehicle and a trailer with a maximum authorised mass of at least 1 250 kg; this combination shall be at least 8 m in length and capable of a speed of at least 80 km/h; the cargo compartment of the trailer shall consist of a closed box body which is at least as wide and as high as the cab; the closed box body may also be slightly less wide than the cab provided that the view to the rear is only possible by use of the external rear-view mirrors of the motor vehicle; the trailer shall be presented with a minimum of 800 kg real total mass; Category D: A category D vehicle with a length of at least 10 m, a width of at least 2,40 m and capable of a speed of at least 80 km/h; fitted with anti-lock brakes and equipped with recording equipment as defined by Regulation (EEC) No 3821/85; Category DE: A combination made up of a category D test vehicle and a trailer with a maximum authorised mass of at least 1 250 kg, a width of at least 2,40 m and capable of a speed of at least 80 km/h; the cargo compartment of the trailer shall consist of a closed box body which is at least 2 m wide and 2 m high; the trailer shall be presented with a minimum of 800 kg real total mass; Category D1: A subcategory D1 vehicle with a maximum authorised mass of at least 4 000 kg, with a length of at least 5 m and capable of a speed of at least 80 km/h; fitted with anti-lock brakes and equipped with recording equipment as defined by Regulation (EEC) No 3821/85; Category D1E: A combination made up of a subcategory D1 test vehicle and a trailer with a maximum authorised mass of at least 1 250 kg and capable of a speed of at least 80 km/h; the cargo compartment of the trailer shall consist of a closed box body which is at least 2 m wide and 2 m high; the trailer shall be presented with a minimum of 800 kg real total mass; Testing vehicles for categories BE, C, CE, C1, C1E, D, DE, D1 and D1E which are not in conformity with the minimum criteria given above but which were in use on or before the moment of entry into force of this Directive, may still be used for a period not exceeding ten years after that date. The requirements related to the load to be carried by these vehicles, may be implemented by Member States up to ten years from the moment of entry into force of Commission Directive 2000/56/EC (3). 6. Skills and behaviour to be tested concerning categories A1, A2 and A 6.1. Preparation and technical check of the vehicle with a bearing on road safety Applicants must demonstrate that they are capable of preparing to ride safely by satisfying the following requirements: 6.1.1. Adjust the protective outfit, such as gloves, boots, clothes and safety helmet; 6.1.2. Perform a random check on the condition of the tyres, brakes, steering, emergency stop switch (if applicable), chain, oil levels, lights, reflectors, direction indicators and audible warning device. 6.2. Special manoeuvres to be tested with a bearing on road safety 6.2.1. Putting the motorcycle on and off its stand and moving it, without the aid of the engine, by walking alongside the vehicle; 6.2.2. Parking the motorcycle on its stand; 6.2.3. At least two manoeuvres to be executed at slow speed, including a slalom; this should allow competence to be assessed in handling of the clutch in combination with the brake, balance, vision direction and position on the motorcycle and the position of the feet on the foot rests; 6.2.4. At least two manoeuvres to be executed at higher speed, of which one manoeuvre in second or third gear, at least 30 km/h and one manoeuvre avoiding an obstacle at a minimum speed of 50 km/h; this should allow competence to be assessed in the position on the motorcycle, vision direction, balance, steering technique and technique of changing gears; 6.2.5. Braking: at least two braking exercises shall be executed, including an emergency brake at a minimum speed of 50 km/h; this should allow competence to be assessed in handling of the front and rear brake, vision direction and the position on the motorcycle. The special manoeuvres mentioned under points 6.2.3 to 6.2.5 have to be implemented at the latest five years after entry into force of Directive 2000/56/EC. 6.3. Behaviour in traffic Applicants must perform all the following actions in normal traffic situations, in complete safety and taking all necessary precautions: 6.3.1. Riding away: after parking, after a stop in traffic; exiting a driveway; 6.3.2. Riding on straight roads; passing oncoming vehicles, including in confined spaces; 6.3.3. Riding round bends; 6.3.4. Crossroads: approaching and crossing of intersections and junctions; 6.3.5. Changing direction: left and right turns; changing lanes; 6.3.6. Approach/exit of motorways or similar (if available): joining from the acceleration lane; leaving on the deceleration lane; 6.3.7. Overtaking/passing: overtaking other traffic (if possible); riding alongside obstacles, e.g. parked cars; being overtaken by other traffic (if appropriate); 6.3.8. Special road features (if available): roundabouts; railway level crossings; tram/bus stops; pedestrian crossings; riding up-/downhill on long slopes; 6.3.9. Taking the necessary precautions when getting off the vehicle. 7. Skills and behaviour to be tested concerning categories B, B1 and BE 7.1. Preparation and technical check of the vehicle with a bearing on road safety Applicants must demonstrate that they are capable of preparing to drive safely by satisfying the following requirements: 7.1.1. Adjusting the seat as necessary to obtain a correct seated position; 7.1.2. Adjusting rear-view mirrors, seat belts and head restraints if available; 7.1.3. Checking that the doors are closed; 7.1.4. Performing a random check on the condition of the tyres, steering, brakes, fluids (e.g. engine oil, coolant, washer fluid), lights, reflectors, direction indicators and audible warning device; 7.1.5. Checking the safety factors relating to vehicle loading: body, sheets, cargo doors, cabin locking, way of loading, securing load (category BE only); 7.1.6. Checking the coupling mechanism and the brake and electrical connections (category BE only). 7.2. Categories B and B1: special manoeuvres to be tested with a bearing on road safety A selection of the following manoeuvres shall be tested (at least two manoeuvres for the four points, including one in reverse gear): 7.2.1. Reversing in a straight line or reversing right or left round a corner while keeping within the correct traffic lane; 7.2.2. Turning the vehicle to face the opposite way, using forward and reverse gears; 7.2.3. Parking the vehicle and leaving a parking space (parallel, oblique or right-angle, forwards or in reverse, on the flat, uphill or downhill); 7.2.4. Braking accurately to a stop; however, performing an emergency stop is optional. 7.3. Category BE: special manoeuvres to be tested with a bearing on road safety 7.3.1. Coupling and uncoupling, or uncoupling and re-coupling a trailer from its motor vehicle; the manoeuvre must involve the towing vehicle being parked alongside the trailer (i.e. not in one line); 7.3.2. Reversing along a curve, the line of which shall be left to the discretion of the Member States; 7.3.3. Parking safely for loading/unloading. 7.4. Behaviour in traffic Applicants must perform all the following actions in normal traffic situations, in complete safety and taking all necessary precautions: 7.4.1. Driving away: after parking, after a stop in traffic; exiting a driveway; 7.4.2. Driving on straight roads; passing oncoming vehicles, including in confined spaces; 7.4.3. Driving round bends; 7.4.4. Crossroads: approaching and crossing of intersections and junctions; 7.4.5. Changing direction: left and right turns; changing lanes; 7.4.6. Approach/exit of motorways or similar (if available): joining from the acceleration lane; leaving on the deceleration lane; 7.4.7. Overtaking/passing: overtaking other traffic (if possible); driving alongside obstacles, e.g. parked cars; being overtaken by other traffic (if appropriate); 7.4.8. Special road features (if available): roundabouts; railway level crossings; tram/bus stops; pedestrian crossings; driving up-/downhill on long slopes; 7.4.9. Taking the necessary precautions when alighting from the vehicle. 8. Skills and behaviour to be tested concerning categories C, CE, C1, C1E, D, DE, D1 and D1E 8.1. Preparation and technical check of the vehicle with a bearing on road safety Applicants must demonstrate that they are capable of preparing to drive safely by satisfying the following requirements: 8.1.1. Adjusting the seat as necessary to obtain a correct seated position; 8.1.2. Adjusting rear-view mirrors, seat belts and head restraints if available; 8.1.3. Random checks on the condition of the tyres, steering, brakes, lights, reflectors, direction indicators and audible warning device; 8.1.4. Checking the power-assisted braking and steering systems; checking the condition of the wheels, wheelnuts, mudguards, windscreen, windows and wipers, fluids (e.g. engine oil, coolant, washer fluid); checking and using the instrument panel including the recording equipment as defined in Regulation (EEC) No 3821/85; 8.1.5. Checking the air pressure, air tanks and the suspension; 8.1.6. Checking the safety factors relating to vehicle loading: body, sheets, cargo doors, loading mechanism (if available), cabin locking (if available), way of loading, securing load (categories C, CE, C1, C1E only); 8.1.7. Checking the coupling mechanism and the brake and electrical connections (categories CE, C1E, DE, D1E only); 8.1.8. Being capable of taking special vehicle safety measures; controlling the body, service doors, emergency exits, first aid equipment, fire extinguishers and other safety equipment (categories D, DE, D1, D1E only); 8.1.9. Reading a road map, route planning, including the use of electronic navigation systems (optional). 8.2. Special manoeuvres to be tested with a bearing on road safety 8.2.1. Coupling and uncoupling, or uncoupling and re-coupling a trailer from its motor vehicle; the manoeuvre must involve the towing vehicle being parked alongside the trailer (i.e. not in one line) (categories CE, C1E, DE, D1E only); 8.2.2. Reversing along a curve, the line of which shall be left to the discretion of the Member States; 8.2.3. Parking safely for loading/unloading at a loading ramp/platform or similar installation (categories C, CE, C1, C1E only); 8.2.4. Parking to let passengers on or off the bus safely (categories D, DE, D1, D1E only). 8.3. Behaviour in traffic Applicants must perform all the following actions in normal traffic situations, in complete safety and taking all necessary precautions: 8.3.1. Driving away: after parking, after a stop in traffic; exiting a driveway; 8.3.2. Driving on straight roads; passing oncoming vehicles, including in confined spaces; 8.3.3. Driving round bends; 8.3.4. Crossroads: approaching and crossing of intersections and junctions; 8.3.5. Changing direction: left and right turns; changing lanes; 8.3.6. Approach/exit of motorways or similar (if available): joining from the acceleration lane; leaving on the deceleration lane; 8.3.7. Overtaking/passing: overtaking other traffic (if possible); driving alongside obstacles, e.g. parked cars; being overtaken by other traffic (if appropriate); 8.3.8. Special road features (if available): roundabouts; railway level crossings; tram/bus stops; pedestrian crossings; driving up-/downhill on long slopes; 8.3.9. Taking the necessary precautions when alighting from the vehicle. 9. Marking of the test of skills and behaviour 9.1. For each of the abovementioned driving situations, the assessment must reflect the degree of ease with which the applicant handles the vehicle controls and his demonstrated capacity to drive in traffic in complete safety. The examiner must feel safe throughout the test. Driving errors or dangerous conduct immediately endangering the safety of the test vehicle, its passengers or other road users shall be penalised by failing the test, whether or not the examiner or accompanying person has to intervene. Nonetheless, the examiner shall be free to decide whether or not the skills and behaviour test should be completed. Driving examiners must be trained to assess correctly the applicants' ability to drive safely. The work of driving examiners must be monitored and supervised, by a body authorised by the Member State, to ensure correct and consistent application of fault assessment in accordance with the standards laid down in this Annex. 9.2. During their assessment, driving examiners shall pay special attention to whether an applicant is showing a defensive and social driving behaviour. This should reflect the overall style of driving and the driving examiner should take this into account in the overall picture of the applicant. It includes adapted and determined (safe) driving, taking into account road and weather conditions, taking into account other traffic, taking into account the interests of other road users (particularly the more vulnerable) and anticipation. 9.3. The driving examiner will furthermore assess whether the applicant is: 9.3.1. Controlling the vehicle; taking into account: proper use of safety belts, rear-view mirrors, head restraints; seat; proper use of lights and other equipment; proper use of clutch, gearbox, accelerator, braking systems (including third braking system, if available), steering; controlling the vehicle under different circumstances, at different speeds; steadiness on the road; the weight and dimensions and characteristics of the vehicle; the weight and type of load (categories BE, C, CE, C1, C1E, DE, D1E only); the comfort of the passengers (categories D, DE, D1, D1E only) (no fast acceleration, smoothly driving and no hard braking); 9.3.2. Driving economically and in an environmentally friendly way, taking into account the revolutions per minute, changing gears, braking and accelerating (categories BE, C, CE, C1, C1E, D, DE, D1, D1E only); 9.3.3. Observation: all-round observation; proper use of mirrors; far, middle, near distance vision; 9.3.4. Priority/giving way: priority at crossroads, intersections and junctions; giving way at other occasions (e.g. changing direction, changing lanes, special manoeuvres); 9.3.5. Correct position on the road: proper position on the road, in lanes, on roundabouts, round bends, suitable for the type and the characteristics of the vehicle; pre-positioning; 9.3.6. Keeping distance: keeping adequate distance to the front and the side; keeping adequate distance from other road users; 9.3.7. Speed: not exceeding the maximum allowed speed; adapting speed to weather/traffic conditions and where appropriate up to national speed limits; driving at such a speed that stopping within distance of the visible and free road is possible; adapting speed to general speed of same kind of road users; 9.3.8. Traffic lights, road signs and other indications: acting correctly at traffic lights; obeying instructions from traffic controllers; acting correctly at road signs (prohibitions or commands); take appropriate action at road markings; 9.3.9. Signalling: give signals where necessary, correctly and properly timed; indicating directions correctly; taking appropriate action with regard to all signals made by other road users; 9.3.10. Braking and stopping: decelerating in time, braking or stopping according to circumstances; anticipation; using the various braking systems (only for categories C, CE, D, DE); using speed reduction systems other than the brakes (only for categories C, CE, D, DE). 10. Length of the test The length of the test and the distance travelled must be sufficient to assess the skills and behaviour laid down in paragraph B of this Annex. In no circumstances should the time spent driving on the road be less than 25 minutes for categories A, A1, A2, B, B1 and BE and 45 minutes for the other categories. This does not include the reception of the applicant, the preparation of the vehicle, the technical check of the vehicle with a bearing on road safety, the special manoeuvres and the announcement of the outcome of the practical test. 11. Location of the test The part of the test to assess the special manoeuvres may be conducted on a special testing ground. Wherever practicable, the part of the test to assess behaviour in traffic should be conducted on roads outside built-up areas, expressways and motorways (or similar), as well as on all kinds of urban streets (residential areas, 30 and 50 km/h areas, urban expressways) which should represent the various types of difficulty likely to be encountered by drivers. It is also desirable for the test to take place in various traffic density conditions. The time spent driving on the road should be used in an optimal way to assess the applicant in all the various traffic areas that can be encountered, with a special emphasis on changing between these areas. II. KNOWLEDGE, SKILL AND BEHAVIOUR FOR DRIVING A POWER-DRIVEN VEHICLE Drivers of all power-driven vehicles must at any moment have the knowledge, skills and behaviour described under points 1 to 9, with a view to be able to:  Recognise traffic dangers and assess their seriousness,  Have sufficient command of their vehicle not to create dangerous situations and to react appropriately should such situations occur,  Comply with road traffic regulations, and in particular those intended to prevent road accidents and to maintain the flow of traffic,  Detect any major technical faults in their vehicles, in particular those posing a safety hazard, and have them remedied in an appropriate fashion,  Take account of all the factors affecting driving behaviour (e.g. alcohol, fatigue, poor eyesight, etc.) so as to retain full use of the faculties needed to drive safely,  Help ensure the safety of all road users, and in particular of the weakest and most exposed by showing due respect for others. Member States may implement the appropriate measures to ensure that drivers who have lost the knowledge, skills and behaviour as described under points 1 to 9 can recover this knowledge and these skills and will continue to exhibit such behaviour required for driving a motor vehicle. (1) OJ L 370, 31.12.1985, p. 1. Regulation as repealed by Regulation (EC) No 561/2006 of the European Parliament and of the Council (OJ L 102, 11.4.2006, p. 1). (2) OJ L 370, 31.12.1985, p. 8. Regulation as last amended by Regulation (EC) No 561/2006. (3) Commission Directive 2000/56/EC of 14 September 2000 amending Council Directive 91/439/EEC on driving licences (OJ L 237, 21.9.2000, p. 45). ANNEX III MINIMUM STANDARDS OF PHYSICAL AND MENTAL FITNESS FOR DRIVING A POWER-DRIVEN VEHICLE DEFINITIONS 1. For the purpose of this Annex, drivers are classified in two groups: 1.1. Group 1: drivers of vehicles of categories A, A1, A2, AM, B, B1 and BE. 1.2. Group 2: drivers of vehicles of categories C, CE, C1, C1E, D, DE, D1 and D1E. 1.3. National legislation may provide for the provisions set out in this Annex for Group 2 drivers to apply to drivers of Category B vehicles using their driving licence for professional purposes (taxis, ambulances, etc.). 2. Similarly, applicants for a first driving licence or for the renewal of a driving licence are classified in the group to which they will belong once the licence has been issued or renewed. MEDICAL EXAMINATIONS 3. Group 1: Applicants shall be required to undergo a medical examination if it becomes apparent, when the necessary formalities are being completed or during the tests which they have to undergo prior to obtaining a driving licence, that they have one or more of the medical disabilities mentioned in this Annex. 4. Group 2: Applicants shall undergo medical examinations before a driving licence is first issued to them and thereafter drivers shall be checked in accordance with the national system in place in the Member State of normal residence whenever their driving licence is renewed 5. The standards set by Member States for the issue or any subsequent renewal of driving licences may be stricter than those set out in this Annex. SIGHT 6. All applicants for a driving licence shall undergo an appropriate investigation to ensure that they have adequate visual acuity for driving power-driven vehicles. Where there is reason to doubt that the applicant's vision is adequate, he shall be examined by a competent medical authority. At this examination attention shall be paid the following in particular: visual acuity, field of vision, twilight vision and progressive eye diseases. For the purpose of this Annex, intra-ocular lenses shall not be considered corrective lenses. Group 1: 6.1. Applicants for a driving licence or for the renewal of such a licence shall have a binocular visual acuity, with corrective lenses if necessary, of at least 0,5 when using both eyes together. Driving licences shall not be issued or renewed if, during the medical examination, it is shown that the horizontal field of vision is less than 120o o, apart from exceptional cases duly justified by a favourable medical opinion and a positive practical test, or that the person concerned suffers from any other eye condition that would compromise safe driving. When a progressive eye disease is detected or declared, driving licences may be issued or renewed subject to the applicant undergoing regular examination by a competent medical authority. 6.2. Applicants for a driving licence, or for the renewal of such a licence, who have total functional loss of vision in one eye or who use only one eye (e.g. in the case of diplopia) must have a visual acuity of at least 0,6, with corrective lenses if necessary. The competent medical authority must certify that this condition of monocular vision has existed sufficiently long to allow adaptation and that the field of vision in this eye is normal. Group 2: 6.3. Applicants for a driving licence or for the renewal of such a licence must have a visual acuity, with corrective lenses if necessary, of at least 0,8 in the better eye and at least 0,5 in the worse eye. If corrective lenses are used to attain the values of 0,8 and 0,5, the uncorrected acuity in each eye must reach 0,05, or else the minimum acuity (0,8 and 0,5) must be achieved either by correction by means of glasses with a power not exceeding plus or minus 8 dioptres or with the aid of contact lenses (uncorrected vision = 0,05). The correction must be well tolerated. Driving licences shall not be issued to or renewed for applications or drivers without a normal binocular field of vision or suffering from diplopia. HEARING 7. Driving licences may be issued to or renewed for applicants or drivers in Group 2 subject to the opinion of the competent medical authorities; particular account will be taken in medical examinations of the scope for compensation. PERSONS WITH A LOCOMOTOR DISABILITY 8. Driving licences shall not be issued to or renewed for applicants or drivers suffering from complaints or abnormalities of the locomotor system which make it dangerous to drive a power-driven vehicle. Group 1: 8.1. Driving licences subject to certain restrictions, if necessary, may be issued to physically disabled applicants or drivers following the issuing of an opinion by a competent medical authority. This opinion must be based on a medical assessment of the complaint or abnormality in question and, where necessary, on a practical test. It must also indicate what type of modification to the vehicle is required and whether the driver needs to be fitted with an orthopaedic device, insofar as the test of skills and behaviour demonstrates that with such a device driving would not to be dangerous. 8.2. Driving licences may be issued to or renewed for any applicant suffering from a progressive complaint on condition that the disabled person is regularly examined to check that the person is still capable of driving the vehicle completely safely. Where the disability is static, driving licences may be issued or renewed without the applicant being subject to regular medical examination. Group 2: 8.3. The competent medical authority shall give due consideration to the additional risks and dangers involved in the driving of vehicles covered by the definition of this group. CARDIOVASCULAR DISEASES 9. Any disease capable of exposing an applicant for a first licence or a driver applying for renewal to a sudden failure of the cardiovascular system such that there is a sudden impairment of the cerebral functions constitutes a danger to road safety. Group 1: 9.1. Driving licences will not to be issued to, or renewed for, applicants or drivers with serious arrhythmia. 9.2. Driving licences may be issued to, or renewed for, applicants or drivers wearing a pacemaker subject to authorised medical opinion and regular medical check-ups. 9.3. The question of whether to issue or renew a licence for applicants or drivers suffering from abnormal arterial blood pressure shall be assessed with reference to the other results of the examination, any associated complications and the danger they might constitute for road safety. 9.4. Generally speaking, a driving licence shall not be issued to or renewed for applicants or drivers suffering from angina during rest or emotion. The issuing or renewal of a driving licence to any applicant or driver having suffered myocardial infarction shall be subject to authorised medical opinion and, if necessary, regular medical check-ups. Group 2: 9.5. The competent medical authority shall give due consideration to the additional risks and dangers involved in the driving of vehicles covered by the definition of this group. DIABETES MELLITUS 10. Driving licences may be issued to, or renewed for, applicants or drivers suffering from diabetes mellitus, subject to authorised medical opinion and regular medical check-ups appropriate to each case. Group 2: 10.1. Only in very exceptional cases may driving licences be issued to, or renewed for, applicants or drivers in this group suffering from diabetes mellitus and requiring insulin treatment, and then only where duly justified by authorised medical opinion and subject to regular medical check-ups. NEUROLOGICAL DISEASES 11. Driving licences shall not be issued to, or renewed for, applicants or drivers suffering from a serious neurological disease, unless the application is supported by authorised medical opinion. Neurological disturbances associated with diseases or surgical intervention affecting the central or peripheral nervous system, which lead to sensory or motor deficiencies and affect balance and coordination, must accordingly be taken into account in relation to their functional effects and the risks of progression. In such cases, the issue or renewal of the licence may be subject to periodic assessment in the event of risk of deterioration. 12. Epileptic seizures or other sudden disturbances of the state of consciousness constitute a serious danger to road safety if they occur in a person driving a power-driven vehicle. Group 1: 12.1. A licence may be issued or renewed subject to an examination by a competent medical authority and to regular medical check-ups. The authority shall decide on the state of the epilepsy or other disturbances of consciousness, its clinical form and progress (no seizure in the last two years, for example), the treatment received and the results thereof. Group 2: 12.2. Driving licences shall not be issued to or renewed for applicants or drivers suffering or liable to suffer from epileptic seizures or other sudden disturbances of the state of consciousness. MENTAL DISORDERS Group 1: 13.1. Driving licences shall not be issued to, or renewed for, applicants or drivers who suffer from:  severe mental disturbance, whether congenital or due to disease, trauma or neurosurgical operations,  severe mental retardation,  severe behavioural problems due to ageing; or personality defects leading to seriously impaired judgment, behaviour or adaptability, unless their application is supported by authorised medical opinion and, if necessary, subject to regular medical check-ups. Group 2: 13.2. The competent medical authority shall give due consideration to the additional risks and dangers involved in the driving of vehicles covered by the definition of this group. ALCOHOL 14. Alcohol consumption constitutes a major danger to road safety. In view of the scale of the problem, the medical profession must be very vigilant. Group 1: 14.1. Driving licences shall not be issued to, or renewed for, applicants or drivers who are dependent on alcohol or unable to refrain from drinking and driving. After a proven period of abstinence and subject to authorised medical opinion and regular medical check-ups, driving licences may be issued to, or renewed for, applicant or drivers who have in the past been dependent on alcohol. Group 2: 14.2. The competent medical authority shall give due consideration to the additional risks and dangers involved in the driving of vehicles covered by the definition of this group. DRUGS AND MEDICINAL PRODUCTS 15. Abuse: Driving licences shall not be issued to or renewed for applicants or drivers who are dependent on psychotropic substances or who are not dependent on such substances but regularly abuse them, whatever category of licence is requested. Regular use: Group 1: 15.1. Driving licences shall not be issued to, or renewed for, applicants or drivers who regularly use psychotropic substances, in whatever form, which can hamper the ability to drive safely where the quantities absorbed are such as to have an adverse effect on driving. This shall apply to all other medicinal products or combinations of medicinal products which affect the ability to drive. Group 2: 15.2. The competent medical authority shall give due consideration to the additional risks and dangers involved in the driving of vehicles covered by the definitions of this group. RENAL DISORDERS Group 1: 16.1. Driving licences may be issued or renewed for applicants and drivers suffering from serious renal insufficiency subject to authorised medical opinion and regular medical check-ups. Group 2: 16.2. Save in exceptional cases duly justified by authorised medical opinion, and subject to regular medical check-ups, driving licences shall not be issued to or renewed for applicants or drivers suffering from serious and irreversible renal deficiency. MISCELLANEOUS PROVISIONS Group 1: 17.1. Subject to authorised medical opinion and, if necessary, regular medical check-ups, driving licences may be issued to or renewed for applications or drivers who have had an organ transplant or an artificial implant which affects the ability to drive. Group 2: 17.2. The competent medical authority shall give due consideration to the additional risks and dangers involved in the driving of vehicles covered by the definition of this group. 18. As a general rule, where applicants or drivers suffer from any disorder which is not mentioned in the preceding paragraph but is liable to be, or to result in, a functional incapacity affecting safety at the wheel, driving licences shall not be issued or renewed unless the application is supported by authorised medical opinion and, if necessary, subject to regular medical check-ups. ANNEX IV MINIMUM STANDARDS FOR PERSONS WHO CONDUCT PRACTICAL DRIVING TESTS 1. Competences required by a driving examiner 1.1. A person authorised to conduct practical assessments in a motor vehicle of the driving performance of a candidate must have knowledge, skills and understanding related to the topics listed in points 1.2 to 1.6. 1.2. The competences of an examiner must be relevant to assessing the performance of a candidate seeking the category of driving licence entitlement for which the driving test is being undertaken. 1.3. Knowledge and understanding of driving and assessment:  theory of driving behaviour,  hazard perception and accident avoidance,  the syllabus underpinning driving test standards,  the requirements of the driving test,  relevant road and traffic legislation, including relevant EU and national legislation and interpretative guidelines,  assessment theory and techniques,  defensive driving. 1.4. Assessment skills:  ability to observe accurately, monitor, and evaluate overall candidate performance, in particular:  correct and comprehensive recognition of dangerous situations,  accurate determination of cause and likely effect of such situations,  achievement of competence and recognition of errors,  uniformity and consistency in assessment,  assimilate information quickly and extract key points,  look ahead, identify potential problems, and develop strategies to deal with them,  provide timely and constructive feedback. 1.5. Personal driving skills:  A person authorised to conduct a practical test for a category of driving licence must be able to drive to a consistently high standard that type of motor vehicle. 1.6. Quality of service:  establish and communicate what the candidate can expect during the test,  communicate clearly, choosing content, style and language to suit the audience and context and deal with enquiries from candidates,  provide clear feedback about the test result,  treat candidates with respect and indiscriminately. 1.7. Knowledge about vehicle technique and physics:  knowledge about vehicle technique such as steering, tyres, brakes, lights, specially for motorcycles and heavy vehicles,  loading safety,  knowledge about vehicle physics such as speed, friction, dynamics, energy. 1.8. Driving in a fuel efficient and environmentally friendly way. 2. General conditions 2.1. A category B driving examiner: (a) must have held a category B licence for at least 3 years; (b) must be at least 23 years old; (c) must have successfully completed the initial qualification provided for in point 3 of this Annex and subsequently followed the quality assurance and the periodic training arrangements as provided for in point 4 of this Annex; (d) must have terminated a vocational education that leads at least to a completion of level 3 as defined by Council Decision 85/368/EEC of 16 July 1985 on the comparability of vocational training qualifications between the Member States of the European Community (1); (e) may not be active as a commercial driving instructor in a driving school simultaneously. 2.2. A driving examiner for the other categories: (a) must hold a driving licence in the category concerned or possess equivalent knowledge through adequate professional qualification; (b) must have successfully completed the initial qualification provided for in point 3 of this Annex and subsequently followed the quality assurance and the periodic training arrangements as provided for in point 4 of this Annex; (c) must have been a qualified category B driving examiner for at least 3 years; this period may be waived provided that the examiner in question can provide evidence of:  at least 5 years of driving in the category concerned, or,  a theoretical and practical assessment of driving ability of a standard higher than that needed to obtain a driving licence thus making that requirement unnecessary, (d) must have completed a vocational education that leads at least to a termination of the level 3 as defined by Decision 85/368/EEC; (e) may not be active as a commercial driving instructor in a driving school simultaneously. 2.3. Equivalences 2.3.1. Member States may authorise an examiner to conduct driving tests for categories AM, A1, A2 and A upon passing the initial qualification prescribed in point 3 for one of these categories. 2.3.2. Member States may authorise an examiner to conduct driving tests for categories C1, C, D1 and D upon passing the initial qualification prescribed in point 3 for one of these categories. 2.3.3. Member States may authorise an examiner to conduct driving tests for categories BE, C1E, CE, D1E and DE upon passing the initial qualification prescribed in point 3 for one of these categories. 3. Initial qualification 3.1. Initial training 3.1.1. Before a person may be authorised to conduct driving tests, that person must satisfactorily complete such training programme as a Member State may specify in order to have the competences set out in point 1. 3.1.2. Member States must determine whether the content of any particular training programme will relate to authorisation to conduct driving tests for one driving licence category, or more than one. 3.2. Examinations 3.2.1. Before a person may be authorised to conduct driving tests, that person must demonstrate a satisfactory standard of knowledge, understanding, skills and aptitude in respect of the subjects listed in point 1. 3.2.2. Member States shall operate an examination process that assesses, in a pedagogically appropriate manner, the competences of the person as defined under point 1, in particular point 1.4. The examination process must include both a theoretical element and a practical element. Computer-based assessment may be used where appropriate. The details concerning the nature and duration of any tests and assessments within the examination shall be at the discretion of the individual Member States. 3.2.3. Member States must determine whether the content of any particular examination will relate to authorisation to conduct driving tests for one driving licence category, or more than one. 4. Quality assurance and periodic training 4.1. Quality assurance 4.1.1. Member States shall have in place quality assurance arrangements to provide for the maintenance of standards of driving examiners. 4.1.2. Quality assurance arrangements should involve the supervision of examiners at work, their further training and re-accreditation, their continuing professional development, and by periodic review of the outcomes of the driving tests that they have conducted. 4.1.3. Member States must provide that each examiner is subject to yearly supervision making use of quality assurance arrangements listed in point 4.1.2. Moreover, the Member States must provide that each examiner is observed conducting tests once every 5 years, for a minimum period cumulatively of at least half a day, allowing the observation of several tests. When issues are identified corrective action should be put in place. The person undertaking the supervision must be a person authorised by the Member State for that purpose. 4.1.4 Member States may provide that where an examiner is authorised to conduct driving tests in more than one category, satisfying the supervision requirement in relation to tests for one category satisfies the requirement for more than one category. 4.1.5 The work of driving examination must be monitored and supervised by a body authorised by the Member State, to ensure correct and consistent application of assessment. 4.2. Periodic training 4.2.1. Member States shall provide that, in order to remain authorised, driving examiners, irrespective of the number of categories for which they are accredited, undertake:  a minimum regular periodic training of four days in total per period of two years in order to:  maintain and refresh the necessary knowledge and examining skills,  to develop new competences that have become essential for the exercise of their profession,  ensure that an examiner continues to conduct tests to a fair and uniform standard,  a minimum periodic training of at least five days in total per period of five years,  in order to develop and maintain the necessary practical driving skills. 4.2.2. Member States shall take the appropriate measures for ensuring that specific training is given promptly to those examiners that have found to be seriously malfunctioning by the quality assurance system in place. 4.2.3. The nature of periodic training may take the form of briefing, classroom training, conventional or electronic-based learning, and it may be undertaken on an individual or group basis. It may include such re-accreditation of standards as Member States consider appropriate. 4.2.4. Member States may provide that where an examiner is authorised to conduct driving tests in more than one category, satisfying the periodic training requirement in relation to tests for one category satisfies the requirement for more than one category, provided the condition set out in point 4.2.5 is satisfied. 4.2.5. Where an examiner has not conducted tests for a category within a 24-month period, the examiner shall undertake a suitable reassessment before being allowed to carry out driving tests relating to that category. That re-assessment may be undertaken as part of the requirement set out in point 4.2.1. 5. Acquired rights 5.1. Member States may allow persons authorised to conduct driving tests immediately before these provisions come into force to continue to conduct driving tests, notwithstanding that they were not authorised in accordance with the general conditions in point 2 or the initial qualification process set out in point 3. 5.2. Such examiners are nonetheless subject to the regular supervision and quality assurance arrangements set out in point 4. (1) OJ L 199, 31.7.1985, p. 56. ANNEX V MINIMUM REQUIREMENTS FOR DRIVER TRAINING AND TESTING FOR COMBINATIONS AS DEFINED IN THE SECOND SUBPARAGRAPH OF ARTICLE 4(4)(B) 1. Member States shall take the necessary measures to:  approve and supervise the training provided for in Article 7(1)(d) or,  organise the test of skills and behaviour provided for in Article 7(1)(d). 2.1. Duration of driver training  at least 7 hours. 3. Content of driver training The driver training shall cover the knowledge, skills and behaviour as described in points 2 and 7 of Annex II. Particular attention shall be paid to:  vehicle movement dynamics, safety criteria, tractor vehicle and trailer (coupling mechanism), correct loading and safety fittings; A practical component shall include the following exercises: acceleration, deceleration, reversing, braking, stopping distance, lane-changing, braking/evasive action, trailer swing, uncoupling from and re-coupling a trailer to its motor vehicle, parking;  Each training participant has to perform the practical component and shall demonstrate its skills and behaviour on public roads,  Vehicle combinations used for the training shall fall within the category of driving licence participants have applied for. 4. Duration and contents of the test of skills and behaviour The length of the test and the distance travelled must be sufficient to assess the skills and behaviour laid down in point 3. ANNEX VI MINIMUM REQUIREMENTS FOR DRIVER TRAINING AND TESTING FOR MOTORCYCLES WITHIN CATEGORY A (PROGRESSIVE ACCESS) 1. Member States shall take the necessary measures to:  approve and supervise the training provided for in Article 7(1)(c) or,  organise the test of skills and behaviour provided for in Article 7(1)(c). 2. Duration of driver training  at least 7 hours. 3. Content of driver training  The driver training shall contain all aspects covered in point 6 of Annex II.  Each participant has to perform the practical components of the training and shall demonstrate its skills and behaviour on public roads.  Motorcycles used for the training shall fall within the category of driving licence participants have applied for. 4. Duration and contents of the test of skills and behaviour The length of the test and the distance travelled must be sufficient to assess the skills and behaviour laid down in point 3 of this Annex. ANNEX VII Part A REPEALED DIRECTIVE AS SUCCESSIVELY AMENDED (referred to in Article 17) Council Directive 91/439/EEC (1) (OJ L 237, 24.8.1991, p. 1) Council Directive 94/72/EC (OJ L 337, 24.12.1994, p. 86) Council Directive 96/47/EC (OJ L 235, 17.9.1996, p. 1) Council Directive 97/26/EC (OJ L 150, 7.6.1997, p. 41) Commission Directive 2000/56/EC (OJ L 237, 21.9.2000, p. 45) Directive 2003/59/EC of the European Parliament and of the Council, only Article 10, paragraph 2 (OJ L 226, 10.9.2003, p. 4) Regulation (EC) No 1882/2003 of the European Parliament and of the Council, only Annex II, point 24 (OJ L 284, 31.10.2003, p. 1) Part B DEADLINES FOR TRANSPOSITION INTO NATIONAL LAW AND FOR APPLICATION (referred to in Article 17) Directive Deadline for transposition Date of application Directive 91/439/EEC 1st July 1994 1st July 1996 Directive 94/72/EC - 1st January1995 Decision 96/427/EC - 16 July 1996 Directive 96/47/EC 1st July 1996 >1st July 1996 Directive 97/26/EC 1st January 1998 1st January 1998 Directive 2000/56/EC 30 September 2003 30 September 2003, 30 September 2008 (Annex II, point 6.2.5) and 30 September 2013 (Annex II point 5.2) Directive 2003/59/EC 10 September 2006 10 September 2008 (passenger transport) and 10 September 2009 (goods transport) (1) Directive 91/439/EEC was also amended by the following act which has not been repealed: 1994 Act of accession. ANNEX VIII CORRELATION TABLE Directive 91/439/EEC This Directive Article 1(1), first sentence Article 1(1) first sentence Article 1(1), second sentence  - Article 1(2) Article 1(2) Article 2(1) - Article 2(2) Article 1(3) - Article 2(1) Article 1(1), second sentence Article 2(2) Article 3(1) Article 3(2) Article 3(3) Article 2(3) - Article 2(4) - Article 3(1), first subparagraph, introductory words Article 4(1), first sentence - Article 4(2), first indent - Article 4(2), second indent Article 3(1), first subparagraph, first indent Article 4(3), first indent Article 3(1), first subparagraph, second indent Article 4(4)(b), first subparagraph Article 3(1), first subparagraph, third indent Article 4(4)(b), second subparagraph Article 3(1), first subparagraph, fourth indent Article 4(4)(c) Article 3(1), first subparagraph, fifth indent Article 4(4)(f) Article 3(1), first subparagraph, sixth indent Article 4(4)(g) Article 3(1), first subparagraph, seventh indent Article 4(4)(j) Article 3(1), first subparagraph, eighth indent Article 4(4)(k) Article 3(2), first subparagraph, introductory words - Article 3(2), first subparagraph, first indent Article 4(3)(a) Article 3(2), first subparagraph, second indent Article 4(4)(a) Article 3(2), first subparagraph, third indent Article 4(4)(d) Article 3(2), first subparagraph, fourth indent Article 4(4)(e) Article 3(2), first subparagraph, fifth indent Article 4(4)(h) Article 3(2), first subparagraph, sixth indent, introductory words Article 4(4)(i) Article 3(2), first subparagraph, sixth indent, first sub-indent - Article 3(2), first subparagraph, sixth indent, second sub-indent - Article 3(3), introductory words - Article 3(3), first indent Article 4(1), third sentence Article 3(3), second indent, first subparagraph Article 4(3), second indent Article 3(3), second indent, second subparagraph - Article 3(3), third indent Article 4(3), first indent Article 3(3), fourth indent Article 4(4), first indent Article 3(3), fifth indent Article 4(4), second indent - Article 4(3) Article 3(4) - Article 3(5) - Article 3(6) Article 4(5), first sentence - Article 4(5), second sentence Article 4 Article 5 Article 5(1) Article 6(1) Article 5(1)(a) Article 6(1)(a) Article 5(1)(b) Article 6(1)(b) Article 5(2), introductory words Article 6(2), introductory words Article 5(2)(a) Article 6(2)(a) Article 5(2)(b) Article 6(2)(b) - Article 6(2)(c) - Article 6(2)(d) - Article 6(2)(e) - Article 6(2)(f) Article 5(3) - Article 5(4) Article 6(4) Article 6(1), introductory words Article 4(1), second sentence Article 6(1)(a), first indent Article 4(3)(a), third indent Article 6(1)(a), second indent Article 4(4)(a), second indent Article 6(1)(b), first indent Article 4(3)(b), second indent Article 4(3)(c), second indent Article 6(1)(b), second indent first alternative Article 4(4)(b), fifth subparagraph Article 6(1)(b), second indent second alternative Article 4(4)(c), second indent Article 6(1)(b), third indent first and second alternative Article 4(4)(g), second indent Article 6(1)(b), third indent third and fourth alternative Article 4(4)(e), third indent Article 6(1)(c), first indent first and second alternative Article 4(4)(k), second indent Article 6(1)(c), first indent third and fourth alternative Article 4(4)(i), second indent Article 6(2) Article 4(6), first subparagraph - Article 4(6), second subparagraph Article 6(3) Article 4(6), third and fourth subparagraphs Article 7(1), introductory words Article 7(1), introductory words Article 7(1)(a) Article 7(1)(a) - Article 7(1)(b) - Article 7(1)(c) - Article 7(1)(d) Article 7(1)(b) Article 7(1)(e) Article 7(2) - Article 7(3) - - Article 7(2) - Article 7(3) Article 7(4) Article 7(4) Article 7(5) Article 7(5)(a) - Article 7(5)(b) - Article 7(5)(c) - Article 7(5)(d) Article 7 a(1) - Article 7 a(2) Article 8 Article 7 b Article 9 - Article 10 Article 8 Article 11 Article 9 Article 12 Article 10 Article 13(1) - Article 13(2) Article 11 Article 14 Article 12(1) - Article 12(2) - Article 12(3) Article 15 - Article 16 Article 13 Article 17, first subparagraph - Article 17, second subparagraph - Article 18 Article 14 Article 19 Annex I - Annex Ia Annex I Annex II Annex II Annex III Annex III - Annex IV - Annex V - Annex VI
============================== "END OF DOC" ==============================
        

: 

In [23]:
laws[laws['CELEX'] == '32015L0413']['CELEX'].iloc[0]
laws[laws['CELEX'] == '32015L0413']['Status'].iloc[0]
laws[laws['CELEX'] == '32015L0413']['Act_type'].iloc[0]
laws[laws['CELEX'] == '32015L0413']['Treaty'].iloc[0]
laws[laws['CELEX'] == '32015L0413']['act_raw_text'].iloc[0]


"13.3.2015 EN Official Journal of the European Union L 68/9 DIRECTIVE (EU) 2015/413 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL of 11 March 2015 facilitating cross-border exchange of information on road-safety-related traffic offences (Text with EEA relevance) THE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE EUROPEAN UNION, Having regard to the Treaty on the Functioning of the European Union, and in particular Article 91(1)(c) thereof, Having regard to the proposal from the European Commission, After transmission of the draft legislative act to the national parliaments, Having regard to the opinion of the European Economic and Social Committee (1), After consulting the Committee of the Regions, Acting in accordance with the ordinary legislative procedure (2), Whereas: (1) Improving road safety is a prime objective of the Union's transport policy. The Union is pursuing a policy to improve road safety with the objective of reducing fatalities, injuries and material damage. An important e

In [ ]:
def search_docs(query):
    celex_ids : list[str,str]
    full_doc_info : list[str]

    retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
    docs = retriever.invoke(query)

    for doc in docs:
        celex_ids.append(doc.metadata['celex'])

    celex_ids = list[Any](set(celex_ids))    

    
    full_doc_info = f""" Doc{i}:\n 
    
      {laws[laws['CELEX'] == celex_id]['act_raw_text'].iloc[0]} 

""" 

    return full_doc_info    



In [ ]:
search_docs("Drug dealing Sentences")

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
docs = retriever.invoke("Drug dealing Sentences")

retrived = []

for i , doc in enumerate (docs):
    retrived.append([f"Doc{i}:"  doc.metadata['status','treaty','act_type','celex']])


[Document(metadata={'subject_matter': 'sources and branches of the law;  European Union law;  justice;  criminal law', 'status': 'In Force', 'legal_basis': '12002M031; 12002M034', 'additional_info': 'CNS 2001/0114', 'chunk_number': 1, 'eurovoc': 'penal code; Community law - national law; criminal procedure; penalty; drug traffic', 'treaty': 'TEU (1992)', 'act_type': 'Decision_FRAMW', 'cites': 'joint_action/1997/396; 31999Y0123%2801%29; joint_action/1998/733', 'celex': '32004F0757', 'authors': 'European Council', 'act_name': 'Council Framework Decision 2004/757/JHA of 25 october 2004 laying down minimum provisions on the constituent elements of criminal acts and penalties in the field of illicit drug trafficking', 'document_length': 13663, 'total_chunks': 6}, page_content="11.11.2004 EN Official Journal of the European Union L 335/8 COUNCIL FRAMEWORK DECISION 2004/757/JHA of 25 october 2004 laying down minimum provisions on the constituent elements of criminal acts and penalties in the 

In [ ]:
retrived = []

for doc in docs:
    retrived.append({doc.metadata['status','treaty']})

In [45]:
docs[0].metadata['celex']

'32004F0757'

In [ ]:
def get_celex_ids(docs):
    celex_ids = []
    for i in docs:
        celex_ids.append(docs[i].metadata['celex'])

    celex_ids = list(set(celex_ids))

    return celex_ids    

In [1]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI."),
    ("human", "Answer this question: {question}")
])

formatted = prompt.invoke({"question": "What is AI?"})
print(formatted)

messages=[SystemMessage(content='You are a helpful AI.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Answer this question: What is AI?', additional_kwargs={}, response_metadata={})]


In [9]:
laws[laws['CELEX'] == '31997R2046']['act_raw_text'].iloc[0]   

"Avis juridique important|31997R2046Council Regulation (EC) No 2046/97 of 13 October 1997 on north-south cooperation in the campaign against drugs and drug addiction Official Journal L 287 , 21/10/1997 P. 0001 - 0005COUNCIL REGULATION (EC) No 2046/97 of 13 October 1997 on north-south cooperation in the campaign against drugs and drug addictionTHE COUNCIL OF THE EUROPEAN UNION,Having regard to the Treaty establishing the European Community, and in particular Article 130w thereof,Having regard to the proposal from the Commission (1),Acting in accordance with the procedure laid down in Article 189c of the Treaty (2),Whereas the impact on the structures of a developing society of an economy based on the production of drugs, or which derives a substantial revenue from them, undermines a country's smooth integration into the world economy;Whereas the breakdown of social structures in developing countries due to drug consumption and the related industry is detrimental to sustainable social de